In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

prices_path = Path("Data/train_prices_15min.csv")
prices = pd.read_csv(prices_path, parse_dates=["delivery_date"])

display(prices.head())


,delivery_date,market_id,hour_index,quarter_index,price_eur_mwh
0,2025-10-01,ZONE_0,1,1,102.60
1,2025-10-01,ZONE_0,1,2,92.24
2,2025-10-01,ZONE_0,1,3,86.03
3,2025-10-01,ZONE_0,1,4,85.39
4,2025-10-01,ZONE_0,2,1,89.99


## 1. Dataset overview

In [2]:
dataset_overview = pd.Series({
    "rows": len(prices),
    "zones": prices["market_id"].nunique(),
    "first_delivery_date": prices["delivery_date"].min(),
    "last_delivery_date": prices["delivery_date"].max(),
    "missing_prices": prices["price_eur_mwh"].isna().sum(),
}, name="value")

display(dataset_overview.to_frame())

zone_summary = (
    prices.groupby("market_id", dropna=False)
    .agg(
        observations=("price_eur_mwh", "size"),
        price_count=("price_eur_mwh", "count"),
        avg_price_eur_mwh=("price_eur_mwh", "mean"),
        median_price_eur_mwh=("price_eur_mwh", "median"),
        min_price_eur_mwh=("price_eur_mwh", "min"),
        max_price_eur_mwh=("price_eur_mwh", "max"),
    )
    .sort_index()
)
zone_summary.index.name = "zone"

display(zone_summary.round(2))


,value
rows,139392
zones,4
first_delivery_date,2025-10-01 00:00:00
last_delivery_date,2026-09-30 00:00:00
missing_prices,0


,observations,price_count,avg_price_eur_mwh,median_price_eur_mwh,min_price_eur_mwh,max_price_eur_mwh
zone,,,,,,
ZONE_0,34848,34848,104.42,104.13,-499.99,747.10
ZONE_1,34848,34848,119.81,120.32,-498.65,645.20
ZONE_2,34848,34848,138.31,134.50,0.00,349.99
ZONE_3,34848,34848,76.75,78.00,-498.65,487.38


## 2. Feature engineering

One row represents one quarter-hour in one zone. Join keys are `delivery_date`, `market_id`, `hour_index` and `quarter_index`. Predictor groups are defined once in `FEATURE_GROUPS`; the experiment tables select explicitly from these groups.

| Group | Columns | Meaning |
| --- | --- | --- |
| `position` | `position_zone`, `position_hour`, `position_quarter`, `position_quarter_of_day` | Zone category; hour 1–24; quarter within hour 1–4; quarter within day 1–96. Hour 1 begins at midnight. |
| `calendar` | `calendar_day_of_week`, `calendar_is_weekend`, `calendar_month`, `calendar_day_of_year` | Weekday (Monday = 0), weekend flag, month 1–12 and day of year. |
| `calendar` | `calendar_month_sin`, `calendar_month_cos` | Sine/cosine of `2π × (month − 1) / 12`, keeping December and January close in the annual cycle. |
| `calendar` | `calendar_year_sin`, `calendar_year_cos` | Sine/cosine of the fraction of the year elapsed, using 365/366 days. |
| `local` | `local_hourly_price_eur_mwh`, `local_prev_hour_price_eur_mwh`, `local_next_hour_price_eur_mwh` | Current, previous and next known hourly prices in the same day/zone. At day boundaries, the unavailable neighbor uses the current price. |
| `daily` | `daily_{mean,min,max,range,std}_price_eur_mwh` | Statistics of the 24 known hourly prices. Range = max − min; standard deviation uses `ddof=0`. |
| `daily` | `daily_hour_01_price_eur_mwh` … `daily_hour_24_price_eur_mwh` | Full known hourly curve, repeated on each row for that day/zone. |
| `reference` | `reference_smooth_price_eur_mwh` | Smooth quarter-hour reference from mean-preserving spline interpolation of hourly averages, implemented with SciPy. |
| `history_hourly` | `history_hourly_deviation_lag_{1,7}d_eur_mwh` | Price minus hourly average at the same zone/hour/quarter exactly 1 or 7 calendar days earlier. |
| `history_smooth` | `history_smooth_deviation_lag_{1,7}d_eur_mwh` | Price minus smooth reference at the same zone/hour/quarter exactly 1 or 7 calendar days earlier. |
| `history_hourly`, `history_smooth` | `history_{hourly,smooth}_deviation_{mean,median}_{7,28}d_eur_mwh` | Mean/median residual over the preceding 7/28 calendar days, excluding today. |
| `history_hourly`, `history_smooth` | `history_{hourly,smooth}_deviation_count_{7,28}d` | Number of observations contributing to each historical window. |
| `crosszone` | `crosszone_ZONE_{0,1,2,3}_hourly_price_eur_mwh` | Each zone's known hourly price at the same date/hour. |
| `crosszone` | `crosszone_ZONE_{0,1,2,3}_minus_local_eur_mwh` | Each zone's hourly price minus the current row's local hourly price. |
| `regime` | `regime_hourly_is_negative` | 1 if the supplied hourly price is below zero, otherwise 0. |
| `regime` | `regime_daily_negative_hours` | Number of negative prices among the day's 24 supplied hourly prices, for the same zone. |
| `temporal` | `temporal_hourly_change_from_previous_eur_mwh` | Current minus previous known hourly price, within the same day/zone. Zero at hour 1 because the previous-price fallback is the current price. |
| `temporal` | `temporal_hourly_change_to_next_eur_mwh` | Next minus current known hourly price, within the same day/zone. Zero at hour 24 because the next-price fallback is the current price. |
| `temporal` | `temporal_smooth_deviation_lag_{2,3}d_eur_mwh` | Observed price minus that historical day's smooth reference, at the same zone/hour/quarter exactly 2 or 3 calendar days earlier. Missing dates stay `NaN`. |

Regime experiments also use the existing `local_hourly_price_eur_mwh`; no duplicate price column is created. `position_zone` remains in the `position` group. All regime inputs are available before the target quarter-hour prices are revealed.

For temporal MLP inputs, the 24 `daily_hour_*` prices form the known daily curve. The four lag columns for days 1, 2, 3 and 7 are also packed into four complete 96-quarter residual curves for the same zone. This reshapes the existing dated features; it creates no extra quarter-hour observations. A missing historical day gives a missing curve.

**Continuous-sequence inputs.** The long-history TCN consumes tensor channels rather than the tabular `FEATURE_GROUPS`. Its historical sequence uses raw `sequence_price`, a binary `sequence_observed` mask, `sequence_time_sin/cos` for quarter-of-day, `sequence_weekday_sin/cos`, `sequence_month_sin/cos`, and `sequence_age_days` relative to the query day's midnight. Age is divided by 28 before entering the model. The current-day sequence uses `sequence_hourly_price`, the same time/calendar encodings at hourly positions, and four `sequence_zone_ZONE_*` indicators. Sine/cosine use periods 96 quarters or 24 hours, 7 weekdays and 12 months. Price channels share a training-only mean and standard deviation. Dates absent from historical data retain masked time slots. These channel schemas are listed in `TCN_HISTORY_CHANNELS` and `TCN_DAY_CHANNELS`; no rolling means, medians or hand-selected residual lags enter this model.

**Hourly coefficient inputs.** Section 4.9 uses one row per date/zone/hour, with its own explicit `COEFFICIENT_FEATURE_GROUPS` selection. It reuses `position` (zone/hour), `calendar`, `local`, `daily` (mean/range/std), `regime`, `temporal` (incoming/outgoing hourly changes), and `crosszone` (prices/spreads) as defined above. The following columns describe this hourly representation; they do not change the quarter-row selections used by earlier models.

| Group | Columns | Meaning |
| --- | --- | --- |
| `temporal` | `temporal_hourly_curvature_eur_mwh` | Outgoing hourly change minus incoming change. Same-day neighbor fallbacks apply. |
| `crosszone` | `crosszone_ZONE_*_change_from_previous_eur_mwh` | Each zone's current minus previous known hourly price; zero at hour 1. |
| `reference` | `reference_{slope,curvature,zigzag}_coefficient_eur_mwh` | Projection of the selected baseline's four prices, minus their hourly mean, onto the three normalized patterns in Section 4.9. |
| `history_coefficient` | `history_coefficient_{slope,curvature,zigzag}_lag_{1,7}d_eur_mwh` | Observed price-minus-baseline coefficients at the same zone/hour exactly 1 or 7 calendar days earlier, using the matching interpolation method. |
| `history_coefficient` | `history_coefficient_{slope,curvature,zigzag}_mean_{7,28}d_eur_mwh` | Mean of those residual coefficients over `[date − window, date)`. |
| `history_coefficient` | `history_coefficient_count_{7,28}d` | Number of complete historical hours contributing to that window. Missing coefficient values stay `NaN`, handled natively by LightGBM; an empty window has count zero. |

`reference_coefficient_price_eur_mwh` stores the selected four-quarter baseline for reconstruction, rather than being a scalar predictor. The slope, curvature and zigzag labels are residual projections, not observed prices used as inputs. No previous-day boundary prices enter the current-day curve features.

**Coefficient target variations.** Section 4.10 keeps these 50 hourly inputs fixed, including polynomial-basis history/reference predictors. Haar, Helmert and training-derived PCA bases change only the three target coordinates and output reconstruction. They introduce no observed target-day inputs.

**Hourly coefficient extensions (Section 4.11).** Predictors keep the same group prefixes. `curve`, `recent_history`, `weekday_profile`, `neighbor_history` and `crosszone_history` below are experiment selections from those groups, rather than different group definitions. The original 50 predictors remain unchanged.

| Group | Columns | Meaning |
| --- | --- | --- |
| `daily` | Existing `daily_hour_01_price_eur_mwh` … `daily_hour_24_price_eur_mwh`, `daily_min_price_eur_mwh`, `daily_max_price_eur_mwh` | Full known hourly curve and extremes for the target day/zone. |
| `local`, `temporal` | `local_hour_{minus,plus}_{2,3,6}_price_eur_mwh`, `temporal_hour_{minus,plus}_{2,3,6}_minus_current_eur_mwh` | Known same-day prices 2/3/6 hours before/after this hour, and neighbor minus current price. Outside-day neighbors stay `NaN`. |
| `history_coefficient` | `history_coefficient_{slope,curvature,zigzag}_lag_{2,3,14,28}d_eur_mwh` | Matching PCHIP residual coefficients exactly that many calendar days earlier. |
| `history_coefficient` | `history_coefficient_{slope,curvature,zigzag}_mean_{3,14,56}d_eur_mwh`, `history_coefficient_count_{3,14,56}d` | Additional trailing means and availability counts over `[date − window, date)`. |
| `history_coefficient` | `history_coefficient_{slope,curvature,zigzag}_{median,std}_{7,28}d_eur_mwh` | Trailing median and population standard deviation (`ddof=0`) of each residual coefficient. |
| `history_coefficient` | `history_coefficient_{slope,curvature,zigzag}_weekday_mean_{28,56}d_eur_mwh`, `history_coefficient_weekday_count_{28,56}d` | Same zone/hour/weekday profile over the previous 28/56 calendar days, with counts. |
| `history_coefficient` | `history_coefficient_{slope,curvature,zigzag}_lag_{1,7}d_{previous,next}_{1,2}h_eur_mwh` | Historical coefficient 1/2 hours before/after this hour, on the day exactly 1/7 days earlier. No cross-day boundary fallback. |
| `crosszone` | `crosszone_history_ZONE_*_{slope,curvature,zigzag}_minus_local_lag_{1,7}d_eur_mwh` | Other-zone minus own-zone residual coefficient at the same historical hour, exactly 1/7 calendar days earlier. |

All coefficient histories use the polynomial basis and the matching PCHIP reference. Missing history stays `NaN`; empty-window counts are zero. Current-day quarter-hour labels never enter predictors.


**References and labels.** `reference_hourly_price_eur_mwh` is retained to calculate hourly residuals and reconstruct prices; it equals `local_hourly_price_eur_mwh`, so it is excluded from predictors. The labels are `price_eur_mwh`, `target_hourly_deviation_eur_mwh = price − hourly reference`, and `target_smooth_deviation_eur_mwh = price − smooth reference`. Labels never enter model inputs. Predict one residual type at a time and add it to its matching reference, followed by the hourly correction.

**History and missing values.** Use strictly earlier dates, with calendar-date joins for lags. Missing historical values remain `NaN` and counts are zero with no history. Ridge and the MLP use training-only median imputation and missingness indicators; CatBoost handles numeric missingness natively.

In [2]:
import pandas as pd
import numpy as np
from scipy.interpolate import CubicSpline

DAY_KEYS = ["delivery_date", "market_id"]
HOUR_KEYS = DAY_KEYS + ["hour_index"]
QUARTER_KEYS = HOUR_KEYS + ["quarter_index"]
FEATURE_PREFIXES = (
    "position_", "calendar_", "local_", "daily_",
    "reference_", "history_", "crosszone_", "regime_", "temporal_",
)


REFERENCE_COLUMNS = {
    "hourly": "reference_hourly_price_eur_mwh",
    "smooth": "reference_smooth_price_eur_mwh",
}
TARGET_COLUMNS = {
    "hourly": "target_hourly_deviation_eur_mwh",
    "smooth": "target_smooth_deviation_eur_mwh",
}


def hourly_from_quarters(quarter_prices):
    """Recreate the challenge's hourly inputs from complete observed hours."""
    data = quarter_prices.copy()
    data["delivery_date"] = pd.to_datetime(data["delivery_date"])
    if data.duplicated(QUARTER_KEYS).any():
        raise ValueError("Duplicate quarter-hour keys.")
    if not data["quarter_index"].isin([1, 2, 3, 4]).all():
        raise ValueError("quarter_index must be 1, 2, 3 or 4.")
    if not np.isfinite(data["price_eur_mwh"]).all():
        raise ValueError("Observed prices must be finite.")
    if not data.groupby(HOUR_KEYS).size().eq(4).all():
        raise ValueError("Each observed hour must contain all four quarters.")
    return (
        data.groupby(HOUR_KEYS, as_index=False)["price_eur_mwh"]
        .mean()
        .rename(columns={"price_eur_mwh": "hourly_price_eur_mwh"})
    )


def smooth_reference(hourly_prices):
    """Expand 24 hourly averages into 96 smooth, hourly-consistent values."""
    hourly = hourly_prices.copy()
    hourly["delivery_date"] = pd.to_datetime(hourly["delivery_date"])
    if hourly.duplicated(HOUR_KEYS).any():
        raise ValueError("Duplicate hourly keys.")
    curve = hourly.pivot(index=DAY_KEYS, columns="hour_index", values="hourly_price_eur_mwh")
    curve = curve.reindex(columns=range(1, 25)).sort_index()
    if not np.isfinite(curve.to_numpy()).all():
        raise ValueError("Every day/zone needs 24 finite hourly prices.")
    if not hourly["hour_index"].isin(range(1, 25)).all():
        raise ValueError("hour_index must be between 1 and 24.")

    # Integral over a one-hour interval equals its hourly average × 1 hour.
    cumulative = np.column_stack([
        np.zeros(len(curve)), np.cumsum(curve.to_numpy(), axis=1)
    ])
    integral_spline = CubicSpline(
        np.arange(25), cumulative, axis=1, bc_type="natural", extrapolate=False
    )
    quarter_boundaries = np.arange(97) / 4
    reference = np.diff(integral_spline(quarter_boundaries), axis=1) * 4

    result = curve.index.to_frame(index=False).loc[
        curve.index.to_frame(index=False).index.repeat(96)
    ].reset_index(drop=True)
    result["hour_index"] = np.tile(np.repeat(np.arange(1, 25), 4), len(curve))
    result["quarter_index"] = np.tile(np.arange(1, 5), len(curve) * 24)
    result["reference_smooth_price_eur_mwh"] = reference.reshape(-1)
    np.testing.assert_allclose(
        reference.reshape(-1, 24, 4).mean(axis=2), curve.to_numpy(), atol=1e-8, rtol=0
    )
    return result

def build_features(hourly_prices, history_15min, target_prices=None):
    """Build features from known hourly inputs and strictly earlier history.

    target_prices is optional and adds labels only; it never feeds into X.
    Historical inputs may contain many dates: every historical feature is
    filtered by the date of its individual prediction row.
    """
    hourly = hourly_prices.copy()
    hourly["delivery_date"] = pd.to_datetime(hourly["delivery_date"])
    hourly = hourly.sort_values(HOUR_KEYS).reset_index(drop=True)
    features = smooth_reference(hourly)

    # Position and calendar.
    features["position_zone"] = features["market_id"].astype("category")
    features["position_hour"] = features["hour_index"]
    features["position_quarter"] = features["quarter_index"]
    features["position_quarter_of_day"] = (
        (features["hour_index"] - 1) * 4 + features["quarter_index"]
    )
    dates = features["delivery_date"].dt
    features["calendar_day_of_week"] = dates.dayofweek
    features["calendar_is_weekend"] = (dates.dayofweek >= 5).astype(int)
    features["calendar_month"] = dates.month
    month_angle = 2 * np.pi * (dates.month - 1) / 12
    features["calendar_month_sin"] = np.sin(month_angle)
    features["calendar_month_cos"] = np.cos(month_angle)
    features["calendar_day_of_year"] = dates.dayofyear
    year_fraction = (dates.dayofyear - 1) / np.where(dates.is_leap_year, 366, 365)
    features["calendar_year_sin"] = np.sin(2 * np.pi * year_fraction)
    features["calendar_year_cos"] = np.cos(2 * np.pi * year_fraction)

    # Local hourly inputs. Boundary neighbors fall back to the current hour.
    hourly["local_hourly_price_eur_mwh"] = hourly["hourly_price_eur_mwh"]
    hourly_group = hourly.groupby(DAY_KEYS)["hourly_price_eur_mwh"]
    hourly["local_prev_hour_price_eur_mwh"] = hourly_group.shift(1).fillna(hourly["hourly_price_eur_mwh"])
    hourly["local_next_hour_price_eur_mwh"] = hourly_group.shift(-1).fillna(hourly["hourly_price_eur_mwh"])
    hourly["temporal_hourly_change_from_previous_eur_mwh"] = hourly["local_hourly_price_eur_mwh"] - hourly["local_prev_hour_price_eur_mwh"]
    hourly["temporal_hourly_change_to_next_eur_mwh"] = hourly["local_next_hour_price_eur_mwh"] - hourly["local_hourly_price_eur_mwh"]
    local_columns = [c for c in hourly if c.startswith(("local_", "temporal_"))]
    features = features.merge(hourly[HOUR_KEYS + local_columns], on=HOUR_KEYS, validate="many_to_one")

    features["reference_hourly_price_eur_mwh"] = features["local_hourly_price_eur_mwh"]
    features["regime_hourly_is_negative"] = features["local_hourly_price_eur_mwh"].lt(0).astype(int)

    # Daily statistics and all 24 known hourly prices.
    daily = hourly.groupby(DAY_KEYS)["hourly_price_eur_mwh"].agg(
        daily_mean_price_eur_mwh="mean",
        daily_min_price_eur_mwh="min",
        daily_max_price_eur_mwh="max",
        daily_std_price_eur_mwh=lambda values: values.std(ddof=0),
        regime_daily_negative_hours=lambda values: values.lt(0).sum(),
    )
    daily["daily_range_price_eur_mwh"] = daily["daily_max_price_eur_mwh"] - daily["daily_min_price_eur_mwh"]
    whole_curve = hourly.pivot(index=DAY_KEYS, columns="hour_index", values="hourly_price_eur_mwh")
    whole_curve.columns = [f"daily_hour_{hour:02d}_price_eur_mwh" for hour in whole_curve.columns]
    features = features.merge(daily.join(whole_curve).reset_index(), on=DAY_KEYS, validate="many_to_one")

    # Every zone's hourly input at the same delivery date/hour.
    cross = hourly.pivot(index=["delivery_date", "hour_index"], columns="market_id", values="hourly_price_eur_mwh")
    cross.columns = [f"crosszone_{zone}_hourly_price_eur_mwh" for zone in cross.columns]
    features = features.merge(cross.reset_index(), on=["delivery_date", "hour_index"], validate="many_to_one")
    for column in cross.columns:
        zone = column.removeprefix("crosszone_").removesuffix("_hourly_price_eur_mwh")
        features[f"crosszone_{zone}_minus_local_eur_mwh"] = features[column] - features["local_hourly_price_eur_mwh"]

    # Both residual types are computed from each historical day's own references.
    history = history_15min.copy()
    history["delivery_date"] = pd.to_datetime(history["delivery_date"])
    historical_columns = [f"historical_{kind}_deviation" for kind in REFERENCE_COLUMNS]
    if history.empty:
        history = pd.DataFrame(columns=QUARTER_KEYS + historical_columns)
        history["delivery_date"] = pd.to_datetime(history["delivery_date"])
        history[historical_columns] = history[historical_columns].astype(float)
    else:
        historical_hourly = hourly_from_quarters(history)
        history = history.merge(smooth_reference(historical_hourly), on=QUARTER_KEYS, validate="one_to_one")
        history = history.merge(
            historical_hourly.rename(columns={"hourly_price_eur_mwh": REFERENCE_COLUMNS["hourly"]}),
            on=HOUR_KEYS, validate="many_to_one",
        )
        for kind, reference_column in REFERENCE_COLUMNS.items():
            history[f"historical_{kind}_deviation"] = history["price_eur_mwh"] - history[reference_column]
        history = history[QUARTER_KEYS + historical_columns]

    # Time-indexed history includes query dates; every rolling window excludes today.
    timeline = pd.concat([history[QUARTER_KEYS], features[QUARTER_KEYS]], ignore_index=True).drop_duplicates()
    timeline = timeline.merge(history, on=QUARTER_KEYS, how="left", validate="one_to_one")
    shape_keys = ["market_id", "hour_index", "quarter_index"]
    timeline = timeline.sort_values(shape_keys + ["delivery_date"])
    for kind in REFERENCE_COLUMNS:
        historical_column = f"historical_{kind}_deviation"
        for lag in ((1, 2, 3, 7) if kind == "smooth" else (1, 7)):
            lagged = history[QUARTER_KEYS + [historical_column]].copy()
            lagged["delivery_date"] += pd.Timedelta(days=lag)
            prefix = "temporal_smooth" if lag in (2, 3) else f"history_{kind}"
            lagged = lagged.rename(columns={historical_column: f"{prefix}_deviation_lag_{lag}d_eur_mwh"})
            features = features.merge(lagged, on=QUARTER_KEYS, how="left", validate="one_to_one")
        for days in (7, 28):
            rolling = (
                timeline.set_index("delivery_date")
                .groupby(shape_keys)[historical_column]
                .rolling(f"{days}D", closed="left", min_periods=1)
                .agg(["mean", "median", "count"])
                .reset_index()
                .rename(columns={
                    "mean": f"history_{kind}_deviation_mean_{days}d_eur_mwh",
                    "median": f"history_{kind}_deviation_median_{days}d_eur_mwh",
                    "count": f"history_{kind}_deviation_count_{days}d",
                })
            )
            features = features.merge(rolling, on=QUARTER_KEYS, how="left", validate="one_to_one")
            count_column = f"history_{kind}_deviation_count_{days}d"
            features[count_column] = features[count_column].fillna(0).astype(int)

    # Attach alternative target labels only after all model inputs are complete.
    if target_prices is not None:
        labels = target_prices[QUARTER_KEYS + ["price_eur_mwh"]].copy()
        labels["delivery_date"] = pd.to_datetime(labels["delivery_date"])
        features = features.merge(labels, on=QUARTER_KEYS, how="left", validate="one_to_one")
        for kind, reference_column in REFERENCE_COLUMNS.items():
            features[TARGET_COLUMNS[kind]] = features["price_eur_mwh"] - features[reference_column]

    feature_columns = [c for prefix in FEATURE_PREFIXES for c in features if c.startswith(prefix)]
    label_columns = [c for c in ["price_eur_mwh", *TARGET_COLUMNS.values()] if c in features]
    return features[QUARTER_KEYS + feature_columns + label_columns].sort_values(QUARTER_KEYS).reset_index(drop=True)

def get_feature_groups(feature_table):
    """One canonical vocabulary for definitions, selection and reporting."""
    groups = {}
    for group in ("position", "calendar", "local", "daily", "reference",
                  "history_hourly", "history_smooth", "crosszone", "regime", "temporal"):
        groups[group] = [c for c in feature_table if c.startswith(group + "_")]
    # The hourly reference equals local_hourly_price_eur_mwh: keep one predictor.
    groups["reference"] = ["reference_smooth_price_eur_mwh"]
    columns = [c for values in groups.values() for c in values]
    if len(columns) != len(set(columns)):
        raise ValueError("Feature groups must not overlap.")
    return groups


train_hourly = hourly_from_quarters(prices)
feature_table = build_features(train_hourly, history_15min=prices, target_prices=prices)
FEATURE_GROUPS = get_feature_groups(feature_table)
display(pd.Series({group: len(columns) for group, columns in FEATURE_GROUPS.items()}, name="feature_count").to_frame())

,feature_count
position,4
calendar,8
local,3
daily,29
reference,1
history_hourly,8
history_smooth,8
crosszone,8
regime,2
temporal,4


## 3. Helpers: output constraints and MAE

Output columns: `delivery_date, market_id, hour_index, quarter_index, predicted_price_eur_mwh`.

Checks require all supplied keys, four zones, 24 hours and 96 predictions per day/zone; exactly one finite price per quarter 1–4; no missing/extra/duplicate keys; and `abs(mean(four predictions) − hourly price) <= 0.0001 EUR/MWh`. Zero and negative prices are valid.

Preliminary price = matching reference + predicted residual. Add `hourly price − mean(four preliminary prices)` to every quarter in the hour to enforce consistency.

The official MAE averages absolute price errors **within each day/zone → across days within each zone → equally across the four zones (25% each)**. Every exported CSV is read back, checked and scored.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

EXPECTED_ZONES = ("ZONE_0", "ZONE_1", "ZONE_2", "ZONE_3")
PREDICTION_COLUMN = "predicted_price_eur_mwh"
OUTPUT_COLUMNS = QUARTER_KEYS + [PREDICTION_COLUMN]
HOURLY_TOLERANCE = 0.0001


def validate_prediction_output(predictions, hourly_inputs, tolerance=HOURLY_TOLERANCE):
    """Raise ValueError if any required schema, coverage or hourly check fails."""
    if not np.isfinite(tolerance) or not 0 <= tolerance <= HOURLY_TOLERANCE:
        raise ValueError("Tolerance cannot exceed the official 0.0001 EUR/MWh.")
    if list(predictions.columns) != OUTPUT_COLUMNS:
        raise ValueError(f"Prediction columns must be {OUTPUT_COLUMNS}, in this order.")
    pred = predictions.copy()
    hourly = hourly_inputs[HOUR_KEYS + ["hourly_price_eur_mwh"]].copy()
    for data, keys, value in (
        (pred, QUARTER_KEYS, PREDICTION_COLUMN),
        (hourly, HOUR_KEYS, "hourly_price_eur_mwh"),
    ):
        data["delivery_date"] = pd.to_datetime(data["delivery_date"], errors="raise")
        if data.empty or data[keys].isna().any().any():
            raise ValueError("Empty input or missing identifier.")
        if data.duplicated(keys).any():
            raise ValueError("Duplicate date/zone/time keys.")
        if not data["hour_index"].isin(range(1, 25)).all():
            raise ValueError("Hour indices must be integers 1–24.")
        if (not pd.api.types.is_numeric_dtype(data[value])
                or pd.api.types.is_bool_dtype(data[value])
                or not np.isfinite(data[value]).all()):
            raise ValueError("Prices must be finite numeric values.")
    if not pred["quarter_index"].isin([1, 2, 3, 4]).all():
        raise ValueError("Quarter indices must be integers 1–4.")
    if not hourly.groupby(DAY_KEYS).size().eq(24).all():
        raise ValueError("Hourly input must have all 24 hours per day/zone.")
    zone_sets = hourly.groupby("delivery_date")["market_id"].agg(lambda values: set(values))
    if not zone_sets.map(lambda zones: zones == set(EXPECTED_ZONES)).all():
        raise ValueError("Hourly input must contain the four official zones for every date.")

    expected_keys = hourly[HOUR_KEYS].merge(pd.DataFrame({"quarter_index": [1, 2, 3, 4]}), how="cross")
    coverage = expected_keys.merge(pred[QUARTER_KEYS], on=QUARTER_KEYS, how="outer", indicator=True, validate="one_to_one")
    if not coverage["_merge"].eq("both").all():
        raise ValueError("Missing or unexpected prediction keys.")
    if not pred.groupby(HOUR_KEYS).size().eq(4).all():
        raise ValueError("Each hour needs exactly four quarters.")
    if not pred.groupby(DAY_KEYS).size().eq(96).all():
        raise ValueError("Each day/zone needs exactly 96 predictions.")
    averages = pred.groupby(HOUR_KEYS, as_index=False)[PREDICTION_COLUMN].mean()
    averages = averages.merge(hourly, on=HOUR_KEYS, validate="one_to_one")
    hourly_errors = (averages[PREDICTION_COLUMN] - averages["hourly_price_eur_mwh"]).abs()
    if not np.isfinite(hourly_errors).all():
        raise ValueError("Calculated hourly consistency errors must be finite.")
    max_error = hourly_errors.max()
    if max_error > tolerance:
        raise ValueError(f"Hourly consistency error {max_error:.10g} exceeds {tolerance} EUR/MWh.")
    return {
        "constraints_passed": True,
        "prediction_rows": len(pred),
        "prediction_days": pred["delivery_date"].nunique(),
        "max_hourly_error_eur_mwh": float(max_error),
    }


def make_predictions(feature_rows, hourly_inputs, predicted_deviations=None, *,
                     reference_type="smooth", reference_column=None):
    """Reconstruct prices using the specified reference, then enforce hourly means."""
    if reference_column is None:
        if reference_type not in REFERENCE_COLUMNS:
            raise ValueError("Supply an explicit reference_column for other references.")
        reference_column = REFERENCE_COLUMNS[reference_type]
    deviations = np.zeros(len(feature_rows)) if predicted_deviations is None else np.asarray(predicted_deviations, dtype=float)
    if deviations.shape != (len(feature_rows),) or not np.isfinite(deviations).all():
        raise ValueError("Need one finite predicted deviation per feature row.")
    output = feature_rows[QUARTER_KEYS].copy()
    output[PREDICTION_COLUMN] = feature_rows[reference_column].to_numpy() + deviations
    hourly = hourly_inputs[HOUR_KEYS + ["hourly_price_eur_mwh"]].copy()
    hourly["delivery_date"] = pd.to_datetime(hourly["delivery_date"])
    output = output.merge(hourly, on=HOUR_KEYS, how="left", validate="many_to_one")
    preliminary_means = output.groupby(HOUR_KEYS)[PREDICTION_COLUMN].transform("mean")
    output[PREDICTION_COLUMN] += output["hourly_price_eur_mwh"] - preliminary_means
    output = output[OUTPUT_COLUMNS].sort_values(QUARTER_KEYS).reset_index(drop=True)
    validate_prediction_output(output, hourly_inputs)
    return output


def challenge_mae(predictions, actual_prices, hourly_inputs):
    """Official date -> zone -> equally weighted four-zone MAE."""
    validate_prediction_output(predictions, hourly_inputs)
    pred = predictions.copy()
    truth = actual_prices[QUARTER_KEYS + ["price_eur_mwh"]].copy()
    for data in (pred, truth):
        data["delivery_date"] = pd.to_datetime(data["delivery_date"])
    if truth[QUARTER_KEYS].isna().any().any() or truth.duplicated(QUARTER_KEYS).any():
        raise ValueError("Observed labels have missing or duplicate keys.")
    if not pd.api.types.is_numeric_dtype(truth["price_eur_mwh"]) or not np.isfinite(truth["price_eur_mwh"]).all():
        raise ValueError("Observed labels must be finite numeric prices.")
    errors = pred.merge(truth, on=QUARTER_KEYS, how="outer", indicator=True, validate="one_to_one")
    if not errors["_merge"].eq("both").all():
        raise ValueError("Predictions and observed labels must have exactly matching keys.")
    errors["absolute_error_eur_mwh"] = (errors[PREDICTION_COLUMN] - errors["price_eur_mwh"]).abs()
    by_day_zone = errors.groupby(DAY_KEYS, as_index=False)["absolute_error_eur_mwh"].mean().rename(columns={"absolute_error_eur_mwh": "mae_eur_mwh"})
    by_zone = by_day_zone.groupby("market_id", as_index=False)["mae_eur_mwh"].mean()
    by_zone = by_zone.set_index("market_id").loc[list(EXPECTED_ZONES)].reset_index()
    return {
        "mae_eur_mwh": float(by_zone["mae_eur_mwh"].mean()),
        "by_day_zone": by_day_zone,
        "by_zone": by_zone,
    }


def evaluate_experiment(name, feature_rows, hourly_inputs, actual_prices,
                        predicted_deviations=None, *, reference_type="smooth", reference_column=None,
                        output_dir=Path("outputs/experiments")):
    """Correct, validate, export, read back, score and display one experiment."""
    predictions = make_predictions(feature_rows, hourly_inputs, predicted_deviations, reference_type=reference_type,
                                   reference_column=reference_column)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / f"{name}_predictions.csv"
    predictions.to_csv(path, index=False, float_format="%.10f", date_format="%Y-%m-%d")
    predictions = pd.read_csv(path, parse_dates=["delivery_date"])
    checks = validate_prediction_output(predictions, hourly_inputs)
    scores = challenge_mae(predictions, actual_prices, hourly_inputs)
    scores["by_day_zone"].to_csv(output_dir / f"{name}_mae_by_day_zone.csv", index=False)
    summary = {"experiment": name, "reference_type": reference_type, "mae_eur_mwh": scores["mae_eur_mwh"], **checks, "prediction_file": str(path)}
    print(f"{name}: MAE = {scores['mae_eur_mwh']:.6f} EUR/MWh | constraints: PASS | max hourly error = {checks['max_hourly_error_eur_mwh']:.3g}")
    return {"summary": summary, "predictions": predictions, "scores": scores}

def historical_residual_profile(query_rows, historical_residuals, *, reference_type="smooth"):
    """Average past residuals per zone/hour/quarter; exclude each query date.

    Each day contributes at most one residual to a profile position.
    Missing history falls back to zero deviation from the selected reference.
    """
    if reference_type not in TARGET_COLUMNS:
        raise ValueError("reference_type must be hourly or smooth.")
    target_column = TARGET_COLUMNS[reference_type]
    shape_keys = ["market_id", "hour_index", "quarter_index"]
    queries = query_rows[QUARTER_KEYS].copy()
    history = historical_residuals[QUARTER_KEYS + [target_column]].copy()
    for data in (queries, history):
        data["delivery_date"] = pd.to_datetime(data["delivery_date"])
        if data[QUARTER_KEYS].isna().any().any() or data.duplicated(QUARTER_KEYS).any():
            raise ValueError("Profile inputs require unique, non-missing quarter-hour keys.")
    if not np.isfinite(history[target_column]).all():
        raise ValueError("Historical residuals must be finite.")
    timeline = pd.concat([history[QUARTER_KEYS], queries], ignore_index=True).drop_duplicates()
    timeline = timeline.merge(history, on=QUARTER_KEYS, how="left", validate="one_to_one")
    timeline = timeline.sort_values(shape_keys + ["delivery_date"])
    residuals = timeline.groupby(shape_keys)[target_column]
    # Shift within the same zone/hour/quarter BEFORE aggregating over dates.
    timeline["profile_residual_sum_eur_mwh"] = residuals.transform(
        lambda values: values.shift(1).expanding(min_periods=1).sum()
    ).fillna(0)
    timeline["profile_observation_count"] = residuals.transform(
        lambda values: values.shift(1).expanding(min_periods=1).count()
    ).fillna(0).astype(int)
    timeline["profile_mean_deviation_eur_mwh"] = (
        timeline["profile_residual_sum_eur_mwh"]
        / timeline["profile_observation_count"].replace(0, np.nan)
    ).fillna(0)
    profile_columns = [
        "profile_residual_sum_eur_mwh", "profile_observation_count",
        "profile_mean_deviation_eur_mwh",
    ]
    # Preserve query order to align the returned deviations with feature rows.
    return queries.merge(timeline[QUARTER_KEYS + profile_columns], on=QUARTER_KEYS,
                         how="left", validate="one_to_one")

def zone_hourly_mae(predictions, actual_prices, zone):
    """Quarter-average MAE per day/hour, then average days for one zone."""
    pred = predictions.loc[predictions.market_id.eq(zone), QUARTER_KEYS + [PREDICTION_COLUMN]].copy()
    truth = actual_prices.loc[actual_prices.market_id.eq(zone), QUARTER_KEYS + ["price_eur_mwh"]].copy()
    for rows in (pred, truth):
        rows["delivery_date"] = pd.to_datetime(rows["delivery_date"])
    errors = pred.merge(truth, on=QUARTER_KEYS, how="outer", validate="one_to_one", indicator=True)
    if errors.empty or not errors["_merge"].eq("both").all():
        raise ValueError("Zone predictions and labels must have identical, non-empty keys.")
    errors["absolute_error_eur_mwh"] = (errors[PREDICTION_COLUMN] - errors["price_eur_mwh"]).abs()
    daily_hour = errors.groupby(HOUR_KEYS, as_index=False)["absolute_error_eur_mwh"].mean()
    return daily_hour.groupby("hour_index", as_index=False).agg(
        mae_eur_mwh=("absolute_error_eur_mwh", "mean"),
        validation_days=("delivery_date", "nunique"),
    )


## 4. Experiments / models

Train on the first 303 available dates, through August 1, 2026. Score the final 60 dates, August 2–September 30. Models stay fixed during validation; history features and profile averages can use actual observations from strictly earlier dates, including earlier validation dates.

Ridge and global CatBoost use the predictor selections below from `FEATURE_GROUPS`. The first six are compared with both **hourly** and **smooth** residual targets; the remaining comparisons use smooth residuals.

| Experiment | Predictor groups |
| --- | --- |
| `calendar` | `position` + `calendar` |
| `calendar_hourly` | Above + `local` + `daily` + `reference` |
| `calendar_hourly_history` | Above + the target's matching `history_hourly` or `history_smooth` group |
| `calendar_hourly_both_histories` | Above + the other history group |
| `all` | Above + `crosszone` + `regime` |
| `history_only` | Only the target's matching history group, as a separate comparison |
| `history_zone` | Smooth history + existing `position_zone` column |
| `history_regime` | Smooth history + `local_hourly_price_eur_mwh` + `regime` |
| `history_zone_regime` | Smooth history + `position_zone` + hourly price + `regime` |

Even when reference prices are excluded from model predictors, the matching reference is still used to reconstruct final prices. All fitting uses the official training data. The final holdout is a model-comparison set; settings here are fixed, not optimized against it.
The last three experiments focus on the best smooth-residual setup. CatBoost uses the selected 1,000-tree settings for these comparisons, matching `history_only_tuned`; Ridge retains `alpha=10`. `history_per_zone` and `history_regime_per_zone` route each zone to its own CatBoost model, with history alone or history plus the same regime inputs.

The MLP subsection compares smooth history, zone/calendar context, the known hourly curve and lagged daily residual curves. Its temporal comparisons use the `temporal` group alongside explicitly selected existing columns; the Ridge/CatBoost selections above stay fixed for comparison. It packs four quarter-hour rows into one hourly input and predicts four residuals together.

The temporal CNN uses the same hourly-context and temporal source-feature selections as the MLP, represented as 24-hour day/zone sequences. It learns shared convolutional filters across hours.

TabM uses the MLP input selections and chronological training protocol. Fixed blends combine corrected component prices; three additional chronological blocks check the 50/50 CNN–CatBoost recipe separately from the final-holdout comparison.

The continuous-history TCN compares 14/28-day raw quarter-hour windows, with a separate encoder for the known current-day hourly curve. It uses its own sequence representation and the same reconstruction, chronological split and official scoring helpers.

The hourly coefficient LightGBM compares spline/PCHIP references and correction weights 1.0/0.8 using three zero-sum shape targets. Its hourly feature selection is explicit and separate from the quarter-row model selections.

Section 4.13 separately evaluates the original 50-feature shared PCHIP coefficient model on September only, training through August 31. Its score is kept separate from the common 60-date comparisons.


In [4]:
import numpy as np
from scipy import sparse
from catboost import CatBoostRegressor
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler


CATBOOST_SELECTED_PARAMETERS = {
    "depth": 6, "learning_rate": 0.03, "iterations": 1000,
    "l2_leaf_reg": 3, "random_strength": 1,
}
ZONE_REGIME_EXPERIMENT_NAMES = ("history_zone", "history_regime", "history_zone_regime")

CALENDAR_INPUTS = [
    "position_zone", "position_hour", "position_quarter", "calendar_day_of_week",
    "calendar_month_sin", "calendar_month_cos",
]

def indicator_matrix(indices, width):
    indices = np.asarray(indices, dtype=int)
    return sparse.csr_matrix((np.ones(len(indices)), (np.arange(len(indices)), indices)), shape=(len(indices), width))

def ridge_design(data, extra_columns):
    """Known calendar basis plus quarter interactions for the selected inputs."""
    quarter = data["position_quarter"].to_numpy(dtype=int) - 1
    hour_quarter = (data["position_hour"].to_numpy(dtype=int) - 1) * 4 + quarter
    weekday_quarter = data["calendar_day_of_week"].to_numpy(dtype=int) * 4 + quarter
    zone_number = data["position_zone"].map({zone: i for i, zone in enumerate(EXPECTED_ZONES)}).to_numpy(dtype=int)
    hq = indicator_matrix(hour_quarter, 96)
    q = indicator_matrix(quarter, 4)
    blocks = [
        hq, indicator_matrix(weekday_quarter, 28), indicator_matrix(zone_number * 4 + quarter, 16),
        hq.multiply(data["calendar_month_sin"].to_numpy()[:, None]),
        hq.multiply(data["calendar_month_cos"].to_numpy()[:, None]),
    ]
    missing_indicators = [c for c in data.columns if c.startswith("missingindicator_")]
    for column in list(extra_columns) + missing_indicators:
        values = data[column].to_numpy(dtype=float)
        if column.startswith("history_") and column.endswith("_eur_mwh"):
            blocks.append(sparse.csr_matrix(values[:, None]))
        else:
            blocks.append(q.multiply(values[:, None]))
    return sparse.hstack(blocks, format="csr")


def residual_context_design(data, numeric_columns):
    """Shared history effects plus optional regime and zone interactions."""
    indicators = [c for c in data if c.startswith("missingindicator_")]
    numeric = sparse.csr_matrix(data[list(numeric_columns) + indicators].to_numpy(dtype=float))
    history = [c for c in numeric_columns if c.startswith("history_") and c.endswith("_eur_mwh")]
    blocks = [numeric]
    for context in ("local_hourly_price_eur_mwh", "regime_hourly_is_negative", "regime_daily_negative_hours"):
        if context in numeric_columns:
            blocks.append(sparse.csr_matrix(data[history].to_numpy(dtype=float) * data[context].to_numpy(dtype=float)[:, None]))
    shared = sparse.hstack(blocks, format="csr")
    if "position_zone" not in data:
        return shared
    zone_index = data["position_zone"].map({zone: i for i, zone in enumerate(EXPECTED_ZONES)}).to_numpy(dtype=int)
    blocks = [shared, indicator_matrix(zone_index, len(EXPECTED_ZONES))]
    blocks += [shared.multiply((zone_index == i)[:, None]) for i in range(len(EXPECTED_ZONES))]
    return sparse.hstack(blocks, format="csr")


def make_ridge_model(columns, alpha=10.0):
    if "position_quarter" not in columns and ("position_zone" in columns or "regime_hourly_is_negative" in columns):
        numeric = [c for c in columns if c != "position_zone"]
        imputation = ColumnTransformer([
            ("numeric", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True), numeric),
        ], remainder="passthrough", verbose_feature_names_out=False).set_output(transform="pandas")
        return Pipeline([
            ("imputation", imputation),
            ("design", FunctionTransformer(residual_context_design, validate=False, kw_args={"numeric_columns": numeric})),
            ("scaling", StandardScaler(with_mean=False)),
            ("regressor", Ridge(alpha=alpha, solver="lsqr", tol=1e-6)),
        ])
    if "position_quarter" not in columns:
        return Pipeline([
            ("imputation", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
            ("scaling", StandardScaler()),
            ("regressor", Ridge(alpha=alpha, solver="cholesky")),
        ])
    numeric = [c for c in columns if c != "position_zone"]
    imputation = ColumnTransformer([
        ("numeric", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True), numeric),
    ], remainder="passthrough", verbose_feature_names_out=False).set_output(transform="pandas")
    return Pipeline([
        ("imputation", imputation),
        ("design", FunctionTransformer(ridge_design, validate=False,
             kw_args={"extra_columns": [c for c in columns if c not in CALENDAR_INPUTS]})),
        ("scaling", StandardScaler(with_mean=False)),
        ("regressor", Ridge(alpha=alpha, solver="lsqr", tol=1e-6)),
    ])


def make_catboost_model(columns, parameters=None):
    settings = dict(iterations=500, depth=6, learning_rate=0.05)
    settings.update(parameters or {})
    return CatBoostRegressor(
        **settings, loss_function="MAE", eval_metric="MAE",
        cat_features=["position_zone"] if "position_zone" in columns else [],
        random_seed=42, thread_count=4, allow_writing_files=False, verbose=False,
    )


def run_models(model_name, training_rows, validation_rows, experiments_by_target,
               validation_hourly, validation_truth):
    results, fitted_models = {}, {}
    for target, configurations in experiments_by_target.items():
        for experiment, columns in configurations.items():
            name = f"{model_name}_{target}_{experiment}"
            parameters = CATBOOST_SELECTED_PARAMETERS if experiment in ZONE_REGIME_EXPERIMENT_NAMES else None
            model = make_ridge_model(columns) if model_name == "ridge" else make_catboost_model(columns, parameters)
            model.fit(training_rows[columns], training_rows[TARGET_COLUMNS[target]])
            result = evaluate_experiment(
                name, validation_rows, validation_hourly, validation_truth,
                predicted_deviations=model.predict(validation_rows[columns]), reference_type=target,
            )
            result["summary"].update(model=model_name, feature_set=experiment, feature_count=len(columns))
            results[name], fitted_models[name] = result, model
    return results, fitted_models


import json

validation_start = pd.Timestamp(np.sort(feature_table.delivery_date.unique())[-60])
fit_rows = feature_table[feature_table.delivery_date < validation_start].reset_index(drop=True)
validation_hourly = train_hourly[train_hourly.delivery_date >= validation_start].copy()
validation_truth = prices[prices.delivery_date >= validation_start].copy()
validation_rows = build_features(validation_hourly, history_15min=prices)

def select_groups(*names):
    return [column for name in names for column in FEATURE_GROUPS[name]]

EXPERIMENTS = {}
for target in REFERENCE_COLUMNS:
    own, other = f"history_{target}", f"history_{'smooth' if target == 'hourly' else 'hourly'}"
    calendar = ("position", "calendar")
    hourly = calendar + ("local", "daily", "reference")
    EXPERIMENTS[target] = {
        "calendar": select_groups(*calendar),
        "calendar_hourly": select_groups(*hourly),
        "calendar_hourly_history": select_groups(*hourly, own),
        "calendar_hourly_both_histories": select_groups(*hourly, own, other),
        "all": select_groups(*hourly, own, other, "crosszone", "regime"),
        "history_only": select_groups(own),
    }

# Controlled additions to the best history-only smooth-residual setup.
regime_inputs = ["local_hourly_price_eur_mwh"] + FEATURE_GROUPS["regime"]
history_inputs = FEATURE_GROUPS["history_smooth"]
EXPERIMENTS["smooth"].update({
    "history_zone": history_inputs + ["position_zone"],
    "history_regime": history_inputs + regime_inputs,
    "history_zone_regime": history_inputs + ["position_zone"] + regime_inputs,
})

output_dir = Path("outputs/experiments")
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / "feature_sets.json").write_text(json.dumps(EXPERIMENTS, indent=2) + "\n")
experiments, models = {}, {}
print(f"Training: {fit_rows.delivery_date.nunique()} dates; validation: {validation_rows.delivery_date.nunique()} dates")

Training: 303 dates; validation: 60 dates


### 4.1 Baselines

For each reference: (1) use the reference alone; (2) add the average residual from all earlier dates for the same zone/hour/quarter (96 averages per zone). With no history, use zero residual.

In [6]:
for target in REFERENCE_COLUMNS:
    for method in ("reference", "historical_profile"):
        deviations = None
        if method == "historical_profile":
            profile = historical_residual_profile(validation_rows, feature_table, reference_type=target)
            deviations = profile["profile_mean_deviation_eur_mwh"].to_numpy()
        name = f"baseline_{target}_{method}"
        experiments[name] = evaluate_experiment(
            name, validation_rows, validation_hourly, validation_truth,
            predicted_deviations=deviations, reference_type=target,
        )
        experiments[name]["summary"].update(model="baseline", feature_set=method, feature_count=0)

baseline_hourly_reference: MAE = 9.753915 EUR/MWh | constraints: PASS | max hourly error = 5.68e-14


baseline_hourly_historical_profile: MAE = 6.828230 EUR/MWh | constraints: PASS | max hourly error = 2.51e-11
baseline_smooth_reference: MAE = 6.495620 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


baseline_smooth_historical_profile: MAE = 5.319968 EUR/MWh | constraints: PASS | max hourly error = 2.51e-11


### 4.2 Ridge regression

Fixed `alpha=10`. Calendar inputs are encoded as hour × quarter, weekday × quarter, zone × quarter and month sine/cosine × hour × quarter. Other calendar/hourly/context values are interacted with quarter; historical residual values enter directly. This lets effects change the within-hour shape. Imputation and scaling use training dates only. The `history_only` experiment uses the selected numeric history columns directly.
For history-based experiments, Ridge uses zone × historical-feature interactions and regime × historical-residual interactions. A categorical zone offset alone would shift all four quarters equally and be removed by hourly correction.


In [7]:
ridge_results, ridge_models = run_models(
    "ridge", fit_rows, validation_rows, EXPERIMENTS, validation_hourly, validation_truth,
)
experiments.update(ridge_results)
models.update(ridge_models)

# Best validation experiment, disaggregated by zone.
best_ridge = min(ridge_results.values(), key=lambda result: result["summary"]["mae_eur_mwh"])
print(f"Best Ridge: {best_ridge['summary']['experiment']} | validation MAE: {best_ridge['summary']['mae_eur_mwh']:.6f} EUR/MWh")
display(best_ridge["scores"]["by_zone"].round(6))

# Hourly validation MAE for the best experiment's worst zone.
worst_ridge_zone = best_ridge["scores"]["by_zone"].set_index("market_id")["mae_eur_mwh"].idxmax()
worst_ridge_hourly = zone_hourly_mae(best_ridge["predictions"], validation_truth, worst_ridge_zone)
print(f"Worst zone for best Ridge: {worst_ridge_zone} — validation MAE by hour (hour 1 = 00:00–01:00)")
display(worst_ridge_hourly.round(6))


ridge_hourly_calendar: MAE = 6.777083 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


ridge_hourly_calendar_hourly: MAE = 5.388596 EUR/MWh | constraints: PASS | max hourly error = 2.51e-11


ridge_hourly_calendar_hourly_history: MAE = 5.360577 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


ridge_hourly_calendar_hourly_both_histories: MAE = 5.197943 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


ridge_hourly_all: MAE = 5.206980 EUR/MWh | constraints: PASS | max hourly error = 2.51e-11


...

ridge_smooth_history_regime: MAE = 5.320902 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


ridge_smooth_history_zone_regime: MAE = 5.345835 EUR/MWh | constraints: PASS | max hourly error = 2.51e-11
Best Ridge: ridge_smooth_history_only | validation MAE: 5.146096 EUR/MWh


,market_id,mae_eur_mwh
0,ZONE_0,3.921702
1,ZONE_1,6.443045
2,ZONE_2,5.529641
3,ZONE_3,4.689998


Worst zone for best Ridge: ZONE_1 — validation MAE by hour (hour 1 = 00:00–01:00)


,hour_index,mae_eur_mwh,validation_days
0,1,4.758999,60
1,2,3.684235,60
2,3,2.352342,60
3,4,2.084355,60
4,5,2.851122,60
5,6,4.074253,60
6,7,4.656883,60
7,8,6.897756,60
8,9,8.058422,60
9,10,8.943985,60


### 4.3 CatBoost

Initial comparisons use 500 iterations, depth 6, learning rate 0.05, MAE loss and seed 42. Uses the same selected input columns as Ridge; zone is categorical when present. Interactions and numeric missing values are handled by CatBoost. No validation labels are passed to `fit`.
The zone/regime experiments use the selected settings so their comparison changes only the inputs. Separate-zone experiments fit four independent CatBoost models on the same training dates, then combine their residual predictions before the shared hourly correction.


In [8]:
catboost_results, catboost_models = run_models(
    "catboost", fit_rows, validation_rows, EXPERIMENTS, validation_hourly, validation_truth,
)
experiments.update(catboost_results)
models.update(catboost_models)


catboost_hourly_calendar: MAE = 6.705284 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


catboost_hourly_calendar_hourly: MAE = 5.418144 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


catboost_hourly_calendar_hourly_history: MAE = 6.734017 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


catboost_hourly_calendar_hourly_both_histories: MAE = 6.668349 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


catboost_hourly_all: MAE = 6.032024 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


...

catboost_smooth_all: MAE = 5.083516 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


catboost_smooth_history_only: MAE = 5.068866 EUR/MWh | constraints: PASS | max hourly error = 2.51e-11


catboost_smooth_history_zone: MAE = 5.071473 EUR/MWh | constraints: PASS | max hourly error = 2.51e-11


catboost_smooth_history_regime: MAE = 5.496317 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


catboost_smooth_history_zone_regime: MAE = 5.491556 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


**Selected settings.** The smooth-target `history_only` model uses depth 6, learning rate 0.03, 1,000 iterations, `l2_leaf_reg=3` and `random_strength=1`. These settings were selected on earlier chronological folds. Fit on all 303 training dates and evaluate on the same final 60 dates.

In [9]:
best_catboost_parameters = CATBOOST_SELECTED_PARAMETERS.copy()
selected_columns = EXPERIMENTS["smooth"]["history_only"]
tuned_catboost_model = CatBoostRegressor(
    **best_catboost_parameters, loss_function="MAE", eval_metric="MAE", cat_features=[],
    random_seed=42, thread_count=4, allow_writing_files=False, verbose=False,
)
tuned_catboost_model.fit(fit_rows[selected_columns], fit_rows[TARGET_COLUMNS["smooth"]])
tuned_name = "catboost_smooth_history_only_tuned"
tuned_catboost = evaluate_experiment(
    tuned_name, validation_rows, validation_hourly, validation_truth,
    predicted_deviations=tuned_catboost_model.predict(validation_rows[selected_columns]), reference_type="smooth",
)
tuned_catboost["summary"].update(model="catboost", feature_set="history_only_tuned", feature_count=len(selected_columns))
catboost_results[tuned_name], catboost_models[tuned_name] = tuned_catboost, tuned_catboost_model
experiments[tuned_name], models[tuned_name] = tuned_catboost, tuned_catboost_model
tuned_catboost_model.save_model(str(output_dir / f"{tuned_name}.cbm"))


catboost_smooth_history_only_tuned: MAE = 5.068817 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


In [10]:
# Four independent zone models, with a fixed feature list and fixed parameters.
per_zone_models = {}
for feature_set, columns in {
    "history_per_zone": history_inputs,
    "history_regime_per_zone": history_inputs + regime_inputs,
}.items():
    residual_predictions = np.empty(len(validation_rows))
    fitted_by_zone = {}
    for zone in EXPECTED_ZONES:
        training = fit_rows.loc[fit_rows.market_id.eq(zone)]
        mask = validation_rows.market_id.eq(zone).to_numpy()
        model = make_catboost_model(columns, CATBOOST_SELECTED_PARAMETERS)
        model.fit(training[columns], training[TARGET_COLUMNS["smooth"]])
        residual_predictions[mask] = model.predict(validation_rows.loc[mask, columns])
        fitted_by_zone[zone] = model
    name = f"catboost_smooth_{feature_set}"
    result = evaluate_experiment(
        name, validation_rows, validation_hourly, validation_truth,
        predicted_deviations=residual_predictions, reference_type="smooth",
    )
    result["summary"].update(model="catboost", feature_set=feature_set, feature_count=len(columns))
    catboost_results[name], experiments[name] = result, result
    per_zone_models[feature_set] = fitted_by_zone
    catboost_models[name], models[name] = fitted_by_zone, fitted_by_zone


catboost_smooth_history_per_zone: MAE = 5.086959 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


catboost_smooth_history_regime_per_zone: MAE = 5.379953 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


In [11]:
# Best validation experiment, disaggregated by zone.
best_catboost = min(catboost_results.values(), key=lambda result: result["summary"]["mae_eur_mwh"])
print(f"Best Catboost: {best_catboost['summary']['experiment']} | validation MAE: {best_catboost['summary']['mae_eur_mwh']:.6f} EUR/MWh")
display(best_catboost["scores"]["by_zone"].round(6))

# Hourly validation MAE for the best experiment's worst zone.
worst_catboost_zone = best_catboost["scores"]["by_zone"].set_index("market_id")["mae_eur_mwh"].idxmax()
worst_catboost_hourly = zone_hourly_mae(best_catboost["predictions"], validation_truth, worst_catboost_zone)
print(f"Worst zone for best Catboost: {worst_catboost_zone} — validation MAE by hour (hour 1 = 00:00–01:00)")
display(worst_catboost_hourly.round(6))


Best Catboost: catboost_smooth_history_only_tuned | validation MAE: 5.068817 EUR/MWh


,market_id,mae_eur_mwh
0,ZONE_0,3.879659
1,ZONE_1,6.318112
2,ZONE_2,5.461259
3,ZONE_3,4.616237


Worst zone for best Catboost: ZONE_1 — validation MAE by hour (hour 1 = 00:00–01:00)


,hour_index,mae_eur_mwh,validation_days
0,1,4.719771,60
1,2,3.608302,60
2,3,2.310286,60
3,4,1.976544,60
4,5,2.734764,60
5,6,4.099478,60
6,7,4.606097,60
7,8,6.716474,60
8,9,8.068579,60
9,10,8.897779,60


### 4.4 Feed-forward neural network (MLP)

A small feed-forward network (MLP) predicts the four smooth-reference residuals of one hour together. Hidden layers have 64 and 32 ReLU units, with 10% dropout. Its four outputs are centered inside the model, so their average is zero during training and inference. The shared reconstruction helper also checks the final hourly averages.

| Experiment | Inputs |
| --- | --- |
| `history_only` | The eight `history_smooth` features at each of the four quarter positions (32 numeric inputs before preprocessing). |
| `history_calendar` | Same history plus `position_zone`, `position_hour`, `calendar_day_of_week` and `calendar_month_sin/cos`. |
| `history_calendar_hourly` | Above plus current/previous/next known hourly prices, the two `temporal` hourly price changes, and all 24 `daily_hour_*` prices. |
| `history_calendar_temporal` | Above plus the full 96-quarter smooth-residual curves from exactly 1, 2, 3 and 7 calendar days earlier. |

Zone, hour and weekday are one-hot encoded. Quarter position is implicit in the ordered input/output slots. Both temporal comparisons retain the same 64/32-unit architecture and training settings. Complete earlier residual curves are assembled from strictly lagged feature columns, never from current-day labels; 1/7-day columns are reused and 2/3-day columns come from `temporal`. The first two experiments remain controls.
Missing numeric inputs use training medians plus missingness flags; numeric values are standardized. AdamW uses learning rate 0.001, weight decay 0.001 and batches of 256 hours. The loss is **MAE**, matching the project metric. Epoch selection uses the last 30 dates within the 303 training dates, with at most 120 epochs and patience 12. Preprocessing is fitted on the earlier 273 dates for that selection. Each model is then initialized again and fitted on all 303 dates for the selected number of epochs, with preprocessing refitted on those dates. The final 60 validation dates do not select epochs or enter training. Training uses CPU with seed 42 and deterministic operations; no pretrained weights are used.

MLPs are an established tabular neural-network baseline ([research reference](https://proceedings.neurips.cc/paper/2021/hash/9d86d83f925f2149e9edb0ac3b49229c-Abstract.html)); the four-output constraint is specific to this reconstruction task. There is no assumption that the neural network will outperform CatBoost.


In [5]:
import torch
from torch import nn

# Keep the neural-network implementation and inference helpers in this notebook.
torch.set_num_threads(4)
torch.use_deterministic_algorithms(True)
NN_CONTEXT_COLUMNS = [
    "position_zone", "position_hour", "calendar_day_of_week",
    "calendar_month_sin", "calendar_month_cos",
]

NN_HOURLY_COLUMNS = (
    FEATURE_GROUPS["local"]
    + [name for name in FEATURE_GROUPS["temporal"] if name.startswith("temporal_hourly_change_")]
    + [f"daily_hour_{hour:02d}_price_eur_mwh" for hour in range(1, 25)]
)
NN_CURVE_COLUMNS = [
    f"{'history_smooth' if lag in (1, 7) else 'temporal_smooth'}_deviation_lag_{lag}d_eur_mwh"
    for lag in (1, 2, 3, 7)
]
NN_FEATURE_SETS = {
    "history_only": {"context": False, "temporal": "none"},
    "history_calendar": {"context": True, "temporal": "none"},
    "history_calendar_hourly": {"context": True, "temporal": "hourly"},
    "history_calendar_temporal": {"context": True, "temporal": "curves"},
}


def nn_feature_columns(context=False, temporal="none"):
    columns = history_inputs + (NN_CONTEXT_COLUMNS if context else [])
    if temporal != "none":
        columns = columns + NN_HOURLY_COLUMNS
    if temporal == "curves":
        columns = columns + [name for name in NN_CURVE_COLUMNS if name not in columns]
    return columns


def nn_hour_blocks(rows, context=False, include_target=False, temporal="none"):
    if temporal not in ("none", "hourly", "curves"):
        raise ValueError("Unknown MLP temporal input mode.")
    ordered = rows.sort_values(QUARTER_KEYS).reset_index(drop=True)
    if ordered.duplicated(QUARTER_KEYS).any() or not ordered.groupby(HOUR_KEYS).size().eq(4).all():
        raise ValueError("The MLP requires four unique quarters per hour.")
    np.testing.assert_array_equal(ordered.quarter_index, np.tile(np.arange(1, 5), len(ordered) // 4))
    hours = ordered.iloc[::4]
    numeric = ordered[history_inputs].to_numpy(dtype=float).reshape(-1, 4 * len(history_inputs))
    categorical = np.empty((len(hours), 0))
    if context:
        numeric = np.column_stack([numeric, hours[["calendar_month_sin", "calendar_month_cos"]]])
        zone = hours.position_zone.map({z: i for i, z in enumerate(EXPECTED_ZONES)}).to_numpy(dtype=int)
        categorical = np.column_stack([
            np.eye(4)[zone], np.eye(24)[hours.position_hour.to_numpy(dtype=int) - 1],
            np.eye(7)[hours.calendar_day_of_week.to_numpy(dtype=int)],
        ])
    if temporal != "none":
        numeric = np.column_stack([numeric, hours[NN_HOURLY_COLUMNS].to_numpy(dtype=float)])
    if temporal == "curves":
        if not ordered.groupby(DAY_KEYS).size().eq(96).all():
            raise ValueError("Daily residual-curve inputs require all 96 query quarters per day/zone.")
        # Each value is already a strictly earlier-date residual, joined in Section 2.
        columns = pd.MultiIndex.from_product([NN_CURVE_COLUMNS, range(1, 25), range(1, 5)])
        curves = ordered.set_index(QUARTER_KEYS)[NN_CURVE_COLUMNS].unstack(["hour_index", "quarter_index"])
        curves = curves.reindex(columns=columns)
        daily_inputs = curves.reindex(pd.MultiIndex.from_frame(hours[DAY_KEYS])).to_numpy(dtype=float)
        numeric = np.column_stack([numeric, daily_inputs])
    target = ordered[TARGET_COLUMNS["smooth"]].to_numpy(dtype=float).reshape(-1, 4) if include_target else None
    return ordered, numeric, categorical, target


def fit_nn_preprocessing(numeric):
    median = np.nanmedian(numeric, axis=0)
    median = np.nan_to_num(median, nan=0.0)
    filled = np.where(np.isnan(numeric), median, numeric)
    scaler = StandardScaler().fit(filled)
    return {"median": median, "mean": scaler.mean_, "scale": scaler.scale_}


def nn_matrix(numeric, categorical, preprocessing):
    missing = np.isnan(numeric)
    filled = np.where(missing, preprocessing["median"], numeric)
    scaled = (filled - preprocessing["mean"]) / preprocessing["scale"]
    matrix = np.column_stack([scaled, missing.astype(float), categorical]).astype(np.float32)
    if not np.isfinite(matrix).all():
        raise ValueError("MLP inputs must be finite after preprocessing.")
    return torch.from_numpy(matrix)


class HourlyResidualMLP(nn.Module):
    def __init__(self, input_width):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_width, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.1), nn.Linear(32, 4),
        )

    def forward(self, inputs):
        residuals = self.network(inputs)
        return residuals - residuals.mean(dim=1, keepdim=True)


def train_neural_model(inputs, targets, epochs=120, stopping_inputs=None, stopping_targets=None,
                       model_class=HourlyResidualMLP, batch_size=256):
    torch.manual_seed(42)
    generator = torch.Generator().manual_seed(42)
    model = model_class(inputs.shape[1])
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.001)
    loss = nn.L1Loss()
    target_scale = max(float(np.abs(targets).mean()), 1.0)
    labels = torch.tensor(targets / target_scale, dtype=torch.float32)
    best_mae, best_epoch, stale = np.inf, epochs, 0
    for epoch in range(1, epochs + 1):
        model.train()
        for indices in torch.randperm(len(inputs), generator=generator).split(batch_size):
            optimizer.zero_grad(set_to_none=True)
            predictions = model(inputs[indices])
            batch_labels = labels[indices]
            # TabM trains every member separately; other networks return one prediction.
            if predictions.ndim == batch_labels.ndim + 1:
                batch_labels = batch_labels.unsqueeze(1).expand_as(predictions)
            batch_loss = loss(predictions, batch_labels)
            batch_loss.backward()
            optimizer.step()
        if stopping_inputs is not None:
            model.eval()
            with torch.no_grad():
                predicted = model(stopping_inputs).numpy() * target_scale
            mae = float(np.abs(predicted - stopping_targets).mean())
            if mae < best_mae - 1e-6:
                best_mae, best_epoch, stale = mae, epoch, 0
            else:
                stale += 1
            if stale >= 12:
                break
    model.eval()
    return model, target_scale, best_epoch, best_mae


def predict_mlp(bundle, rows):
    ordered, numeric, categorical, _ = nn_hour_blocks(rows, bundle["context"], temporal=bundle.get("temporal", "none"))
    inputs = nn_matrix(numeric, categorical, bundle["preprocessing"])
    bundle["model"].eval()
    with torch.no_grad():
        residuals = bundle["model"](inputs).numpy().astype(float) * bundle["target_scale"]
    # Restore the caller's quarter-hour order, including when rows are shuffled.
    values = pd.Series(residuals.reshape(-1), index=pd.MultiIndex.from_frame(ordered[QUARTER_KEYS]))
    return values.reindex(pd.MultiIndex.from_frame(rows[QUARTER_KEYS])).to_numpy()


def save_mlp(bundle, path):
    torch.save({
        "state_dict": bundle["model"].state_dict(), "context": bundle["context"],
        "input_width": bundle["model"].network[0].in_features,
        "target_scale": bundle["target_scale"], "epochs": bundle["epochs"],
        "history_columns": history_inputs, "reference_type": "smooth",
        "temporal": bundle.get("temporal", "none"),
        "feature_columns": nn_feature_columns(bundle["context"], bundle.get("temporal", "none")),
        "preprocessing": {key: value.tolist() for key, value in bundle["preprocessing"].items()},
    }, path)


def load_mlp(path):
    saved = torch.load(path, map_location="cpu", weights_only=True)
    if saved["history_columns"] != history_inputs or saved["reference_type"] != "smooth":
        raise ValueError("Checkpoint feature order or reference does not match this notebook.")
    temporal = saved.get("temporal", "none")
    if saved.get("feature_columns", nn_feature_columns(saved["context"], temporal)) != nn_feature_columns(saved["context"], temporal):
        raise ValueError("Checkpoint temporal feature order does not match this notebook.")
    model = HourlyResidualMLP(saved["input_width"])
    model.load_state_dict(saved["state_dict"])
    model.eval()
    return {
        "model": model, "context": saved["context"], "temporal": temporal, "epochs": saved["epochs"],
        "target_scale": saved["target_scale"],
        "preprocessing": {key: np.asarray(value) for key, value in saved["preprocessing"].items()},
    }


In [8]:
nn_stop_start = pd.Timestamp(np.sort(fit_rows.delivery_date.unique())[-30])
nn_earlier = fit_rows[fit_rows.delivery_date.lt(nn_stop_start)]
nn_stopping = fit_rows[fit_rows.delivery_date.ge(nn_stop_start)]
nn_results, nn_models = {}, {}
feature_manifest = {**EXPERIMENTS, "mlp": {name: nn_feature_columns(**settings) for name, settings in NN_FEATURE_SETS.items()}}
(output_dir / "feature_sets.json").write_text(json.dumps(feature_manifest, indent=2) + "\n")
for feature_set, settings in NN_FEATURE_SETS.items():
    context, temporal = settings["context"], settings["temporal"]
    _, early_numeric, early_categorical, early_target = nn_hour_blocks(nn_earlier, context, include_target=True, temporal=temporal)
    _, stop_numeric, stop_categorical, stop_target = nn_hour_blocks(nn_stopping, context, include_target=True, temporal=temporal)
    early_preprocessing = fit_nn_preprocessing(early_numeric)
    _, _, selected_epochs, stopping_mae = train_neural_model(
        nn_matrix(early_numeric, early_categorical, early_preprocessing), early_target,
        stopping_inputs=nn_matrix(stop_numeric, stop_categorical, early_preprocessing),
        stopping_targets=stop_target,
    )
    _, numeric, categorical, target = nn_hour_blocks(fit_rows, context, include_target=True, temporal=temporal)
    preprocessing = fit_nn_preprocessing(numeric)
    model, target_scale, _, _ = train_neural_model(nn_matrix(numeric, categorical, preprocessing), target, epochs=selected_epochs)
    bundle = {
        "model": model, "context": context, "temporal": temporal, "preprocessing": preprocessing,
        "target_scale": target_scale, "epochs": selected_epochs,
    }
    name = f"mlp_smooth_{feature_set}"
    print(f"{name}: selected {selected_epochs} epochs on earlier training dates (internal MAE {stopping_mae:.6f}); refitted on all 303 training dates.")
    result = evaluate_experiment(
        name, validation_rows, validation_hourly, validation_truth,
        predicted_deviations=predict_mlp(bundle, validation_rows), reference_type="smooth",
    )
    result["summary"].update(
        model="mlp", feature_set=feature_set,
        feature_count=len(nn_feature_columns(**settings)),
    )
    nn_results[name], nn_models[name] = result, bundle
    experiments[name], models[name] = result, bundle
    save_mlp(bundle, output_dir / f"{name}.pt")

display(pd.DataFrame([result["summary"] for result in nn_results.values()])[
    ["feature_set", "feature_count", "mae_eur_mwh"]
].round(6))
best_nn = min(nn_results.values(), key=lambda result: result["summary"]["mae_eur_mwh"])
print(f"Best MLP: {best_nn['summary']['experiment']} | validation MAE: {best_nn['summary']['mae_eur_mwh']:.6f} EUR/MWh")
display(best_nn["scores"]["by_zone"].round(6))
worst_nn_zone = best_nn["scores"]["by_zone"].set_index("market_id")["mae_eur_mwh"].idxmax()
worst_nn_hourly = zone_hourly_mae(best_nn["predictions"], validation_truth, worst_nn_zone)
print(f"Worst zone for best MLP: {worst_nn_zone} — validation MAE by hour (hour 1 = 00:00–01:00)")
display(worst_nn_hourly.round(6))


mlp_smooth_history_only: selected 6 epochs on earlier training dates (internal MAE 4.046902); refitted on all 303 training dates.
mlp_smooth_history_only: MAE = 5.063933 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


mlp_smooth_history_calendar: selected 24 epochs on earlier training dates (internal MAE 4.004975); refitted on all 303 training dates.
mlp_smooth_history_calendar: MAE = 5.036322 EUR/MWh | constraints: PASS | max hourly error = 2.51e-11


mlp_smooth_history_calendar_hourly: selected 8 epochs on earlier training dates (internal MAE 3.848185); refitted on all 303 training dates.
mlp_smooth_history_calendar_hourly: MAE = 4.986964 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


mlp_smooth_history_calendar_temporal: selected 3 epochs on earlier training dates (internal MAE 3.962858); refitted on all 303 training dates.
mlp_smooth_history_calendar_temporal: MAE = 5.067417 EUR/MWh | constraints: PASS | max hourly error = 2.51e-11


,feature_set,feature_count,mae_eur_mwh
0,history_only,8,5.063933
1,history_calendar,13,5.036322
2,history_calendar_hourly,42,4.986964
3,history_calendar_temporal,44,5.067417


Best MLP: mlp_smooth_history_calendar_hourly | validation MAE: 4.986964 EUR/MWh


,market_id,mae_eur_mwh
0,ZONE_0,3.713918
1,ZONE_1,6.190824
2,ZONE_2,5.526590
3,ZONE_3,4.516523


Worst zone for best MLP: ZONE_1 — validation MAE by hour (hour 1 = 00:00–01:00)


,hour_index,mae_eur_mwh,validation_days
0,1,4.803643,60
1,2,3.944382,60
2,3,2.535447,60
3,4,2.392013,60
4,5,2.812723,60
5,6,3.719624,60
6,7,4.572034,60
7,8,6.522695,60
8,9,8.216780,60
9,10,8.161879,60


### 4.5 Temporal CNN

The CNN processes a complete day/zone as a sequence of **24 hours** and outputs four smooth-reference residuals per hour. A 1×1 convolution maps inputs to 32 channels; three residual blocks use width-3 convolutions with dilations 1, 2 and 4, ReLU and 10% dropout. A 1×1 output layer produces four quarters. The convolutional receptive field spans up to 15 neighboring hours within the day; zero padding keeps 24 outputs. The four residuals are centered separately within each hour before calculating MAE.

| Experiment | Inputs |
| --- | --- |
| `history_calendar_hourly` | Same source features as the corresponding MLP: `history_smooth`, zone/hour/weekday/month context, local hourly prices, hourly changes and all 24 known hourly prices. |
| `history_calendar_temporal` | Above plus the `temporal` 2/3-day smooth-residual lags. Together with the existing 1/7-day lags, all four historical residual curves are available across the 24-hour input sequence. |

History features occupy their corresponding delivery-hour positions; convolution shares weights across hours. The 24 known hourly prices are also supplied as day-level context. Neighboring hours can be used on both sides because the whole hourly curve and all historical features are available at inference. No current-day quarter-hour labels enter inputs.

The CNN reuses the MLP's training-only median imputation, scaling, missingness flags, MAE loss, AdamW settings, seed 42 and chronological epoch selection. Batches contain 32 complete day/zone sequences. Epochs are selected on the last 30 training dates, using the earlier 273 dates to fit weights and preprocessing; each model is then reset and refitted on all 303 training dates. The same final 60 dates are used only for comparison.


In [6]:
CNN_FEATURE_SETS = {
    "history_calendar_hourly": "hourly", "history_calendar_temporal": "curves",
}


def cnn_day_blocks(rows, temporal="hourly", include_target=False):
    if temporal not in ("hourly", "curves"):
        raise ValueError("CNN temporal inputs must be hourly or curves.")
    ordered, numeric, categorical, target = nn_hour_blocks(
        rows, context=True, include_target=include_target, temporal="hourly",
    )
    if not ordered.groupby(DAY_KEYS).size().eq(96).all():
        raise ValueError("CNN inputs require 24 complete hours per day/zone.")
    np.testing.assert_array_equal(ordered.hour_index.to_numpy().reshape(-1, 96),
                                 np.tile(np.repeat(np.arange(1, 25), 4), (len(ordered) // 96, 1)))
    if temporal == "curves":
        extra = [name for name in NN_CURVE_COLUMNS if name not in history_inputs]
        numeric = np.column_stack([numeric, ordered[extra].to_numpy(dtype=float).reshape(-1, 4 * len(extra))])
    target = target.reshape(-1, 24, 4).transpose(0, 2, 1) if include_target else None
    return ordered, numeric, categorical, target


def cnn_tensor(numeric, categorical, preprocessing):
    hourly = nn_matrix(numeric, categorical, preprocessing)
    return hourly.reshape(-1, 24, hourly.shape[1]).transpose(1, 2).contiguous()


class DailyResidualCNN(nn.Module):
    def __init__(self, input_width):
        super().__init__()
        self.input_layer = nn.Conv1d(input_width, 32, kernel_size=1)
        self.blocks = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(32, 32, kernel_size=3, dilation=dilation, padding=dilation),
                nn.ReLU(), nn.Dropout(0.1), nn.Conv1d(32, 32, kernel_size=1),
            ) for dilation in (1, 2, 4)
        ])
        self.output_layer = nn.Conv1d(32, 4, kernel_size=1)

    def forward(self, inputs):
        state = torch.relu(self.input_layer(inputs))
        for block in self.blocks:
            state = torch.relu(state + block(state))
        residuals = self.output_layer(state)
        return residuals - residuals.mean(dim=1, keepdim=True)


def predict_cnn(bundle, rows):
    ordered, numeric, categorical, _ = cnn_day_blocks(rows, bundle["temporal"])
    inputs = cnn_tensor(numeric, categorical, bundle["preprocessing"])
    bundle["model"].eval()
    with torch.no_grad():
        residuals = bundle["model"](inputs).transpose(1, 2).numpy().astype(float) * bundle["target_scale"]
    values = pd.Series(residuals.reshape(-1), index=pd.MultiIndex.from_frame(ordered[QUARTER_KEYS]))
    return values.reindex(pd.MultiIndex.from_frame(rows[QUARTER_KEYS])).to_numpy()


def save_cnn(bundle, path):
    torch.save({
        "state_dict": bundle["model"].state_dict(), "temporal": bundle["temporal"],
        "input_width": bundle["model"].input_layer.in_channels,
        "target_scale": bundle["target_scale"], "epochs": bundle["epochs"],
        "reference_type": "smooth", "architecture": "daily_residual_cnn_v1",
        "history_columns": history_inputs,
        "feature_columns": nn_feature_columns(context=True, temporal=bundle["temporal"]),
        "preprocessing": {key: value.tolist() for key, value in bundle["preprocessing"].items()},
    }, path)


def load_cnn(path):
    saved = torch.load(path, map_location="cpu", weights_only=True)
    if (saved["architecture"] != "daily_residual_cnn_v1" or saved["reference_type"] != "smooth"
            or saved["history_columns"] != history_inputs
            or saved["feature_columns"] != nn_feature_columns(context=True, temporal=saved["temporal"])):
        raise ValueError("Checkpoint architecture, reference or feature order does not match this notebook.")
    model = DailyResidualCNN(saved["input_width"])
    model.load_state_dict(saved["state_dict"])
    model.eval()
    return {
        "model": model, "temporal": saved["temporal"], "epochs": saved["epochs"],
        "target_scale": saved["target_scale"],
        "preprocessing": {key: np.asarray(value) for key, value in saved["preprocessing"].items()},
    }


In [11]:
cnn_results, cnn_models = {}, {}
feature_manifest["cnn"] = {name: nn_feature_columns(context=True, temporal=mode) for name, mode in CNN_FEATURE_SETS.items()}
(output_dir / "feature_sets.json").write_text(json.dumps(feature_manifest, indent=2) + "\n")
for feature_set, temporal in CNN_FEATURE_SETS.items():
    _, early_numeric, early_categorical, early_target = cnn_day_blocks(nn_earlier, temporal, include_target=True)
    _, stop_numeric, stop_categorical, stop_target = cnn_day_blocks(nn_stopping, temporal, include_target=True)
    early_preprocessing = fit_nn_preprocessing(early_numeric)
    _, _, selected_epochs, stopping_mae = train_neural_model(
        cnn_tensor(early_numeric, early_categorical, early_preprocessing), early_target,
        stopping_inputs=cnn_tensor(stop_numeric, stop_categorical, early_preprocessing),
        stopping_targets=stop_target, model_class=DailyResidualCNN, batch_size=32,
    )
    _, numeric, categorical, target = cnn_day_blocks(fit_rows, temporal, include_target=True)
    preprocessing = fit_nn_preprocessing(numeric)
    model, target_scale, _, _ = train_neural_model(
        cnn_tensor(numeric, categorical, preprocessing), target, epochs=selected_epochs,
        model_class=DailyResidualCNN, batch_size=32,
    )
    bundle = {"model": model, "temporal": temporal, "preprocessing": preprocessing,
              "target_scale": target_scale, "epochs": selected_epochs}
    name = f"cnn_smooth_{feature_set}"
    print(f"{name}: selected {selected_epochs} epochs on earlier training dates (internal MAE {stopping_mae:.6f}); refitted on all 303 training dates.")
    result = evaluate_experiment(
        name, validation_rows, validation_hourly, validation_truth,
        predicted_deviations=predict_cnn(bundle, validation_rows), reference_type="smooth",
    )
    result["summary"].update(model="cnn", feature_set=feature_set,
                             feature_count=len(nn_feature_columns(context=True, temporal=temporal)))
    cnn_results[name], cnn_models[name] = result, bundle
    experiments[name], models[name] = result, bundle
    save_cnn(bundle, output_dir / f"{name}.pt")

display(pd.DataFrame([result["summary"] for result in cnn_results.values()])[
    ["feature_set", "feature_count", "mae_eur_mwh"]
].round(6))
best_cnn = min(cnn_results.values(), key=lambda result: result["summary"]["mae_eur_mwh"])
print(f"Best CNN: {best_cnn['summary']['experiment']} | validation MAE: {best_cnn['summary']['mae_eur_mwh']:.6f} EUR/MWh")
display(best_cnn["scores"]["by_zone"].round(6))
worst_cnn_zone = best_cnn["scores"]["by_zone"].set_index("market_id")["mae_eur_mwh"].idxmax()
worst_cnn_hourly = zone_hourly_mae(best_cnn["predictions"], validation_truth, worst_cnn_zone)
print(f"Worst zone for best CNN: {worst_cnn_zone} — validation MAE by hour (hour 1 = 00:00–01:00)")
display(worst_cnn_hourly.round(6))


cnn_smooth_history_calendar_hourly: selected 30 epochs on earlier training dates (internal MAE 3.775287); refitted on all 303 training dates.
cnn_smooth_history_calendar_hourly: MAE = 4.951240 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


cnn_smooth_history_calendar_temporal: selected 29 epochs on earlier training dates (internal MAE 3.789417); refitted on all 303 training dates.
cnn_smooth_history_calendar_temporal: MAE = 5.003660 EUR/MWh | constraints: PASS | max hourly error = 2.51e-11


,feature_set,feature_count,mae_eur_mwh
0,history_calendar_hourly,42,4.95124
1,history_calendar_temporal,44,5.00366


Best CNN: cnn_smooth_history_calendar_hourly | validation MAE: 4.951240 EUR/MWh


,market_id,mae_eur_mwh
0,ZONE_0,3.689214
1,ZONE_1,6.245462
2,ZONE_2,5.395656
3,ZONE_3,4.474626


Worst zone for best CNN: ZONE_1 — validation MAE by hour (hour 1 = 00:00–01:00)


,hour_index,mae_eur_mwh,validation_days
0,1,4.780418,60
1,2,3.659669,60
2,3,2.255834,60
3,4,2.176631,60
4,5,2.822811,60
5,6,4.235246,60
6,7,5.034647,60
7,8,6.711051,60
8,9,8.909729,60
9,10,8.010684,60


### 4.6 TabM

Uses the official [TabM package](https://github.com/yandex-research/tabm), initialized from scratch on official challenge data. Fixed settings: 16 members sharing weights, two 64-unit blocks and 10% dropout. The two input selections are `history_calendar` and `history_calendar_hourly`, matching the MLP's source columns, packing and preprocessing.

Each member predicts four smooth-reference residuals, centered within the hour. Training minimizes the average **MAE of individual members**; inference averages their centered predictions, following the official implementation. Epoch selection evaluates the averaged prediction on the earlier internal chronological split. Training-only medians/scaling, AdamW, seed 42, 120-epoch cap and patience 12 are shared with the other neural networks. Refitting uses all 303 training dates for the selected number of epochs; the final 60 dates remain the comparison set. No pretrained weights or external data are used.


In [8]:
from tabm import TabM

TABM_PARAMETERS = {"arch_type": "tabm", "k": 16, "n_blocks": 2, "d_block": 64, "dropout": 0.1}
TABM_FEATURE_SETS = {"history_calendar": "none", "history_calendar_hourly": "hourly"}


class HourlyResidualTabM(nn.Module):
    def __init__(self, input_width, parameters=None):
        super().__init__()
        self.input_width = input_width
        self.ensemble = TabM.make(n_num_features=input_width, d_out=4, **(parameters or TABM_PARAMETERS))

    def forward(self, inputs):
        members = self.ensemble(inputs)
        members = members - members.mean(dim=-1, keepdim=True)
        return members if self.training else members.mean(dim=1)


def save_tabm(bundle, path):
    torch.save({
        "state_dict": bundle["model"].state_dict(), "parameters": TABM_PARAMETERS,
        "input_width": bundle["model"].input_width, "context": True,
        "temporal": bundle["temporal"], "target_scale": bundle["target_scale"],
        "epochs": bundle["epochs"], "reference_type": "smooth", "history_columns": history_inputs,
        "feature_columns": nn_feature_columns(context=True, temporal=bundle["temporal"]),
        "preprocessing": {key: value.tolist() for key, value in bundle["preprocessing"].items()},
    }, path)


def load_tabm(path):
    saved = torch.load(path, map_location="cpu", weights_only=True)
    if (saved["reference_type"] != "smooth" or saved["history_columns"] != history_inputs
            or saved["feature_columns"] != nn_feature_columns(context=True, temporal=saved["temporal"])):
        raise ValueError("Checkpoint reference or feature order does not match this notebook.")
    model = HourlyResidualTabM(saved["input_width"], saved["parameters"])
    model.load_state_dict(saved["state_dict"])
    model.eval()
    return {"model": model, "context": True, "temporal": saved["temporal"],
            "target_scale": saved["target_scale"], "epochs": saved["epochs"],
            "preprocessing": {key: np.asarray(value) for key, value in saved["preprocessing"].items()}}


In [10]:
tabm_results, tabm_models = {}, {}
feature_manifest["tabm"] = {name: nn_feature_columns(context=True, temporal=mode) for name, mode in TABM_FEATURE_SETS.items()}
(output_dir / "feature_sets.json").write_text(json.dumps(feature_manifest, indent=2) + "\n")
for feature_set, temporal in TABM_FEATURE_SETS.items():
    _, early_numeric, early_categorical, early_target = nn_hour_blocks(nn_earlier, True, True, temporal)
    _, stop_numeric, stop_categorical, stop_target = nn_hour_blocks(nn_stopping, True, True, temporal)
    early_preprocessing = fit_nn_preprocessing(early_numeric)
    _, _, selected_epochs, stopping_mae = train_neural_model(
        nn_matrix(early_numeric, early_categorical, early_preprocessing), early_target,
        stopping_inputs=nn_matrix(stop_numeric, stop_categorical, early_preprocessing),
        stopping_targets=stop_target, model_class=HourlyResidualTabM,
    )
    _, numeric, categorical, target = nn_hour_blocks(fit_rows, True, True, temporal)
    preprocessing = fit_nn_preprocessing(numeric)
    model, target_scale, _, _ = train_neural_model(
        nn_matrix(numeric, categorical, preprocessing), target, epochs=selected_epochs,
        model_class=HourlyResidualTabM,
    )
    bundle = {"model": model, "context": True, "temporal": temporal, "preprocessing": preprocessing,
              "target_scale": target_scale, "epochs": selected_epochs}
    name = f"tabm_smooth_{feature_set}"
    print(f"{name}: selected {selected_epochs} epochs on earlier training dates (internal MAE {stopping_mae:.6f}); refitted on all 303 training dates.")
    result = evaluate_experiment(name, validation_rows, validation_hourly, validation_truth,
                                 predicted_deviations=predict_mlp(bundle, validation_rows), reference_type="smooth")
    result["summary"].update(model="tabm", feature_set=feature_set,
                             feature_count=len(nn_feature_columns(context=True, temporal=temporal)))
    tabm_results[name], tabm_models[name] = result, bundle
    experiments[name], models[name] = result, bundle
    save_tabm(bundle, output_dir / f"{name}.pt")

display(pd.DataFrame([result["summary"] for result in tabm_results.values()])[
    ["feature_set", "feature_count", "mae_eur_mwh"]
].round(6))
best_tabm = min(tabm_results.values(), key=lambda result: result["summary"]["mae_eur_mwh"])
print(f"Best TabM: {best_tabm['summary']['experiment']} | validation MAE: {best_tabm['summary']['mae_eur_mwh']:.6f} EUR/MWh")
display(best_tabm["scores"]["by_zone"].round(6))
worst_tabm_zone = best_tabm["scores"]["by_zone"].set_index("market_id")["mae_eur_mwh"].idxmax()
worst_tabm_hourly = zone_hourly_mae(best_tabm["predictions"], validation_truth, worst_tabm_zone)
print(f"Worst zone for best TabM: {worst_tabm_zone} — validation MAE by hour (hour 1 = 00:00–01:00)")
display(worst_tabm_hourly.round(6))


tabm_smooth_history_calendar: selected 79 epochs on earlier training dates (internal MAE 3.995261); refitted on all 303 training dates.
tabm_smooth_history_calendar: MAE = 5.038548 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


tabm_smooth_history_calendar_hourly: selected 72 epochs on earlier training dates (internal MAE 3.765806); refitted on all 303 training dates.
tabm_smooth_history_calendar_hourly: MAE = 4.878572 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


,feature_set,feature_count,mae_eur_mwh
0,history_calendar,13,5.038548
1,history_calendar_hourly,42,4.878572


Best TabM: tabm_smooth_history_calendar_hourly | validation MAE: 4.878572 EUR/MWh


,market_id,mae_eur_mwh
0,ZONE_0,3.625629
1,ZONE_1,6.045589
2,ZONE_2,5.394838
3,ZONE_3,4.448232


Worst zone for best TabM: ZONE_1 — validation MAE by hour (hour 1 = 00:00–01:00)


,hour_index,mae_eur_mwh,validation_days
0,1,4.725523,60
1,2,3.605248,60
2,3,2.201612,60
3,4,2.085822,60
4,5,2.614314,60
5,6,3.483660,60
6,7,4.567997,60
7,8,6.183352,60
8,9,8.150499,60
9,10,8.175282,60


### 4.7 Fixed blends

Two recipes combine final quarter-hour prices with equal weights, aligned by the four output keys:

- `cnn_catboost_history`: 50% `cnn_smooth_history_calendar_hourly` + 50% `catboost_smooth_history_only_tuned`.
- `cnn_catboost_tabm`: one third each of those models and `tabm_smooth_history_calendar_hourly`.

Recipes, component choices and weights are fixed. Every component already respects the supplied hourly means, so its convex average does too; the shared final correction and official checks are still applied. The exported recipe file references component checkpoints and supports inference without retraining. The final 60 dates are an already inspected comparison set.

The 50/50 recipe also receives three retrospective chronological checks inside the 303 training dates. Each check scores 30 complete dates after an expanding training window. CNN epoch selection uses the last 30 dates *within that fold's training window*, then refits on that window; CatBoost retains its selected settings. Historical features may use actual observations from strictly earlier dates, including earlier scoring dates, while weights and preprocessing stay fixed within each scoring block. These are additional robustness checks of an already selected recipe, not untouched tests of model selection. Fold exports are checked and scored with the same helpers, separately from the final-holdout summary.


In [11]:
BLEND_RECIPES = {
    "cnn_catboost_history": ["cnn_smooth_history_calendar_hourly", "catboost_smooth_history_only_tuned"],
    "cnn_catboost_tabm": ["cnn_smooth_history_calendar_hourly", "catboost_smooth_history_only_tuned", "tabm_smooth_history_calendar_hourly"],
}


def blend_prediction_frames(frames, hourly_inputs, weights=None):
    if not len(frames):
        raise ValueError("Blend needs at least one component.")
    weights = np.full(len(frames), 1 / len(frames)) if weights is None else np.asarray(weights, dtype=float)
    if (not len(frames) or weights.shape != (len(frames),) or not np.isfinite(weights).all()
            or (weights < 0).any() or not np.isclose(weights.sum(), 1, atol=1e-12, rtol=0)):
        raise ValueError("Blend needs finite, nonnegative weights summing to one.")
    total, index = None, None
    for frame, weight in zip(frames, weights):
        validate_prediction_output(frame, hourly_inputs)
        frame = frame.copy()
        frame["delivery_date"] = pd.to_datetime(frame["delivery_date"])
        values = frame.set_index(QUARTER_KEYS)[PREDICTION_COLUMN].sort_index()
        if index is None:
            index, total = values.index, np.zeros(len(values))
        if not values.index.equals(index):
            raise ValueError("Blend components must have exactly matching keys.")
        total += weight * values.to_numpy()
    output = pd.Series(total, index=index, name=PREDICTION_COLUMN).reset_index()
    validate_prediction_output(output, hourly_inputs)
    return output


def evaluate_blend(name, frames, rows, hourly_inputs, truth, output_dir=output_dir):
    prediction = blend_prediction_frames(frames, hourly_inputs)
    aligned = rows[QUARTER_KEYS + [REFERENCE_COLUMNS["smooth"]]].merge(prediction, on=QUARTER_KEYS, validate="one_to_one")
    deviations = aligned[PREDICTION_COLUMN] - aligned[REFERENCE_COLUMNS["smooth"]]
    return evaluate_experiment(name, rows, hourly_inputs, truth, deviations.to_numpy(),
                               reference_type="smooth", output_dir=output_dir)


def predict_blend(rows, hourly_inputs, components, fitted_models, weights=None):
    frames = []
    for name in components:
        fitted = fitted_models[name]
        if name.startswith("cnn_"):
            deviations = predict_cnn(fitted, rows)
        elif name.startswith("tabm_"):
            deviations = predict_mlp(fitted, rows)
        elif name == "catboost_smooth_history_only_tuned":
            deviations = fitted.predict(rows[history_inputs])
        else:
            raise ValueError(f"Unsupported blend component: {name}")
        frames.append(make_predictions(rows, hourly_inputs, deviations, reference_type="smooth"))
    return blend_prediction_frames(frames, hourly_inputs, weights)


def load_blend(path, recipe):
    saved = json.loads(Path(path).read_text())[recipe]
    fitted = {}
    for component in saved["components"]:
        model_path = Path(path).parent / component["checkpoint"]
        name = component["experiment"]
        if name.startswith("cnn_"):
            fitted[name] = load_cnn(model_path)
        elif name.startswith("tabm_"):
            fitted[name] = load_tabm(model_path)
        else:
            fitted[name] = CatBoostRegressor().load_model(str(model_path))
    return saved["members"], saved["weights"], fitted


In [12]:
blend_results, blend_manifest = {}, {}
for recipe, members in BLEND_RECIPES.items():
    name = f"blend_smooth_{recipe}"
    result = evaluate_blend(name, [experiments[member]["predictions"] for member in members],
                            validation_rows, validation_hourly, validation_truth)
    result["summary"].update(model="blend", feature_set=recipe, feature_count=len(nn_feature_columns(True, "hourly")))
    blend_results[name], experiments[name] = result, result
    blend_manifest[recipe] = {
        "members": members, "weights": [1 / len(members)] * len(members), "reference_type": "smooth",
        "components": [{"experiment": member, "checkpoint": f"{member}.{'cbm' if member.startswith('catboost_') else 'pt'}"} for member in members],
    }
(output_dir / "blend_recipes.json").write_text(json.dumps(blend_manifest, indent=2) + "\n")
display(pd.DataFrame([result["summary"] for result in blend_results.values()])[["feature_set", "mae_eur_mwh"]].round(6))
best_blend = min(blend_results.values(), key=lambda result: result["summary"]["mae_eur_mwh"])
print(f"Best blend: {best_blend['summary']['experiment']} | validation MAE: {best_blend['summary']['mae_eur_mwh']:.6f} EUR/MWh")
display(best_blend["scores"]["by_zone"].round(6))
worst_blend_zone = best_blend["scores"]["by_zone"].set_index("market_id")["mae_eur_mwh"].idxmax()
worst_blend_hourly = zone_hourly_mae(best_blend["predictions"], validation_truth, worst_blend_zone)
print(f"Worst zone for best blend: {worst_blend_zone} — validation MAE by hour (hour 1 = 00:00–01:00)")
display(worst_blend_hourly.round(6))


blend_smooth_cnn_catboost_history: MAE = 4.823639 EUR/MWh | constraints: PASS | max hourly error = 5e-11


blend_smooth_cnn_catboost_tabm: MAE = 4.784831 EUR/MWh | constraints: PASS | max hourly error = 5e-11


,feature_set,mae_eur_mwh
0,cnn_catboost_history,4.823639
1,cnn_catboost_tabm,4.784831


Best blend: blend_smooth_cnn_catboost_tabm | validation MAE: 4.784831 EUR/MWh


,market_id,mae_eur_mwh
0,ZONE_0,3.465207
1,ZONE_1,5.993508
2,ZONE_2,5.331533
3,ZONE_3,4.349077


Worst zone for best blend: ZONE_1 — validation MAE by hour (hour 1 = 00:00–01:00)


,hour_index,mae_eur_mwh,validation_days
0,1,4.672565,60
1,2,3.601135,60
2,3,2.225235,60
3,4,2.040334,60
4,5,2.682338,60
5,6,3.715340,60
6,7,4.520900,60
7,8,6.330327,60
8,9,8.045444,60
9,10,8.049149,60


In [13]:
# Additional expanding-window checks of the fixed 50/50 CNN–CatBoost recipe.
blend_cv_results = []
cv_directory = output_dir / "blend_cv"
training_dates = np.sort(fit_rows.delivery_date.unique())
for fold, end in enumerate((len(training_dates) - 60, len(training_dates) - 30, len(training_dates)), start=1):
    scoring_dates = training_dates[end - 30:end]
    fold_training = fit_rows[fit_rows.delivery_date.lt(scoring_dates[0])].copy()
    fold_query = feature_table[feature_table.delivery_date.isin(scoring_dates)].copy()
    fold_hourly = train_hourly[train_hourly.delivery_date.isin(scoring_dates)].copy()
    fold_truth = prices[prices.delivery_date.isin(scoring_dates)].copy()
    stop_start = pd.Timestamp(np.sort(fold_training.delivery_date.unique())[-30])
    early = fold_training[fold_training.delivery_date.lt(stop_start)]
    stopping = fold_training[fold_training.delivery_date.ge(stop_start)]
    _, en, ec, ey = cnn_day_blocks(early, "hourly", include_target=True)
    _, sn, sc, sy = cnn_day_blocks(stopping, "hourly", include_target=True)
    early_preprocessing = fit_nn_preprocessing(en)
    _, _, epochs, _ = train_neural_model(
        cnn_tensor(en, ec, early_preprocessing), ey,
        stopping_inputs=cnn_tensor(sn, sc, early_preprocessing), stopping_targets=sy,
        model_class=DailyResidualCNN, batch_size=32,
    )
    _, numeric, categorical, target = cnn_day_blocks(fold_training, "hourly", include_target=True)
    preprocessing = fit_nn_preprocessing(numeric)
    model, scale, _, _ = train_neural_model(cnn_tensor(numeric, categorical, preprocessing), target,
                                            epochs=epochs, model_class=DailyResidualCNN, batch_size=32)
    cnn = {"model": model, "temporal": "hourly", "preprocessing": preprocessing, "target_scale": scale, "epochs": epochs}
    catboost = make_catboost_model(history_inputs, CATBOOST_SELECTED_PARAMETERS)
    catboost.fit(fold_training[history_inputs], fold_training[TARGET_COLUMNS["smooth"]])
    cnn_result = evaluate_experiment(f"fold_{fold}_cnn", fold_query, fold_hourly, fold_truth,
                                     predict_cnn(cnn, fold_query), reference_type="smooth", output_dir=cv_directory)
    catboost_result = evaluate_experiment(f"fold_{fold}_catboost", fold_query, fold_hourly, fold_truth,
                                          catboost.predict(fold_query[history_inputs]), reference_type="smooth", output_dir=cv_directory)
    blend = evaluate_blend(f"fold_{fold}_blend", [cnn_result["predictions"], catboost_result["predictions"]],
                           fold_query, fold_hourly, fold_truth, output_dir=cv_directory)
    record = {
        "fold": fold, "training_dates": fold_training.delivery_date.nunique(),
        "training_end": fold_training.delivery_date.max(), "epoch_selection_start": stop_start,
        "scoring_start": pd.Timestamp(scoring_dates[0]), "scoring_end": pd.Timestamp(scoring_dates[-1]),
        "cnn_epochs": epochs, "cnn_mae": cnn_result["scores"]["mae_eur_mwh"],
        "catboost_mae": catboost_result["scores"]["mae_eur_mwh"], "blend_mae": blend["scores"]["mae_eur_mwh"],
        "max_hourly_error_eur_mwh": max(result["summary"]["max_hourly_error_eur_mwh"] for result in (cnn_result, catboost_result, blend)),
        "constraints_passed": all(result["summary"]["constraints_passed"] for result in (cnn_result, catboost_result, blend)),
    }
    blend_cv_results.append(record)
    pd.DataFrame(blend_cv_results).to_csv(output_dir / "blend_chronological_folds.csv", index=False)
    print(f"Chronological fold {fold}: {record['scoring_start'].date()}–{record['scoring_end'].date()} | CNN {record['cnn_mae']:.6f}, CatBoost {record['catboost_mae']:.6f}, blend {record['blend_mae']:.6f}")
blend_cv_results = pd.DataFrame(blend_cv_results)
display(blend_cv_results[["fold", "training_dates", "scoring_start", "scoring_end", "cnn_epochs", "cnn_mae", "catboost_mae", "blend_mae", "constraints_passed"]].round({"cnn_mae": 6, "catboost_mae": 6, "blend_mae": 6}))
print("Mean of the three equally sized chronological folds:")
display(blend_cv_results[["cnn_mae", "catboost_mae", "blend_mae"]].mean().round(6).to_frame("mae_eur_mwh"))


fold_1_cnn: MAE = 3.948459 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11
fold_1_catboost: MAE = 4.144769 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


fold_1_blend: MAE = 3.990029 EUR/MWh | constraints: PASS | max hourly error = 5e-11
Chronological fold 1: 2026-05-04–2026-06-02 | CNN 3.948459, CatBoost 4.144769, blend 3.990029


fold_2_cnn: MAE = 4.234442 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11
fold_2_catboost: MAE = 4.444542 EUR/MWh | constraints: PASS | max hourly error = 2.51e-11


fold_2_blend: MAE = 4.073423 EUR/MWh | constraints: PASS | max hourly error = 5e-11
Chronological fold 2: 2026-06-03–2026-07-02 | CNN 4.234442, CatBoost 4.444542, blend 4.073423


fold_3_cnn: MAE = 3.906480 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11
fold_3_catboost: MAE = 4.049028 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


fold_3_blend: MAE = 3.832697 EUR/MWh | constraints: PASS | max hourly error = 5e-11
Chronological fold 3: 2026-07-03–2026-08-01 | CNN 3.906480, CatBoost 4.049028, blend 3.832697


,fold,training_dates,scoring_start,scoring_end,cnn_epochs,cnn_mae,catboost_mae,blend_mae,constraints_passed
0,1,213,2026-05-04,2026-06-02,5,3.948459,4.144769,3.990029,True
1,2,243,2026-06-03,2026-07-02,45,4.234442,4.444542,4.073423,True
2,3,273,2026-07-03,2026-08-01,21,3.906480,4.049028,3.832697,True


Mean of the three equally sized chronological folds:


,mae_eur_mwh
cnn_mae,4.029794
catboost_mae,4.212780
blend_mae,3.965383


### 4.8 Dilated TCN with continuous history

This model reads two sequences directly: the previous **14 or 28 calendar days of raw quarter-hour prices**, and the current day's **24 supplied hourly prices**. It uses the sequence channels defined in Section 2. No rolling residual summaries or selected daily lags enter its inputs. Zone weights are shared across all four zones.

A learned width-4, stride-4 convolution maps each historical hour's four quarters to 24 channels. Nine causal residual blocks use kernel width 3 and dilations **1, 2, 4, 8, 16, 32, 64, 128, 256**. Their receptive field is 1,023 hourly patches, covering both lookback windows. A separate convolutional encoder reads the known current-day curve. Four-head cross-attention lets each current-day hour select among the encoded historical hours. A shared decoder returns four smooth-reference residuals for each of the 24 hours; outputs are centered separately within each hour.

Calendar gaps retain their positions on the time grid, including excluded DST dates. Unavailable observations and dates before the dataset begins have an observation mask of zero. Missing historical prices become zero after training-only price standardization. Historical hours with no observed quarters are masked out of attention; an all-missing history has a zero-context fallback. No forward filling or calendar compression is used.

Both lookbacks use the same fixed 24-channel architecture, 10% dropout, AdamW learning rate 0.001, weight decay 0.001, gradient norm cap 1, batches of 16 day/zone sequences and seed 42. Epoch selection uses the earlier 273 training dates and the last 30 training dates for stopping, with a 120-epoch cap and patience 12. Price scaling is fitted only on input sequences from the fitting dates. Each model is reset and refitted on all 303 training dates for its selected number of epochs, then scored on the same final 60 dates. Historical inputs are selected separately for each query date and always end before that date, including during validation.

The smooth reference is used only for the residual target and final reconstruction. Raw-history packing, preprocessing, training, checkpoint loading and inference are all implemented below. In the experiment results, this model's `feature_count` is the total number of named input channels across its two sequences.


In [7]:
from torch.nn import functional as F

TCN_HISTORY_CHANNELS = [
    "sequence_price", "sequence_observed", "sequence_time_sin", "sequence_time_cos",
    "sequence_weekday_sin", "sequence_weekday_cos", "sequence_month_sin", "sequence_month_cos",
    "sequence_age_days",
]
TCN_DAY_CHANNELS = [
    "sequence_hourly_price", "sequence_time_sin", "sequence_time_cos",
    "sequence_weekday_sin", "sequence_weekday_cos", "sequence_month_sin", "sequence_month_cos",
] + [f"sequence_zone_{zone}" for zone in EXPECTED_ZONES]
TCN_PARAMETERS = {"width": 24, "dilations": [1, 2, 4, 8, 16, 32, 64, 128, 256], "dropout": 0.1, "heads": 4}


def tcn_sequences(hourly_inputs, history_15min, history_days, target_prices=None):
    if history_days not in (14, 28):
        raise ValueError("TCN history_days must be 14 or 28.")
    hourly = hourly_inputs.copy()
    hourly["delivery_date"] = pd.to_datetime(hourly["delivery_date"])
    rows = smooth_reference(hourly).sort_values(QUARTER_KEYS).reset_index(drop=True)
    curve = hourly.pivot(index=DAY_KEYS, columns="hour_index", values="hourly_price_eur_mwh").reindex(columns=range(1, 25)).sort_index()
    sample_keys = curve.index.to_frame(index=False)
    sample_count = len(sample_keys)
    steps = history_days * 96
    source = history_15min[QUARTER_KEYS + ["price_eur_mwh"]].copy()
    source["delivery_date"] = pd.to_datetime(source["delivery_date"])
    if source.duplicated(QUARTER_KEYS).any() or not source.hour_index.isin(range(1, 25)).all() or not source.quarter_index.isin(range(1, 5)).all():
        raise ValueError("Historical observations need unique, valid quarter-hour keys.")
    source["quarter_of_day"] = (source.hour_index - 1) * 4 + source.quarter_index
    daily_history = source.pivot(index=DAY_KEYS, columns="quarter_of_day", values="price_eur_mwh").reindex(columns=range(1, 97))
    dates = sample_keys.delivery_date.to_numpy(dtype="datetime64[D]")[:, None] - np.arange(history_days, 0, -1).astype("timedelta64[D]")[None, :]
    index = pd.MultiIndex.from_arrays([dates.reshape(-1), np.repeat(sample_keys.market_id.to_numpy(), history_days)], names=DAY_KEYS)
    values = daily_history.reindex(index).to_numpy(dtype=np.float32).reshape(sample_count, steps)
    history = np.empty((sample_count, len(TCN_HISTORY_CHANNELS), steps), dtype=np.float32)
    history[:, 0] = values
    history[:, 1] = np.isfinite(values)
    time_angle = np.tile(2 * np.pi * np.arange(96) / 96, history_days)
    history[:, 2], history[:, 3] = np.sin(time_angle), np.cos(time_angle)
    historical_dates = pd.DatetimeIndex(dates.reshape(-1))
    weekday = np.repeat(historical_dates.dayofweek.to_numpy().reshape(sample_count, history_days), 96, axis=1)
    month = np.repeat(historical_dates.month.to_numpy().reshape(sample_count, history_days), 96, axis=1)
    history[:, 4], history[:, 5] = np.sin(2 * np.pi * weekday / 7), np.cos(2 * np.pi * weekday / 7)
    history[:, 6], history[:, 7] = np.sin(2 * np.pi * (month - 1) / 12), np.cos(2 * np.pi * (month - 1) / 12)
    history[:, 8] = (np.arange(steps) - steps) / 96
    day = np.empty((sample_count, len(TCN_DAY_CHANNELS), 24), dtype=np.float32)
    day[:, 0] = curve.to_numpy(dtype=np.float32)
    angle = 2 * np.pi * np.arange(24) / 24
    day[:, 1], day[:, 2] = np.sin(angle), np.cos(angle)
    weekday = sample_keys.delivery_date.dt.dayofweek.to_numpy()[:, None]
    month = sample_keys.delivery_date.dt.month.to_numpy()[:, None]
    day[:, 3], day[:, 4] = np.sin(2 * np.pi * weekday / 7), np.cos(2 * np.pi * weekday / 7)
    day[:, 5], day[:, 6] = np.sin(2 * np.pi * (month - 1) / 12), np.cos(2 * np.pi * (month - 1) / 12)
    for i, zone in enumerate(EXPECTED_ZONES):
        day[:, 7 + i] = sample_keys.market_id.eq(zone).to_numpy()[:, None]
    target = None
    if target_prices is not None:
        labels = target_prices[QUARTER_KEYS + ["price_eur_mwh"]].copy()
        labels["delivery_date"] = pd.to_datetime(labels["delivery_date"])
        matched = rows.merge(labels, on=QUARTER_KEYS, how="left", validate="one_to_one")
        target = (matched.price_eur_mwh - matched.reference_smooth_price_eur_mwh).to_numpy().reshape(-1, 24, 4).transpose(0, 2, 1)
        if not np.isfinite(target).all():
            raise ValueError("TCN targets must be complete and finite.")
    return {"rows": rows, "sample_keys": sample_keys, "history": history, "day": day, "target": target}


def fit_tcn_preprocessing(data):
    observed = data["history"][:, 0][data["history"][:, 1].astype(bool)]
    values = np.concatenate([observed, data["day"][:, 0].reshape(-1)]).astype(float)
    return {"price_mean": float(values.mean()), "price_scale": max(float(values.std()), 1.0)}


def tcn_tensors(data, preprocessing):
    history, day = data["history"].copy(), data["day"].copy()
    mean, scale = preprocessing["price_mean"], preprocessing["price_scale"]
    history[:, 0] = (np.nan_to_num(history[:, 0], nan=mean, posinf=mean, neginf=mean) - mean) / scale
    history[:, 8] /= 28
    day[:, 0] = (day[:, 0] - mean) / scale
    if not np.isfinite(history).all() or not np.isfinite(day).all():
        raise ValueError("TCN inputs must be finite after preprocessing.")
    return torch.from_numpy(history), torch.from_numpy(day)


class CausalHistoryBlock(nn.Module):
    def __init__(self, width, dilation, dropout):
        super().__init__()
        self.left_padding = 2 * dilation
        self.convolution = nn.Conv1d(width, width, 3, dilation=dilation)
        self.output = nn.Sequential(nn.ReLU(), nn.Dropout(dropout), nn.Conv1d(width, width, 1))

    def forward(self, state):
        return torch.relu(state + self.output(self.convolution(F.pad(state, (self.left_padding, 0)))))


class ContinuousHistoryTCN(nn.Module):
    def __init__(self, parameters=None):
        super().__init__()
        parameters = parameters or TCN_PARAMETERS
        width, dropout = parameters["width"], parameters["dropout"]
        self.history_patch = nn.Conv1d(len(TCN_HISTORY_CHANNELS), width, 4, stride=4)
        self.history_blocks = nn.Sequential(*[CausalHistoryBlock(width, d, dropout) for d in parameters["dilations"]])
        self.history_norm = nn.LayerNorm(width)
        self.day_encoder = nn.Sequential(nn.Conv1d(len(TCN_DAY_CHANNELS), width, 3, padding=1), nn.ReLU(),
                                         nn.Conv1d(width, width, 3, padding=1), nn.ReLU())
        self.attention = nn.MultiheadAttention(width, parameters["heads"], dropout=dropout, batch_first=True)
        self.decoder = nn.Sequential(nn.Linear(2 * width, 2 * width), nn.ReLU(), nn.Dropout(dropout), nn.Linear(2 * width, 4))

    def forward(self, history, day):
        observed_hours = history[:, 1].reshape(len(history), -1, 4).bool().any(dim=-1)
        encoded = self.history_norm(self.history_blocks(torch.relu(self.history_patch(history))).transpose(1, 2))
        encoded = encoded.masked_fill(~observed_hours[..., None], 0)
        padding = ~observed_hours
        # One zero-valued key keeps attention finite when the entire history is missing.
        padding = padding.clone()
        padding[~observed_hours.any(dim=1), -1] = False
        queries = self.day_encoder(day).transpose(1, 2)
        context, _ = self.attention(queries, encoded, encoded, key_padding_mask=padding, need_weights=False)
        context = context.masked_fill(~observed_hours.any(dim=1)[:, None, None], 0)
        residuals = self.decoder(torch.cat([queries, context], dim=-1)).transpose(1, 2)
        return residuals - residuals.mean(dim=1, keepdim=True)


def tcn_forward_batches(model, inputs, target_scale=1.0):
    model.eval()
    with torch.no_grad():
        return np.concatenate([model(inputs[0][i:i + 16], inputs[1][i:i + 16]).numpy().astype(float) * target_scale
                               for i in range(0, len(inputs[0]), 16)])


def train_tcn(inputs, targets, epochs=120, stopping_inputs=None, stopping_targets=None):
    torch.manual_seed(42)
    generator = torch.Generator().manual_seed(42)
    model = ContinuousHistoryTCN()
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.001)
    scale = max(float(np.abs(targets).mean()), 1.0)
    labels = torch.tensor(targets / scale, dtype=torch.float32)
    best_epoch, best_mae, stale = epochs, np.inf, 0
    for epoch in range(1, epochs + 1):
        model.train()
        for indices in torch.randperm(len(labels), generator=generator).split(16):
            optimizer.zero_grad(set_to_none=True)
            loss = F.l1_loss(model(inputs[0][indices], inputs[1][indices]), labels[indices])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        if stopping_inputs is not None:
            mae = float(np.abs(tcn_forward_batches(model, stopping_inputs, scale) - stopping_targets).mean())
            if mae < best_mae - 1e-6:
                best_epoch, best_mae, stale = epoch, mae, 0
            else:
                stale += 1
            if stale >= 12:
                break
    model.eval()
    return model, scale, best_epoch, best_mae


def tcn_deviations(bundle, data):
    prediction = tcn_forward_batches(bundle["model"], tcn_tensors(data, bundle["preprocessing"]), bundle["target_scale"])
    return prediction.transpose(0, 2, 1).reshape(-1)


def predict_tcn(bundle, hourly_inputs, history_15min):
    data = tcn_sequences(hourly_inputs, history_15min, bundle["history_days"])
    return make_predictions(data["rows"], hourly_inputs, tcn_deviations(bundle, data), reference_type="smooth")


def save_tcn(bundle, path):
    torch.save({"state_dict": bundle["model"].state_dict(), "parameters": TCN_PARAMETERS,
                "history_days": bundle["history_days"], "preprocessing": bundle["preprocessing"],
                "target_scale": bundle["target_scale"], "epochs": bundle["epochs"],
                "history_channels": TCN_HISTORY_CHANNELS, "day_channels": TCN_DAY_CHANNELS,
                "reference_type": "smooth", "architecture": "continuous_history_tcn_v1"}, path)


def load_tcn(path):
    saved = torch.load(path, map_location="cpu", weights_only=True)
    if (saved["architecture"] != "continuous_history_tcn_v1" or saved["reference_type"] != "smooth"
            or saved["history_channels"] != TCN_HISTORY_CHANNELS or saved["day_channels"] != TCN_DAY_CHANNELS):
        raise ValueError("TCN checkpoint architecture, reference or channel order does not match.")
    model = ContinuousHistoryTCN(saved["parameters"])
    model.load_state_dict(saved["state_dict"])
    model.eval()
    return {"model": model, "history_days": saved["history_days"], "preprocessing": saved["preprocessing"],
            "target_scale": saved["target_scale"], "epochs": saved["epochs"]}


In [9]:
tcn_results, tcn_models = {}, {}
tcn_training_summary = []
feature_manifest["tcn"] = {f"history_{days}d": {"history_days": days, "history_channels": TCN_HISTORY_CHANNELS,
                                               "day_channels": TCN_DAY_CHANNELS} for days in (14, 28)}
(output_dir / "feature_sets.json").write_text(json.dumps(feature_manifest, indent=2) + "\n")
for days in (14, 28):
    early_hourly = train_hourly[train_hourly.delivery_date.lt(nn_stop_start)]
    stop_hourly = train_hourly[train_hourly.delivery_date.ge(nn_stop_start) & train_hourly.delivery_date.lt(validation_start)]
    fitting_hourly = train_hourly[train_hourly.delivery_date.lt(validation_start)]
    early_data = tcn_sequences(early_hourly, prices, days, target_prices=prices)
    stop_data = tcn_sequences(stop_hourly, prices, days, target_prices=prices)
    early_preprocessing = fit_tcn_preprocessing(early_data)
    _, _, epochs, stopping_mae = train_tcn(
        tcn_tensors(early_data, early_preprocessing), early_data["target"],
        stopping_inputs=tcn_tensors(stop_data, early_preprocessing), stopping_targets=stop_data["target"],
    )
    tcn_training_summary.append({"history_days": days, "selected_epochs": epochs, "internal_mae_eur_mwh": stopping_mae})
    pd.DataFrame(tcn_training_summary).to_csv(output_dir / "tcn_training_summary.csv", index=False)
    print(f"TCN {days} days: selected {epochs} epochs using earlier training dates; refitting on all 303 dates.")
    del early_data, stop_data
    fitting_data = tcn_sequences(fitting_hourly, prices, days, target_prices=prices)
    preprocessing = fit_tcn_preprocessing(fitting_data)
    model, target_scale, _, _ = train_tcn(tcn_tensors(fitting_data, preprocessing), fitting_data["target"], epochs=epochs)
    del fitting_data
    bundle = {"model": model, "history_days": days, "preprocessing": preprocessing, "target_scale": target_scale, "epochs": epochs}
    query_data = tcn_sequences(validation_hourly, prices, days)
    name = f"tcn_smooth_history_{days}d"
    print(f"{name}: selected {epochs} epochs on earlier training dates (internal MAE {stopping_mae:.6f}); refitted on all 303 training dates.")
    result = evaluate_experiment(name, query_data["rows"], validation_hourly, validation_truth,
                                 tcn_deviations(bundle, query_data), reference_type="smooth")
    result["summary"].update(model="tcn", feature_set=f"history_{days}d",
                             feature_count=len(TCN_HISTORY_CHANNELS) + len(TCN_DAY_CHANNELS))
    tcn_training_summary[-1]["validation_mae_eur_mwh"] = result["scores"]["mae_eur_mwh"]
    pd.DataFrame(tcn_training_summary).to_csv(output_dir / "tcn_training_summary.csv", index=False)
    tcn_results[name], tcn_models[name] = result, bundle
    experiments[name], models[name] = result, bundle
    save_tcn(bundle, output_dir / f"{name}.pt")
    del query_data

display(pd.DataFrame([result["summary"] for result in tcn_results.values()])[["feature_set", "mae_eur_mwh"]].round(6))
best_tcn = min(tcn_results.values(), key=lambda result: result["summary"]["mae_eur_mwh"])
print(f"Best continuous-history TCN: {best_tcn['summary']['experiment']} | validation MAE: {best_tcn['summary']['mae_eur_mwh']:.6f} EUR/MWh")
display(best_tcn["scores"]["by_zone"].round(6))
worst_tcn_zone = best_tcn["scores"]["by_zone"].set_index("market_id")["mae_eur_mwh"].idxmax()
worst_tcn_hourly = zone_hourly_mae(best_tcn["predictions"], validation_truth, worst_tcn_zone)
print(f"Worst zone for best TCN: {worst_tcn_zone} — validation MAE by hour (hour 1 = 00:00–01:00)")
display(worst_tcn_hourly.round(6))


TCN 14 days: selected 18 epochs using earlier training dates; refitting on all 303 dates.


tcn_smooth_history_14d: selected 18 epochs on earlier training dates (internal MAE 3.981308); refitted on all 303 training dates.


tcn_smooth_history_14d: MAE = 5.447029 EUR/MWh | constraints: PASS | max hourly error = 2.51e-11


TCN 28 days: selected 31 epochs using earlier training dates; refitting on all 303 dates.


tcn_smooth_history_28d: selected 31 epochs on earlier training dates (internal MAE 3.973588); refitted on all 303 training dates.


...

,feature_set,mae_eur_mwh
0,history_14d,5.447029
1,history_28d,5.204911


Best continuous-history TCN: tcn_smooth_history_28d | validation MAE: 5.204911 EUR/MWh


,market_id,mae_eur_mwh
0,ZONE_0,4.119214
1,ZONE_1,6.449818
2,ZONE_2,5.572692
3,ZONE_3,4.677919


Worst zone for best TCN: ZONE_1 — validation MAE by hour (hour 1 = 00:00–01:00)


,hour_index,mae_eur_mwh,validation_days
0,1,5.317693,60
1,2,3.613943,60
2,3,2.244546,60
3,4,2.025715,60
4,5,2.622304,60
5,6,4.052420,60
6,7,4.855476,60
7,8,6.298922,60
8,9,7.831745,60
9,10,8.225111,60


### 4.9 Hourly coefficient LightGBM

Predict three coefficients per hour, then reconstruct four quarter-hour corrections using an orthonormal zero-sum basis:

| Component | Normalized pattern across quarters 1–4 |
| --- | --- |
| Slope | `[-3, -1, 1, 3] / sqrt(20)` |
| Curvature | `[1, -1, -1, 1] / 2` |
| Zigzag | `[-1, 3, -3, 1] / sqrt(20)` |

These patterns span every possible zero-sum four-quarter shape. Targets are `(observed prices − baseline) @ basis.T`; three independent LightGBM regressors predict their coefficients. Final prices are `baseline + correction_weight × (predicted coefficients @ basis)`. Compare weights **1.0 and 0.8**, with the same fitted models and the same final 60 validation dates.

Compare two references: our existing cumulative natural cubic spline (`smooth`), and **PCHIP on hourly price centers** (`pchip`). For PCHIP, integrate the price interpolant over each quarter to obtain an interval average, use constant prices outside the first/last hourly center, then shift each hour's four values to its supplied hourly mean. This recentering may introduce hour-boundary jumps. The precise PCHIP construction is explicit here; it is not an exact replication of an unspecified implementation. [SciPy PCHIP documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.PchipInterpolator.html).

Both use the same hourly predictor groups defined in Section 2, including baseline-shape and historical residual coefficients. Histories use calendar-day lags and strictly earlier dates; previous validation dates may contribute observed history. Model weights remain fixed. Missing history stays `NaN` with availability counts. Fixed settings: 800 trees, 31 leaves, learning rate 0.03, minimum 60 observations per leaf, L2 regularization 5, seed 42. Coefficient L1 loss is a training surrogate; comparison uses the **official reconstructed-price MAE**, not coefficient error. [LightGBM documentation](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.LGBMRegressor.html).

Checkpoints and their feature/basis metadata are saved locally. `load_coefficient_model(metadata_path)` and `predict_coefficient_model(bundle, test_hourly, history_15min, correction_weight=0.8)` provide inference without retraining; the returned DataFrame has the required submission columns.


In [6]:
from scipy.interpolate import PchipInterpolator
from lightgbm import LGBMRegressor, Booster

COEFFICIENT_NAMES = ("slope", "curvature", "zigzag")
COEFFICIENT_BASIS = np.array([
    np.array([-3., -1., 1., 3.]) / np.sqrt(20),
    np.array([1., -1., -1., 1.]) / 2,
    np.array([-1., 3., -3., 1.]) / np.sqrt(20),
])
COEFFICIENT_REFERENCE = "reference_coefficient_price_eur_mwh"
COEFFICIENT_TARGETS = [f"target_{name}_coefficient_eur_mwh" for name in COEFFICIENT_NAMES]
COEFFICIENT_PARAMETERS = dict(
    objective="regression_l1", n_estimators=800, num_leaves=31,
    learning_rate=0.03, min_child_samples=60, reg_lambda=5,
    random_state=42, n_jobs=4, verbosity=-1, deterministic=True, force_col_wise=True,
)


def coefficient_reference(hourly_inputs, method):
    """Return a complete, hourly-mean-consistent reference in quarter-key order."""
    reference = smooth_reference(hourly_inputs).rename(
        columns={"reference_smooth_price_eur_mwh": COEFFICIENT_REFERENCE})
    if method == "smooth":
        return reference
    if method != "pchip":
        raise ValueError("Reference must be smooth or pchip.")
    hourly = hourly_inputs.copy()
    hourly["delivery_date"] = pd.to_datetime(hourly.delivery_date)
    curve = hourly.pivot(index=DAY_KEYS, columns="hour_index", values="hourly_price_eur_mwh")
    values = curve.reindex(columns=range(1, 25)).sort_index().to_numpy()
    centers = np.arange(24) + 0.5
    bounds = np.arange(97) / 4
    integral = PchipInterpolator(centers, values, axis=1, extrapolate=False).antiderivative()
    # Integrate constant extensions for [0, .5] and [23.5, 24].
    cumulative = integral(np.clip(bounds, 0.5, 23.5))
    cumulative += values[:, :1] * np.minimum(bounds - 0.5, 0)[None, :]
    cumulative += values[:, -1:] * np.maximum(bounds - 23.5, 0)[None, :]
    quarters = (np.diff(cumulative, axis=1) * 4).reshape(-1, 24, 4)
    quarters += values[:, :, None] - quarters.mean(axis=2, keepdims=True)
    reference[COEFFICIENT_REFERENCE] = quarters.reshape(-1)
    return reference


def residual_coefficients(quarter_prices, method):
    """Project complete observed-hour residuals; never used as current-day inputs."""
    hourly = hourly_from_quarters(quarter_prices)
    reference = coefficient_reference(hourly, method)
    actual = quarter_prices[QUARTER_KEYS + ["price_eur_mwh"]].copy()
    actual["delivery_date"] = pd.to_datetime(actual.delivery_date)
    joined = reference.merge(actual, on=QUARTER_KEYS, how="outer", validate="one_to_one", indicator=True)
    if not joined._merge.eq("both").all():
        raise ValueError("Coefficient labels require complete matching days/hours/quarters.")
    joined = joined.sort_values(QUARTER_KEYS)
    residuals = (joined.price_eur_mwh - joined[COEFFICIENT_REFERENCE]).to_numpy().reshape(-1, 4)
    np.testing.assert_allclose(residuals.sum(axis=1), 0, atol=1e-8, rtol=0)
    result = joined[HOUR_KEYS].drop_duplicates().reset_index(drop=True)
    result[COEFFICIENT_TARGETS] = residuals @ COEFFICIENT_BASIS.T
    return result


def build_coefficient_features(hourly_inputs, history_15min, method):
    """One hourly row; matching-reference histories are strictly earlier per query."""
    # Reuse existing known-input construction, without quarter-history predictors.
    base = build_features(hourly_inputs, history_15min.iloc[:0])
    base = base[base.quarter_index.eq(1)].reset_index(drop=True)
    columns = HOUR_KEYS + ["position_zone", "position_hour"] + FEATURE_GROUPS["calendar"]
    columns += FEATURE_GROUPS["local"] + FEATURE_GROUPS["crosszone"] + FEATURE_GROUPS["regime"]
    columns += [f"daily_{name}_price_eur_mwh" for name in ("mean", "range", "std")]
    columns += ["temporal_hourly_change_from_previous_eur_mwh", "temporal_hourly_change_to_next_eur_mwh"]
    features = base[columns].copy()
    features["position_zone"] = pd.Categorical(features.market_id, categories=EXPECTED_ZONES)
    features["temporal_hourly_curvature_eur_mwh"] = (
        features.temporal_hourly_change_to_next_eur_mwh - features.temporal_hourly_change_from_previous_eur_mwh)
    for zone in EXPECTED_ZONES:
        column = f"crosszone_{zone}_hourly_price_eur_mwh"
        previous = features.groupby(DAY_KEYS)[column].shift(1).fillna(features[column])
        features[f"crosszone_{zone}_change_from_previous_eur_mwh"] = features[column] - previous
    reference = coefficient_reference(hourly_inputs, method)
    reference_values = reference[COEFFICIENT_REFERENCE].to_numpy().reshape(-1, 4)
    shape = (reference_values - reference_values.mean(axis=1, keepdims=True)) @ COEFFICIENT_BASIS.T
    for i, name in enumerate(COEFFICIENT_NAMES):
        features[f"reference_{name}_coefficient_eur_mwh"] = shape[:, i]

    history = history_15min.copy()
    history["delivery_date"] = pd.to_datetime(history.delivery_date)
    history = history[history.delivery_date.lt(features.delivery_date.max())]
    if history.empty:
        history = features[HOUR_KEYS].iloc[:0].copy()
        history[COEFFICIENT_TARGETS] = np.empty((0, 3))
    else:
        history = residual_coefficients(history, method)
    shape_keys = ["market_id", "hour_index"]
    for days in (1, 7):
        lagged = history.copy()
        lagged["delivery_date"] += pd.Timedelta(days=days)
        lagged = lagged.rename(columns={target: f"history_coefficient_{name}_lag_{days}d_eur_mwh"
                                       for name, target in zip(COEFFICIENT_NAMES, COEFFICIENT_TARGETS)})
        features = features.merge(lagged, on=HOUR_KEYS, how="left", validate="one_to_one")
    timeline = pd.concat([history[HOUR_KEYS], features[HOUR_KEYS]], ignore_index=True).drop_duplicates()
    timeline = timeline.merge(history, on=HOUR_KEYS, how="left", validate="one_to_one")
    timeline = timeline.sort_values(shape_keys + ["delivery_date"]).set_index("delivery_date")
    for days in (7, 28):
        window = timeline.groupby(shape_keys)[COEFFICIENT_TARGETS].rolling(f"{days}D", closed="left", min_periods=1)
        means = window.mean().reset_index().rename(columns={
            target: f"history_coefficient_{name}_mean_{days}d_eur_mwh"
            for name, target in zip(COEFFICIENT_NAMES, COEFFICIENT_TARGETS)})
        counts = window.count()[COEFFICIENT_TARGETS[0]].reset_index().rename(
            columns={COEFFICIENT_TARGETS[0]: f"history_coefficient_count_{days}d"})
        features = features.merge(means, on=HOUR_KEYS, how="left", validate="one_to_one")
        features = features.merge(counts, on=HOUR_KEYS, how="left", validate="one_to_one")
        column = f"history_coefficient_count_{days}d"
        features[column] = features[column].fillna(0).astype(int)
    return features.sort_values(HOUR_KEYS).reset_index(drop=True), reference


def coefficient_deviations(bundle, hourly_features, correction_weight):
    coefficients = np.column_stack([model.predict(hourly_features[bundle["columns"]])
                                    for model in bundle["regressors"]])
    return (correction_weight * coefficients @ np.asarray(bundle["basis"])).reshape(-1)


def save_coefficient_model(bundle, metadata_path):
    metadata_path = Path(metadata_path)
    metadata = {key: bundle[key] for key in ("method", "columns", "groups", "basis", "parameters")}
    metadata["zone_categories"] = list(EXPECTED_ZONES)
    metadata["training_last_date"] = str(fit_rows.delivery_date.max().date())
    metadata["model_files"] = []
    for name, regressor in zip(COEFFICIENT_NAMES, bundle["regressors"]):
        filename = f"{metadata_path.stem}_{name}.txt"
        regressor.booster_.save_model(str(metadata_path.parent / filename))
        metadata["model_files"].append(filename)
    metadata_path.write_text(json.dumps(metadata, indent=2) + "\n")


def load_coefficient_model(metadata_path):
    metadata_path = Path(metadata_path)
    bundle = json.loads(metadata_path.read_text())
    if bundle["zone_categories"] != list(EXPECTED_ZONES):
        raise ValueError("Checkpoint zone categories do not match this notebook.")
    bundle["regressors"] = [Booster(model_file=str(metadata_path.parent / file)) for file in bundle["model_files"]]
    return bundle


def predict_coefficient_model(bundle, test_hourly, history_15min, correction_weight=0.8):
    features, reference = build_coefficient_features(test_hourly, history_15min, bundle["method"])
    deviations = coefficient_deviations(bundle, features, correction_weight)
    return make_predictions(reference, test_hourly, deviations,
                            reference_type=bundle["method"], reference_column=COEFFICIENT_REFERENCE)


In [8]:
coefficient_results, coefficient_models = {}, {}
coefficient_training_hourly = train_hourly[train_hourly.delivery_date.lt(validation_start)]
for method in ("smooth", "pchip"):
    training, _ = build_coefficient_features(coefficient_training_hourly, prices, method)
    targets = residual_coefficients(prices[prices.delivery_date.lt(validation_start)], method)
    training = training.merge(targets, on=HOUR_KEYS, validate="one_to_one")
    validation, reference = build_coefficient_features(validation_hourly, prices, method)
    COEFFICIENT_FEATURE_GROUPS = {
        group: [column for column in validation if column.startswith(group + "_")]
        for group in ("position", "calendar", "local", "daily", "temporal", "crosszone", "regime", "reference", "history_coefficient")
    }
    columns = [column for group in COEFFICIENT_FEATURE_GROUPS.values() for column in group]
    assert len(columns) == len(set(columns)) and not set(columns).intersection(COEFFICIENT_TARGETS)
    regressors = []
    for target in COEFFICIENT_TARGETS:
        regressor = LGBMRegressor(**COEFFICIENT_PARAMETERS)
        regressor.fit(training[columns], training[target], categorical_feature=["position_zone"])
        regressors.append(regressor)
    bundle = dict(method=method, columns=columns, groups=COEFFICIENT_FEATURE_GROUPS,
                  basis=COEFFICIENT_BASIS.tolist(), parameters=COEFFICIENT_PARAMETERS, regressors=regressors)
    coefficient_models[method] = bundle
    save_coefficient_model(bundle, output_dir / f"lightgbm_coefficients_{method}.json")
    if method == "pchip":
        name = "baseline_pchip_reference"
        result = evaluate_experiment(name, reference, validation_hourly, validation_truth,
                                     reference_type=method, reference_column=COEFFICIENT_REFERENCE)
        result["summary"].update(model="baseline", feature_set="reference", feature_count=0)
        experiments[name] = result
    for weight in (1.0, 0.8):
        feature_set = f"coefficients_weight_{weight:.1f}"
        name = f"lightgbm_{method}_{feature_set}"
        result = evaluate_experiment(
            name, reference, validation_hourly, validation_truth,
            predicted_deviations=coefficient_deviations(bundle, validation, weight),
            reference_type=method, reference_column=COEFFICIENT_REFERENCE,
        )
        result["summary"].update(model="lightgbm", feature_set=feature_set, feature_count=len(columns))
        coefficient_results[name] = experiments[name] = result
        models[name] = bundle

feature_manifest["lightgbm_coefficients"] = {method: bundle["groups"] for method, bundle in coefficient_models.items()}
(output_dir / "feature_sets.json").write_text(json.dumps(feature_manifest, indent=2) + "\n")
display(pd.DataFrame([result["summary"] for result in coefficient_results.values()])[
    ["experiment", "feature_count", "mae_eur_mwh", "constraints_passed"]].round(6))
best_coefficient = min(coefficient_results.values(), key=lambda result: result["summary"]["mae_eur_mwh"])
print("Best hourly coefficient experiment:", best_coefficient["summary"]["experiment"])
display(best_coefficient["scores"]["by_zone"].round(6))
worst_coefficient_zone = best_coefficient["scores"]["by_zone"].sort_values("mae_eur_mwh", ascending=False).iloc[0].market_id
print("MAE by hour for worst zone:", worst_coefficient_zone)
worst_coefficient_hourly = zone_hourly_mae(best_coefficient["predictions"], validation_truth, worst_coefficient_zone)
display(worst_coefficient_hourly.round(6))


lightgbm_smooth_coefficients_weight_1.0: MAE = 5.029807 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


lightgbm_smooth_coefficients_weight_0.8: MAE = 4.818016 EUR/MWh | constraints: PASS | max hourly error = 2.51e-11


baseline_pchip_reference: MAE = 6.655777 EUR/MWh | constraints: PASS | max hourly error = 2.51e-11


lightgbm_pchip_coefficients_weight_1.0: MAE = 5.008962 EUR/MWh | constraints: PASS | max hourly error = 2.51e-11


lightgbm_pchip_coefficients_weight_0.8: MAE = 4.797342 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


,experiment,feature_count,mae_eur_mwh,constraints_passed
0,lightgbm_smooth_coefficients_weight_1.0,50,5.029807,True
1,lightgbm_smooth_coefficients_weight_0.8,50,4.818016,True
2,lightgbm_pchip_coefficients_weight_1.0,50,5.008962,True
3,lightgbm_pchip_coefficients_weight_0.8,50,4.797342,True


Best hourly coefficient experiment: lightgbm_pchip_coefficients_weight_0.8


,market_id,mae_eur_mwh
0,ZONE_0,3.478044
1,ZONE_1,6.032080
2,ZONE_2,5.341404
3,ZONE_3,4.337842


MAE by hour for worst zone: ZONE_1


,hour_index,mae_eur_mwh,validation_days
0,1,4.838810,60
1,2,3.513619,60
2,3,2.224424,60
3,4,2.031781,60
4,5,2.571669,60
5,6,3.599051,60
6,7,4.545374,60
7,8,6.402519,60
8,9,8.513131,60
9,10,8.424378,60


### 4.10 Coefficient-model experiments

Keep the same 50 hourly predictors and test **12 configurations**. Changing only the target basis isolates its effect: baseline-shape and history predictors remain in the original, interpretable polynomial basis.

| Change | Variations |
| --- | --- |
| Reference | Smooth versus PCHIP with the original LightGBM/polynomial control; remaining variants use PCHIP. |
| Basis | Original polynomial; Haar (`[1,1,-1,-1]/2`, `[1,-1,0,0]/sqrt(2)`, `[0,0,1,-1]/sqrt(2)`); Helmert contrasts; PCA axes learned from training residuals only. All retain three orthonormal, zero-sum vectors. |
| Learner | LightGBM MAE, LightGBM MSE, a smaller-leaf LightGBM MAE model, three CatBoost MAE regressors, joint CatBoost MultiRMSE, multi-output Ridge, and multi-output Extra Trees. |
| Correction weights | Fixed 1.0 / 0.8; shared weight from `0.0, 0.1, …, 1.2`; three separate component weights bounded to `[0, 1.2]`, fitted against reconstructed-price MAE. |

**Chronological selection.** Fit on the first 243 dates and predict the next 30; fit on the first 273 dates and predict the following 30. All these dates precede August 2. Learn correction weights on the first block and compare their performance on the second block. Select a configuration/weight policy per model family and a weight policy for each alternative LightGBM-MAE basis using that second block, then refit the selected model on all 303 training dates and recalibrate its weights on both earlier prediction blocks. The final 60 dates are used only for the final comparison. Histories always exclude each query date; PCA and preprocessing use the corresponding fit fold only. No final-holdout label selects a setting.

This is a controlled search, not a full Cartesian grid. Joint MSE models use the polynomial basis because an orthogonal target rotation preserves their squared-error geometry. Merely rescaling vectors would also conflate basis direction and regularization, so all bases are normalized. PCA changes coordinates without dropping any residual dimension. Component weights can change the relative slope/curvature/detail amplitudes while preserving hourly means.

The notebook shows compact selection and weight-sensitivity tables, final model-family and basis comparisons (duplicate configurations are omitted), zone MAEs and the worst zone by hour. Full trial records are saved in `outputs/experiments/coefficient_search/`. Selected bundles include models, preprocessing, basis and weights; use `load_coefficient_search(path)` and `predict_coefficient_search(bundle, test_hourly, history_15min)` for inference. [CatBoost MultiRMSE](https://catboost.ai/docs/en/concepts/loss-functions-multiregression), [Extra Trees](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.ExtraTreesRegressor.html).


In [7]:
from scipy.linalg import helmert
from scipy.optimize import linprog
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.preprocessing import OneHotEncoder
import joblib

COEFFICIENT_WEIGHT_GRID = np.round(np.arange(0, 1.21, 0.1), 1)
COEFFICIENT_TRIALS = [
    dict(id="smooth_polynomial_lgbm_mae", method="smooth", basis="polynomial", learner="lgbm_mae", family="lightgbm"),
    dict(id="pchip_polynomial_lgbm_mae", method="pchip", basis="polynomial", learner="lgbm_mae", family="lightgbm"),
    *[dict(id=f"pchip_{basis}_lgbm_mae", method="pchip", basis=basis, learner="lgbm_mae", family="lightgbm")
      for basis in ("haar", "helmert", "pca")],
    *[dict(id=f"pchip_polynomial_{learner}", method="pchip", basis="polynomial", learner=learner, family=family)
      for learner, family in (("lgbm_mse", "lightgbm"), ("lgbm_small", "lightgbm"),
                              ("catboost_mae", "catboost"), ("catboost_joint", "catboost"),
                              ("ridge", "ridge"), ("extra_trees", "extra_trees"))],
    dict(id="pchip_pca_catboost_mae", method="pchip", basis="pca", learner="catboost_mae", family="catboost"),
]


def coefficient_basis(kind, training_coefficients):
    if kind == "polynomial":
        basis = COEFFICIENT_BASIS.copy()
    elif kind == "haar":
        basis = np.array([[.5, .5, -.5, -.5], [1/np.sqrt(2), -1/np.sqrt(2), 0, 0],
                          [0, 0, 1/np.sqrt(2), -1/np.sqrt(2)]])
    elif kind == "helmert":
        basis = helmert(4)
    elif kind == "pca":
        _, axes = np.linalg.eigh(np.cov(training_coefficients, rowvar=False))
        basis = axes[:, ::-1].T @ COEFFICIENT_BASIS
        # Resolve arbitrary eigenvector signs for reproducible metadata.
        basis *= np.where(basis[np.arange(3), np.abs(basis).argmax(axis=1)] < 0, -1, 1)[:, None]
    else:
        raise ValueError("Unknown coefficient basis.")
    np.testing.assert_allclose(basis @ basis.T, np.eye(3), atol=1e-12, rtol=0)
    np.testing.assert_allclose(basis.sum(axis=1), 0, atol=1e-12, rtol=0)
    return basis


def fit_coefficient_trial(config, features, original_targets, columns):
    basis = coefficient_basis(config["basis"], original_targets)
    targets = original_targets.copy() if config["basis"] == "polynomial" else original_targets @ COEFFICIENT_BASIS @ basis.T
    learner = config["learner"]
    if learner.startswith("lgbm"):
        parameters = COEFFICIENT_PARAMETERS.copy()
        if learner == "lgbm_mse":
            parameters["objective"] = "regression"
        if learner == "lgbm_small":
            parameters.update(num_leaves=15, min_child_samples=100, n_estimators=1200)
        estimators = [LGBMRegressor(**parameters).fit(features[columns], targets[:, i],
                       categorical_feature=["position_zone"]) for i in range(3)]
    elif learner.startswith("catboost"):
        parameters = dict(iterations=800, depth=6, learning_rate=.03, l2_leaf_reg=5,
                          random_seed=42, thread_count=4, verbose=False, allow_writing_files=False,
                          cat_features=["position_zone"])
        # CatBoost does not accept numeric NaN categories; only zone is categorical here.
        if learner == "catboost_joint":
            estimators = [CatBoostRegressor(**parameters, loss_function="MultiRMSE").fit(features[columns], targets)]
        else:
            estimators = [CatBoostRegressor(**parameters, loss_function="MAE").fit(features[columns], targets[:, i]) for i in range(3)]
    else:
        categorical = ["position_zone", "position_hour", "calendar_day_of_week"]
        numeric = [c for c in columns if c not in categorical]
        preprocessing = ColumnTransformer([
            ("numeric", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True), numeric),
            ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
        ])
        estimator = (Ridge(alpha=100) if learner == "ridge" else ExtraTreesRegressor(
            n_estimators=200, min_samples_leaf=12, max_features=1.0, n_jobs=4, random_state=42))
        steps = [("preprocessing", preprocessing)]
        if learner == "ridge":
            steps.append(("scale", StandardScaler()))
        steps.append(("model", estimator))
        estimators = [Pipeline(steps).fit(features[columns], targets)]
    return dict(config=config, basis=basis, columns=columns, estimators=estimators,
                zone_categories=list(EXPECTED_ZONES), training_last_date=str(features.delivery_date.max().date()))


def predict_trial_coefficients(bundle, features):
    output = [model.predict(features[bundle["columns"]]) for model in bundle["estimators"]]
    return np.column_stack(output) if len(output) == 3 else np.asarray(output[0])


def coefficient_component_design(coefficients, basis):
    # One column per component, one row per predicted quarter.
    return (coefficients[:, :, None] * np.asarray(basis)[None, :, :]).transpose(0, 2, 1).reshape(-1, 3)


def reconstructed_coefficient_mae(rows, residuals, design, weights):
    errors = np.abs(np.asarray(residuals).reshape(-1) - design @ np.asarray(weights)).reshape(-1, 4).mean(axis=1)
    daily = rows[HOUR_KEYS].copy()
    daily["error"] = errors
    return float(daily.groupby(DAY_KEYS).error.mean().groupby("market_id").mean().reindex(EXPECTED_ZONES).mean())


def calibrate_coefficient_weights(policy, rows, residuals, design):
    if policy.startswith("fixed_"):
        return np.repeat(float(policy.removeprefix("fixed_")), 3)
    if policy == "global":
        scores = [reconstructed_coefficient_mae(rows, residuals, design, np.repeat(w, 3)) for w in COEFFICIENT_WEIGHT_GRID]
        return np.repeat(COEFFICIENT_WEIGHT_GRID[np.argmin(scores)], 3)
    if policy != "component":
        raise ValueError("Unknown correction-weight policy.")
    # Exact price-MAE calibration by linear programming, with official day/zone weights.
    day_hour_counts = rows.groupby(DAY_KEYS).hour_index.transform("size").to_numpy()
    zone_days = rows[DAY_KEYS].drop_duplicates().groupby("market_id").size()
    day_weights = 1 / (4 * rows.market_id.map(zone_days).to_numpy() * day_hour_counts)
    objective_weights = np.repeat(day_weights / 4, 4)
    target = np.asarray(residuals).reshape(-1)
    identity = sparse.eye(len(target), format="csr")
    matrix = sparse.csr_matrix(design)
    inequalities = sparse.vstack([sparse.hstack([matrix, -identity]), sparse.hstack([-matrix, -identity])], format="csr")
    solution = linprog(np.r_[np.zeros(3), objective_weights], A_ub=inequalities,
                       b_ub=np.r_[target, -target], bounds=[(0, 1.2)]*3 + [(0, None)]*len(target),
                       method="highs")
    if not solution.success:
        raise RuntimeError(solution.message)
    return solution.x[:3]


def save_coefficient_search(bundle, path):
    joblib.dump(bundle, path)


def load_coefficient_search(path):
    bundle = joblib.load(path)
    if bundle["zone_categories"] != list(EXPECTED_ZONES):
        raise ValueError("Checkpoint zone categories do not match this notebook.")
    return bundle


def predict_coefficient_search(bundle, test_hourly, history_15min):
    features, reference = build_coefficient_features(test_hourly, history_15min, bundle["config"]["method"])
    coefficients = predict_trial_coefficients(bundle, features)
    deviations = coefficient_component_design(coefficients, bundle["basis"]) @ bundle["weights"]
    return make_predictions(reference, test_hourly, deviations, reference_type=bundle["config"]["method"],
                            reference_column=COEFFICIENT_REFERENCE)


In [9]:
coefficient_search_dir = output_dir / "coefficient_search"
coefficient_search_dir.mkdir(parents=True, exist_ok=True)
training_dates = np.sort(fit_rows.delivery_date.unique())
coefficient_folds = [(pd.Timestamp(training_dates[-60]), pd.Timestamp(training_dates[-30])),
                     (pd.Timestamp(training_dates[-30]), validation_start)]
coefficient_search_data = {}
coefficient_columns = json.loads((output_dir / "lightgbm_coefficients_pchip.json").read_text())["columns"]
for method in ("smooth", "pchip"):
    features, _ = build_coefficient_features(coefficient_training_hourly, prices, method)
    targets = residual_coefficients(prices[prices.delivery_date.lt(validation_start)], method)
    features = features.merge(targets, on=HOUR_KEYS, validate="one_to_one")
    coefficient_search_data[method] = features

coefficient_fold_predictions, coefficient_selection_records, coefficient_weight_records = {}, [], []
for config in COEFFICIENT_TRIALS:
    data = coefficient_search_data[config["method"]]
    folds = []
    for fold, (start, end) in enumerate(coefficient_folds, 1):
        earlier = data[data.delivery_date.lt(start)].reset_index(drop=True)
        query = data[data.delivery_date.ge(start) & data.delivery_date.lt(end)].reset_index(drop=True)
        bundle = fit_coefficient_trial(config, earlier, earlier[COEFFICIENT_TARGETS].to_numpy(), coefficient_columns)
        coefficients = predict_trial_coefficients(bundle, query)
        residuals = query[COEFFICIENT_TARGETS].to_numpy() @ COEFFICIENT_BASIS
        design = coefficient_component_design(coefficients, bundle["basis"])
        folds.append(dict(rows=query[HOUR_KEYS], residuals=residuals, design=design, basis=bundle["basis"]))
    coefficient_fold_predictions[config["id"]] = folds
    first, second = folds
    for weight in COEFFICIENT_WEIGHT_GRID:
        coefficient_weight_records.append(dict(config=config["id"], weight=weight,
            calibration_mae=reconstructed_coefficient_mae(first["rows"], first["residuals"], first["design"], np.repeat(weight, 3)),
            selection_mae=reconstructed_coefficient_mae(second["rows"], second["residuals"], second["design"], np.repeat(weight, 3))))
    for policy in ("fixed_1.0", "fixed_0.8", "global", "component"):
        weights = calibrate_coefficient_weights(policy, first["rows"], first["residuals"], first["design"])
        coefficient_selection_records.append(dict(config=config["id"], family=config["family"], method=config["method"],
            basis=config["basis"], learner=config["learner"], policy=policy, weights=json.dumps(weights.tolist()),
            calibration_mae=reconstructed_coefficient_mae(first["rows"], first["residuals"], first["design"], weights),
            selection_mae=reconstructed_coefficient_mae(second["rows"], second["residuals"], second["design"], weights)))
    print("Completed chronological comparisons:", config["id"], flush=True)

coefficient_selection = pd.DataFrame(coefficient_selection_records)
coefficient_selection.to_csv(coefficient_search_dir / "selection.csv", index=False)
coefficient_weight_sensitivity = pd.DataFrame(coefficient_weight_records)
coefficient_weight_sensitivity.to_csv(coefficient_search_dir / "weight_sensitivity.csv", index=False)
joblib.dump(coefficient_fold_predictions, coefficient_search_dir / "fold_predictions.joblib")
selected_families = coefficient_selection.sort_values("selection_mae").groupby("family", sort=False).head(1)
control = coefficient_selection[coefficient_selection.config.eq("pchip_polynomial_lgbm_mae")].sort_values("selection_mae").iloc[0]
print("Selected using the second earlier block; weights were learned on the first block:")
display(selected_families[["family", "method", "basis", "learner", "policy", "weights", "selection_mae"]].reset_index(drop=True).round(6))
print("PCHIP/polynomial/LightGBM scalar-weight sensitivity (the selection column is diagnostic):")
display(coefficient_weight_sensitivity[coefficient_weight_sensitivity.config.eq(control.config)].drop(columns="config").reset_index(drop=True).round(6))
print("Original PCHIP/LightGBM weight policies, fitted on the first block and scored on the second:")
display(coefficient_selection[coefficient_selection.config.eq(control.config)][["policy", "weights", "calibration_mae", "selection_mae"]].reset_index(drop=True).round(6))
print("Best weight policy for each PCHIP LightGBM-MAE basis:")
basis_comparison = coefficient_selection[coefficient_selection.learner.eq("lgbm_mae") & coefficient_selection.method.eq("pchip")]
display(basis_comparison.sort_values("selection_mae").groupby("basis",sort=False).head(1)[["basis","policy","selection_mae"]].reset_index(drop=True).round(6))


Completed chronological comparisons: smooth_polynomial_lgbm_mae


Completed chronological comparisons: pchip_polynomial_lgbm_mae


Completed chronological comparisons: pchip_haar_lgbm_mae


Completed chronological comparisons: pchip_helmert_lgbm_mae


Completed chronological comparisons: pchip_pca_lgbm_mae


...

,weight,calibration_mae,selection_mae
0,0.0,5.878586,5.232755
1,0.1,5.536325,4.902378
2,0.2,5.210478,4.592996
3,0.3,4.902662,4.312390
4,0.4,4.620732,4.070457
5,0.5,4.368661,3.879766
6,0.6,4.152870,3.743970
7,0.7,3.988381,3.671210
8,0.8,3.876819,3.657886
9,0.9,3.818024,3.702888


Best weight policy for each PCHIP LightGBM-MAE basis:


,basis,policy,selection_mae
0,polynomial,fixed_0.8,3.657886
1,pca,fixed_0.8,3.665092
2,helmert,fixed_0.8,3.673029
3,haar,fixed_0.8,3.688854


Original PCHIP/LightGBM weight policies (first-block fitting, second-block scoring):


policy,weights,calibration_mae,selection_mae
fixed_1.0,"[1.0, 1.0, 1.0]",3.814620,3.798450
fixed_0.8,"[0.8, 0.8, 0.8]",3.876819,3.657886
global,"[1.0, 1.0, 1.0]",3.814620,3.798450
component,"[0.9550455342182692, 0.9054402201656024, 1.0554515858019848]",3.807492,3.750985


In [9]:
coefficient_search_results, coefficient_search_models = {}, {}
selected_configurations = [("control", control)] + [(row.family, row) for _, row in selected_families.iterrows()]
basis_selections = basis_comparison.sort_values("selection_mae").groupby("basis", sort=False).head(1)
selected_configurations += [(f"basis_{row.basis}", row) for _,row in basis_selections.iterrows() if row.basis != "polynomial"]
coefficient_refit_cache, seen_configurations = {}, set()
for label, selected in selected_configurations:
    if (selected.config, selected.policy) in seen_configurations:
        continue
    seen_configurations.add((selected.config, selected.policy))
    config = next(c for c in COEFFICIENT_TRIALS if c["id"] == selected.config)
    folds = coefficient_fold_predictions[config["id"]]
    calibration_rows = pd.concat([fold["rows"] for fold in folds], ignore_index=True)
    calibration_residuals = np.concatenate([fold["residuals"] for fold in folds])
    calibration_design = np.concatenate([fold["design"] for fold in folds])
    weights = calibrate_coefficient_weights(selected.policy, calibration_rows, calibration_residuals, calibration_design)
    # PCA axes differ by fold. Its component weights are intentionally kept in the
    # fold-specific, variance-ordered coordinates during calibration, then applied
    # to the refit's variance-ordered axes. Signs cancel in coefficient × basis.
    data = coefficient_search_data[config["method"]]
    if selected.config not in coefficient_refit_cache:
        coefficient_refit_cache[selected.config] = fit_coefficient_trial(
            config, data, data[COEFFICIENT_TARGETS].to_numpy(), coefficient_columns)
    bundle = dict(coefficient_refit_cache[selected.config], weights=weights,
                  weight_policy=selected.policy, selection_mae=float(selected.selection_mae))
    name = f"coefficient_search_{label}"
    save_coefficient_search(bundle, coefficient_search_dir / f"{name}.joblib")
    validation, reference = build_coefficient_features(validation_hourly, prices, config["method"])
    design = coefficient_component_design(predict_trial_coefficients(bundle, validation), bundle["basis"])
    result = evaluate_experiment(name, reference, validation_hourly, validation_truth,
        predicted_deviations=design @ weights, reference_type=config["method"], reference_column=COEFFICIENT_REFERENCE)
    result["summary"].update(model=f"coefficient_{label}", feature_set="selected_coefficients", feature_count=len(coefficient_columns),
        selected_config=config["id"], weight_policy=selected.policy, correction_weights=json.dumps(weights.tolist()),
        selection_mae_eur_mwh=float(selected.selection_mae))
    coefficient_search_results[name] = experiments[name] = result
    coefficient_search_models[name] = models[name] = bundle

coefficient_final_table = pd.DataFrame([r["summary"] for r in coefficient_search_results.values()])
coefficient_final_table.to_csv(coefficient_search_dir / "final_comparison.csv", index=False)
display(coefficient_final_table[["experiment", "selected_config", "weight_policy", "correction_weights",
                                 "selection_mae_eur_mwh", "mae_eur_mwh", "constraints_passed"]].round(6))
best_coefficient_search = min(coefficient_search_results.values(), key=lambda r: r["summary"]["mae_eur_mwh"])
print("Best final-holdout coefficient experiment:", best_coefficient_search["summary"]["experiment"])
display(best_coefficient_search["scores"]["by_zone"].round(6))
worst_coefficient_search_zone = best_coefficient_search["scores"]["by_zone"].sort_values("mae_eur_mwh", ascending=False).iloc[0].market_id
print("MAE by hour for worst zone:", worst_coefficient_search_zone)
worst_coefficient_search_hourly = zone_hourly_mae(best_coefficient_search["predictions"], validation_truth, worst_coefficient_search_zone)
display(worst_coefficient_search_hourly.round(6))
feature_manifest["coefficient_search"] = {name: {"config": b["config"], "columns": b["columns"],
    "basis": b["basis"].tolist(), "weights": b["weights"].tolist()} for name,b in coefficient_search_models.items()}
(output_dir / "feature_sets.json").write_text(json.dumps(feature_manifest, indent=2) + "\n")


coefficient_search_control: MAE = 4.797342 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


coefficient_search_catboost: MAE = 4.823057 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


coefficient_search_extra_trees: MAE = 4.828894 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


coefficient_search_ridge: MAE = 4.995387 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


coefficient_search_basis_pca: MAE = 4.810557 EUR/MWh | constraints: PASS | max hourly error = 2.51e-11


...

Best final-holdout coefficient experiment: coefficient_search_control


,market_id,mae_eur_mwh
0,ZONE_0,3.478044
1,ZONE_1,6.032080
2,ZONE_2,5.341404
3,ZONE_3,4.337842


MAE by hour for worst zone: ZONE_1


,hour_index,mae_eur_mwh,validation_days
0,1,4.838810,60
1,2,3.513619,60
2,3,2.224424,60
3,4,2.031781,60
4,5,2.571669,60
5,6,3.599051,60
6,7,4.545374,60
7,8,6.402519,60
8,9,8.513131,60
9,10,8.424378,60


57863

### 4.11 Coefficient feature comparisons

Keep **PCHIP + polynomial basis + three LightGBM MAE regressors × 0.8** fixed, including the original tree settings and 50-feature control. Test the additions defined in Section 2 separately and in combinations:

| Experiment | Inputs beyond the original 50 |
| --- | --- |
| `curve` | Full daily curve, min/max and wider same-day hourly context. |
| `recent_history` | Residual lags 2/3/14/28 days; means/counts 3/14/56 days; median/std 7/28 days. |
| `weekday_profile` | Same-weekday residual means/counts over 28/56 days. |
| `neighbor_history` | Neighboring-hour residual coefficients from 1/7 days earlier. |
| `crosszone_history` | Cross-zone historical residual-coefficient differences. |
| `all_histories` | All four historical additions above. |
| `all` | Curve plus all historical additions. |

Compare the control and seven predefined feature selections on the same two earlier 30-date chronological blocks used in Section 4.10 (243/273 fit dates), with fixed correction weight 0.8. Rank by their equally weighted two-block official MAE, then refit each expanded selection on all 303 training dates and compare on the final 60 dates. The feature lists and model settings are fixed before evaluating that final period. Each model has frozen weights during evaluation; histories update only from strictly earlier actual observations. Both earlier blocks and the repeatedly inspected final period remain model-comparison evidence rather than an untouched test.

Detailed fold scores, feature manifests and model bundles are saved in `outputs/experiments/coefficient_features/`. `predict_coefficient_features(bundle, test_hourly, history_15min)` builds the matching extended inputs and returns the required prediction schema. All construction and inference code stays in this notebook.


In [8]:
def build_expanded_coefficient_features(hourly_inputs, history_15min):
    """Original coefficient predictors plus explicitly selected, causal extensions."""
    features, reference = build_coefficient_features(hourly_inputs, history_15min, "pchip")
    additions = {name: [] for name in ("curve", "recent_history", "weekday_profile", "neighbor_history", "crosszone_history")}
    hourly = hourly_inputs.copy()
    hourly["delivery_date"] = pd.to_datetime(hourly.delivery_date)
    curve = hourly.pivot(index=DAY_KEYS, columns="hour_index", values="hourly_price_eur_mwh").reindex(columns=range(1,25))
    curve.columns = [f"daily_hour_{hour:02d}_price_eur_mwh" for hour in curve.columns]
    curve["daily_min_price_eur_mwh"] = curve.min(axis=1)
    curve["daily_max_price_eur_mwh"] = curve.iloc[:,:24].max(axis=1)
    features = features.merge(curve.reset_index(), on=DAY_KEYS, validate="many_to_one")
    additions["curve"] += list(curve.columns)
    for offset in (-6,-3,-2,2,3,6):
        label = f"{'minus' if offset < 0 else 'plus'}_{abs(offset)}"
        price_column = f"local_hour_{label}_price_eur_mwh"
        change_column = f"temporal_hour_{label}_minus_current_eur_mwh"
        features[price_column] = features.groupby(DAY_KEYS).local_hourly_price_eur_mwh.shift(-offset)
        features[change_column] = features[price_column] - features.local_hourly_price_eur_mwh
        additions["curve"] += [price_column, change_column]

    history = history_15min.copy()
    history["delivery_date"] = pd.to_datetime(history.delivery_date)
    history = history[history.delivery_date.lt(features.delivery_date.max())]
    if history.empty:
        history = features[HOUR_KEYS].iloc[:0].copy()
        history[COEFFICIENT_TARGETS] = np.empty((0,3))
    else:
        history = residual_coefficients(history, "pchip")
    shape_keys = ["market_id", "hour_index"]
    for days in (2,3,14,28):
        lagged = history.copy()
        lagged["delivery_date"] += pd.Timedelta(days=days)
        mapping = {target: f"history_coefficient_{name}_lag_{days}d_eur_mwh"
                   for name,target in zip(COEFFICIENT_NAMES,COEFFICIENT_TARGETS)}
        features = features.merge(lagged.rename(columns=mapping),on=HOUR_KEYS,how="left",validate="one_to_one")
        additions["recent_history"] += list(mapping.values())
    timeline = pd.concat([history[HOUR_KEYS],features[HOUR_KEYS]],ignore_index=True).drop_duplicates()
    timeline = timeline.merge(history,on=HOUR_KEYS,how="left",validate="one_to_one")
    timeline["weekday"] = timeline.delivery_date.dt.dayofweek
    timeline = timeline.sort_values(shape_keys+["delivery_date"]).set_index("delivery_date")
    for days in (3,7,14,28,56):
        rolling = timeline.groupby(shape_keys)[COEFFICIENT_TARGETS].rolling(f"{days}D",closed="left",min_periods=1)
        statistics = {"mean": rolling.mean()} if days in (3,14,56) else {"median": rolling.median(), "std": rolling.std(ddof=0)}
        for statistic,values in statistics.items():
            mapping = {target:f"history_coefficient_{name}_{statistic}_{days}d_eur_mwh"
                       for name,target in zip(COEFFICIENT_NAMES,COEFFICIENT_TARGETS)}
            features = features.merge(values.reset_index().rename(columns=mapping),on=HOUR_KEYS,how="left",validate="one_to_one")
            additions["recent_history"] += list(mapping.values())
        if days in (3,14,56):
            column = f"history_coefficient_count_{days}d"
            counts = rolling.count()[COEFFICIENT_TARGETS[0]].rename(column).reset_index()
            features = features.merge(counts,on=HOUR_KEYS,how="left",validate="one_to_one")
            features[column] = features[column].fillna(0).astype(int)
            additions["recent_history"].append(column)
    for days in (28,56):
        rolling = timeline.groupby(shape_keys+["weekday"])[COEFFICIENT_TARGETS].rolling(f"{days}D",closed="left",min_periods=1)
        mapping = {target:f"history_coefficient_{name}_weekday_mean_{days}d_eur_mwh"
                   for name,target in zip(COEFFICIENT_NAMES,COEFFICIENT_TARGETS)}
        means = rolling.mean().reset_index().drop(columns="weekday").rename(columns=mapping)
        column = f"history_coefficient_weekday_count_{days}d"
        counts = rolling.count()[COEFFICIENT_TARGETS[0]].rename(column).reset_index().drop(columns="weekday")
        features = features.merge(means,on=HOUR_KEYS,how="left",validate="one_to_one")
        features = features.merge(counts,on=HOUR_KEYS,how="left",validate="one_to_one")
        features[column] = features[column].fillna(0).astype(int)
        additions["weekday_profile"] += list(mapping.values())+[column]
    for days in (1,7):
        for offset in (-2,-1,1,2):
            neighbor = history.copy()
            neighbor["delivery_date"] += pd.Timedelta(days=days)
            neighbor["hour_index"] -= offset
            label = f"{'previous' if offset<0 else 'next'}_{abs(offset)}h"
            mapping = {target:f"history_coefficient_{name}_lag_{days}d_{label}_eur_mwh"
                       for name,target in zip(COEFFICIENT_NAMES,COEFFICIENT_TARGETS)}
            features = features.merge(neighbor.rename(columns=mapping),on=HOUR_KEYS,how="left",validate="one_to_one")
            additions["neighbor_history"] += list(mapping.values())
        for zone in EXPECTED_ZONES:
            other = history[history.market_id.eq(zone)].drop(columns="market_id").copy()
            other["delivery_date"] += pd.Timedelta(days=days)
            mapping = {target:f"crosszone_history_{zone}_{name}_minus_local_lag_{days}d_eur_mwh"
                       for name,target in zip(COEFFICIENT_NAMES,COEFFICIENT_TARGETS)}
            features = features.merge(other.rename(columns=mapping),on=["delivery_date","hour_index"],how="left",validate="many_to_one")
            for name,target in zip(COEFFICIENT_NAMES,COEFFICIENT_TARGETS):
                features[mapping[target]] -= features[f"history_coefficient_{name}_lag_{days}d_eur_mwh"]
            additions["crosszone_history"] += list(mapping.values())
    return features.sort_values(HOUR_KEYS).reset_index(drop=True), reference, additions


def coefficient_feature_selections(original_columns, additions):
    variants = {"control":list(original_columns)}
    variants.update({name:list(original_columns)+columns for name,columns in additions.items()})
    histories = [column for name,columns in additions.items() if name != "curve" for column in columns]
    variants["all_histories"] = list(original_columns)+histories
    variants["all"] = list(original_columns)+additions["curve"]+histories
    for columns in variants.values():
        if len(columns)!=len(set(columns)) or set(columns).intersection(COEFFICIENT_TARGETS):
            raise ValueError("Feature selections must be unique and must exclude target labels.")
    return variants


def predict_coefficient_features(bundle, test_hourly, history_15min):
    features,reference,_ = build_expanded_coefficient_features(test_hourly,history_15min)
    deviations = coefficient_component_design(predict_trial_coefficients(bundle,features),bundle["basis"]) @ bundle["weights"]
    return make_predictions(reference,test_hourly,deviations,reference_type="pchip",reference_column=COEFFICIENT_REFERENCE)


In [10]:
coefficient_feature_dir = output_dir / "coefficient_features"
coefficient_feature_dir.mkdir(parents=True,exist_ok=True)
feature_training_hourly = train_hourly[train_hourly.delivery_date.lt(validation_start)]
feature_training,_ ,coefficient_additions = build_expanded_coefficient_features(feature_training_hourly,prices)
feature_training = feature_training.merge(residual_coefficients(prices[prices.delivery_date.lt(validation_start)],"pchip"),on=HOUR_KEYS,validate="one_to_one")
feature_validation,feature_reference,_ = build_expanded_coefficient_features(validation_hourly,prices)
original_coefficient_columns = json.loads((output_dir/"lightgbm_coefficients_pchip.json").read_text())["columns"]
COEFFICIENT_FEATURE_VARIANTS = coefficient_feature_selections(original_coefficient_columns,coefficient_additions)
feature_training_dates = np.sort(fit_rows.delivery_date.unique())
coefficient_feature_folds = [(pd.Timestamp(feature_training_dates[-60]),pd.Timestamp(feature_training_dates[-30])),
                             (pd.Timestamp(feature_training_dates[-30]),validation_start)]
feature_model_config = dict(id="pchip_polynomial_lgbm_mae",method="pchip",basis="polynomial",learner="lgbm_mae",family="lightgbm")
coefficient_feature_cv_records = []
for variant,columns in COEFFICIENT_FEATURE_VARIANTS.items():
    for fold,(start,end) in enumerate(coefficient_feature_folds,1):
        earlier = feature_training[feature_training.delivery_date.lt(start)].reset_index(drop=True)
        query = feature_training[feature_training.delivery_date.ge(start)&feature_training.delivery_date.lt(end)].reset_index(drop=True)
        bundle = fit_coefficient_trial(feature_model_config,earlier,earlier[COEFFICIENT_TARGETS].to_numpy(),columns)
        design = coefficient_component_design(predict_trial_coefficients(bundle,query),bundle["basis"])
        mae = reconstructed_coefficient_mae(query,query[COEFFICIENT_TARGETS].to_numpy()@COEFFICIENT_BASIS,design,np.repeat(.8,3))
        coefficient_feature_cv_records.append(dict(feature_set=variant,feature_count=len(columns),fold=fold,
            training_dates=earlier.delivery_date.nunique(),validation_dates=query.delivery_date.nunique(),
            first_validation_date=str(start.date()),end_exclusive=str(end.date()),mae_eur_mwh=mae))
    pd.DataFrame(coefficient_feature_cv_records).to_csv(coefficient_feature_dir/"chronological_folds.csv",index=False)
    print("Completed feature comparison:",variant,flush=True)
coefficient_feature_cv = pd.DataFrame(coefficient_feature_cv_records)
coefficient_feature_cv_summary = coefficient_feature_cv.groupby(["feature_set","feature_count"],as_index=False).mae_eur_mwh.mean().rename(columns={"mae_eur_mwh":"cv_mae_eur_mwh"}).sort_values("cv_mae_eur_mwh")
coefficient_feature_cv_summary.to_csv(coefficient_feature_dir/"chronological_summary.csv",index=False)
selected_coefficient_features = coefficient_feature_cv_summary.iloc[0].feature_set
print("Best feature selection on the two earlier blocks:",selected_coefficient_features)
display(coefficient_feature_cv_summary.round(6))


Completed feature comparison: control


Completed feature comparison: curve


Completed feature comparison: recent_history


Completed feature comparison: weekday_profile


Completed feature comparison: neighbor_history


Completed feature comparison: crosszone_history


Completed feature comparison: all_histories


Completed feature comparison: all


Best feature selection on the two earlier blocks: curve


,feature_set,feature_count,cv_mae_eur_mwh
4,curve,88,3.761878
2,control,50,3.767353
0,all,180,3.771975
6,recent_history,86,3.776897
3,crosszone_history,74,3.777777
7,weekday_profile,58,3.779705
5,neighbor_history,74,3.782854
1,all_histories,142,3.788398


In [11]:
coefficient_feature_results,coefficient_feature_models = {},{}
for variant,columns in COEFFICIENT_FEATURE_VARIANTS.items():
    if variant=="control":
        continue  # Existing 50-feature PCHIP × 0.8 export is the identical control.
    bundle = fit_coefficient_trial(feature_model_config,feature_training,feature_training[COEFFICIENT_TARGETS].to_numpy(),columns)
    bundle.update(weights=np.repeat(.8,3),feature_set=variant,groups={
        group:[c for c in columns if c.startswith(group+"_")]
        for group in ("position","calendar","local","daily","temporal","crosszone","regime","reference","history_coefficient")})
    name = f"coefficient_features_{variant}"
    save_coefficient_search(bundle,coefficient_feature_dir/f"{name}.joblib")
    design = coefficient_component_design(predict_trial_coefficients(bundle,feature_validation),bundle["basis"])
    result = evaluate_experiment(name,feature_reference,validation_hourly,validation_truth,
        predicted_deviations=design@bundle["weights"],reference_type="pchip",reference_column=COEFFICIENT_REFERENCE)
    cv_mae = coefficient_feature_cv_summary.set_index("feature_set").loc[variant,"cv_mae_eur_mwh"]
    result["summary"].update(model="coefficient_features",feature_set=variant,feature_count=len(columns),
        cv_mae_eur_mwh=float(cv_mae),selected_by_cv=variant==selected_coefficient_features)
    coefficient_feature_results[name] = experiments[name] = result
    coefficient_feature_models[name] = models[name] = bundle
coefficient_feature_final_table = pd.DataFrame([r["summary"] for r in coefficient_feature_results.values()])
control_result = dict(experiments["lightgbm_pchip_coefficients_weight_0.8"]["summary"],feature_set="control",
    cv_mae_eur_mwh=float(coefficient_feature_cv_summary.set_index("feature_set").loc["control","cv_mae_eur_mwh"]),selected_by_cv=selected_coefficient_features=="control")
coefficient_feature_comparison = pd.concat([pd.DataFrame([control_result]),coefficient_feature_final_table],ignore_index=True)
control_mae = control_result["mae_eur_mwh"]
coefficient_feature_comparison["mae_improvement_vs_control"] = control_mae-coefficient_feature_comparison.mae_eur_mwh
coefficient_feature_comparison.to_csv(coefficient_feature_dir/"final_comparison.csv",index=False)
print("Final 60-date comparison; positive improvement means lower MAE:")
display(coefficient_feature_comparison[["feature_set","feature_count","cv_mae_eur_mwh","selected_by_cv","mae_eur_mwh","mae_improvement_vs_control","constraints_passed"]].round(6))
coefficient_feature_candidates = {"control":experiments["lightgbm_pchip_coefficients_weight_0.8"],
    **{r["summary"]["feature_set"]:r for r in coefficient_feature_results.values()}}
best_coefficient_features = min(coefficient_feature_candidates.values(),key=lambda r:r["summary"]["mae_eur_mwh"])
print("Best final-period feature experiment:",best_coefficient_features["summary"]["experiment"])
print("Earlier-block selected feature set:",selected_coefficient_features)
display(best_coefficient_features["scores"]["by_zone"].round(6))
worst_coefficient_feature_zone = best_coefficient_features["scores"]["by_zone"].sort_values("mae_eur_mwh",ascending=False).iloc[0].market_id
print("MAE by hour for worst zone:",worst_coefficient_feature_zone)
worst_coefficient_feature_hourly = zone_hourly_mae(best_coefficient_features["predictions"],validation_truth,worst_coefficient_feature_zone)
display(worst_coefficient_feature_hourly.round(6))
feature_manifest["coefficient_features"] = {"additions":coefficient_additions,"variants":COEFFICIENT_FEATURE_VARIANTS,
    "groups":{group:[c for c in COEFFICIENT_FEATURE_VARIANTS["all"] if c.startswith(group+"_")]
              for group in ("position","calendar","local","daily","temporal","crosszone","regime","reference","history_coefficient")}}
(coefficient_feature_dir/"feature_sets.json").write_text(json.dumps(feature_manifest["coefficient_features"],indent=2)+"\n")
(output_dir/"feature_sets.json").write_text(json.dumps(feature_manifest,indent=2)+"\n")


coefficient_features_curve: MAE = 4.816225 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


coefficient_features_recent_history: MAE = 4.795485 EUR/MWh | constraints: PASS | max hourly error = 2.51e-11


coefficient_features_weekday_profile: MAE = 4.802966 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


coefficient_features_neighbor_history: MAE = 4.815472 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11


coefficient_features_crosszone_history: MAE = 4.816234 EUR/MWh | constraints: PASS | max hourly error = 2.51e-11


...

Best final-period feature experiment: coefficient_features_all
Earlier-block selected feature set: curve


,market_id,mae_eur_mwh
0,ZONE_0,3.502340
1,ZONE_1,6.040857
2,ZONE_2,5.315574
3,ZONE_3,4.319145


MAE by hour for worst zone: ZONE_1


,hour_index,mae_eur_mwh,validation_days
0,1,4.830123,60
1,2,3.396557,60
2,3,2.193784,60
3,4,2.021449,60
4,5,2.618109,60
5,6,3.557534,60
6,7,4.641163,60
7,8,6.756789,60
8,9,8.301207,60
9,10,8.185473,60


113386

### 4.12 Zone-specific coefficient models

Compare the shared **PCHIP + polynomial basis + LightGBM × 0.8** model with four independent zone models. Each zone has three coefficient regressors, for **12 regressors total**. Keep the original 50 predictors and all LightGBM settings fixed. `position_zone` is constant within each zone model; known cross-zone hourly context remains available. Each zone model fits only its own zone's residual labels: 7,272 hourly examples over 303 dates, versus 29,088 examples for the shared model.

Use the same two earlier chronological blocks (243/273 fit dates, then 30 evaluation dates each), then fit all four models through August 1 and evaluate the same final 60 dates. Shared-model predictions for those blocks and the final period are reused from the identical earlier control. Histories use strictly earlier actual observations, including previous evaluation dates; learned model weights stay fixed within each block. No zone-specific tuning, correction weights or selection of different model types by zone is performed.

Positive MAE improvements below favor the zone-specific model. Compare overall and per-zone MAEs, and both models by hour for the worst zone of the better final-period model. Saved bundles route each prediction row to its zone automatically: load `outputs/experiments/coefficient_zones/zone_models.joblib` with `load_coefficient_search`, then call `predict_zone_coefficients(bundle, test_hourly, history_15min)`. All fitting, feature construction and inference remain in this notebook.


In [8]:
def fit_zone_coefficients(training_rows, columns):
    """Fit three fixed polynomial-coefficient regressors per zone, on own-zone labels."""
    if set(training_rows.market_id.unique()) != set(EXPECTED_ZONES):
        raise ValueError("Training must contain all four official zones.")
    config = dict(id="pchip_polynomial_lgbm_mae", method="pchip", basis="polynomial", learner="lgbm_mae", family="lightgbm")
    children = {}
    for zone in EXPECTED_ZONES:
        zone_rows = training_rows[training_rows.market_id.eq(zone)].reset_index(drop=True)
        child = fit_coefficient_trial(config, zone_rows, zone_rows[COEFFICIENT_TARGETS].to_numpy(), columns)
        child.update(training_zone=zone, training_rows=len(zone_rows), training_dates=zone_rows.delivery_date.nunique())
        children[zone] = child
    return dict(zone_models=children, columns=list(columns), basis=COEFFICIENT_BASIS.copy(),
                weights=np.repeat(.8,3), method="pchip", zone_categories=list(EXPECTED_ZONES),
                parameters=COEFFICIENT_PARAMETERS.copy(), training_last_date=str(training_rows.delivery_date.max().date()))


def routed_zone_coefficients(bundle, features):
    """Preserve input row order while routing each row to its zone's three regressors."""
    if features.market_id.isna().any() or not features.market_id.isin(EXPECTED_ZONES).all():
        raise ValueError("Prediction rows require a valid official zone.")
    coefficients = np.empty((len(features),3),dtype=float)
    for zone in EXPECTED_ZONES:
        mask = features.market_id.eq(zone).to_numpy()
        if mask.any():
            coefficients[mask] = predict_trial_coefficients(bundle["zone_models"][zone],features.loc[mask])
    if not np.isfinite(coefficients).all():
        raise ValueError("Coefficient predictions must be finite.")
    return coefficients


def predict_zone_coefficients(bundle, test_hourly, history_15min):
    features, reference = build_coefficient_features(test_hourly,history_15min,"pchip")
    coefficients = routed_zone_coefficients(bundle,features)
    deviations = coefficient_component_design(coefficients,bundle["basis"]) @ bundle["weights"]
    return make_predictions(reference,test_hourly,deviations,reference_type="pchip",reference_column=COEFFICIENT_REFERENCE)


In [9]:
coefficient_zone_dir = output_dir / "coefficient_zones"
coefficient_zone_dir.mkdir(parents=True,exist_ok=True)
coefficient_zone_columns = json.loads((output_dir/"lightgbm_coefficients_pchip.json").read_text())["columns"]
zone_training_hourly = train_hourly[train_hourly.delivery_date.lt(validation_start)]
zone_training,zone_training_reference = build_coefficient_features(zone_training_hourly,prices,"pchip")
zone_training = zone_training.merge(residual_coefficients(prices[prices.delivery_date.lt(validation_start)],"pchip"),on=HOUR_KEYS,validate="one_to_one")
zone_dates = np.sort(fit_rows.delivery_date.unique())
coefficient_zone_folds = [(pd.Timestamp(zone_dates[-60]),pd.Timestamp(zone_dates[-30])),
                          (pd.Timestamp(zone_dates[-30]),validation_start)]
shared_zone_folds = joblib.load(output_dir/"coefficient_search"/"fold_predictions.joblib")["pchip_polynomial_lgbm_mae"]
coefficient_zone_cv_records,coefficient_zone_cv_zone_records = [],[]
for fold,((start,end),shared_fold) in enumerate(zip(coefficient_zone_folds,shared_zone_folds),1):
    earlier = zone_training[zone_training.delivery_date.lt(start)].reset_index(drop=True)
    query = zone_training[zone_training.delivery_date.ge(start)&zone_training.delivery_date.lt(end)].reset_index(drop=True)
    pd.testing.assert_frame_equal(query[HOUR_KEYS],shared_fold["rows"].reset_index(drop=True))
    bundle = fit_zone_coefficients(earlier,coefficient_zone_columns)
    query_reference = zone_training_reference[zone_training_reference.delivery_date.ge(start)&zone_training_reference.delivery_date.lt(end)]
    query_hourly = train_hourly[train_hourly.delivery_date.ge(start)&train_hourly.delivery_date.lt(end)]
    query_truth = prices[prices.delivery_date.ge(start)&prices.delivery_date.lt(end)]
    per_zone_design = coefficient_component_design(routed_zone_coefficients(bundle,query),bundle["basis"])
    for model,deviations in (("shared",shared_fold["design"]@np.repeat(.8,3)),("per_zone",per_zone_design@bundle["weights"])):
        result = evaluate_experiment(f"fold_{fold}_{model}",query_reference,query_hourly,query_truth,
            predicted_deviations=deviations,reference_type="pchip",reference_column=COEFFICIENT_REFERENCE,output_dir=coefficient_zone_dir/"cv")
        coefficient_zone_cv_records.append(dict(model=model,fold=fold,training_dates=earlier.delivery_date.nunique(),
            validation_dates=query.delivery_date.nunique(),first_validation_date=str(start.date()),end_exclusive=str(end.date()),
            mae_eur_mwh=result["scores"]["mae_eur_mwh"]))
        for row in result["scores"]["by_zone"].to_dict("records"):
            coefficient_zone_cv_zone_records.append(dict(model=model,fold=fold,**row))
coefficient_zone_cv = pd.DataFrame(coefficient_zone_cv_records)
coefficient_zone_cv.to_csv(coefficient_zone_dir/"chronological_folds.csv",index=False)
pd.DataFrame(coefficient_zone_cv_zone_records).to_csv(coefficient_zone_dir/"chronological_folds_by_zone.csv",index=False)
coefficient_zone_cv_summary = coefficient_zone_cv.groupby("model",as_index=False).mae_eur_mwh.mean().rename(columns={"mae_eur_mwh":"cv_mae_eur_mwh"})
print("Earlier chronological blocks, using official reconstructed-price MAE:")
display(coefficient_zone_cv_summary.round(6))

coefficient_zone_bundle = fit_zone_coefficients(zone_training,coefficient_zone_columns)
save_coefficient_search(coefficient_zone_bundle,coefficient_zone_dir/"zone_models.joblib")
zone_validation,zone_reference = build_coefficient_features(validation_hourly,prices,"pchip")
zone_design = coefficient_component_design(routed_zone_coefficients(coefficient_zone_bundle,zone_validation),coefficient_zone_bundle["basis"])
coefficient_zone_name = "lightgbm_pchip_coefficients_per_zone"
coefficient_zone_result = evaluate_experiment(coefficient_zone_name,zone_reference,validation_hourly,validation_truth,
    predicted_deviations=zone_design@coefficient_zone_bundle["weights"],reference_type="pchip",reference_column=COEFFICIENT_REFERENCE)
coefficient_zone_result["summary"].update(model="coefficient_zone",feature_set="coefficients_per_zone",feature_count=len(coefficient_zone_columns),
    cv_mae_eur_mwh=float(coefficient_zone_cv_summary.set_index("model").loc["per_zone","cv_mae_eur_mwh"]))
experiments[coefficient_zone_name],models[coefficient_zone_name] = coefficient_zone_result,coefficient_zone_bundle
coefficient_zone_shared = experiments["lightgbm_pchip_coefficients_weight_0.8"]
coefficient_zone_comparison = pd.DataFrame([
    dict(model=model,cv_mae_eur_mwh=float(coefficient_zone_cv_summary.set_index("model").loc[model,"cv_mae_eur_mwh"]),
         mae_eur_mwh=result["summary"]["mae_eur_mwh"],constraints_passed=result["summary"]["constraints_passed"])
    for model,result in (("shared",coefficient_zone_shared),("per_zone",coefficient_zone_result))])
coefficient_zone_comparison["improvement_vs_shared"] = coefficient_zone_shared["summary"]["mae_eur_mwh"]-coefficient_zone_comparison.mae_eur_mwh
coefficient_zone_comparison.to_csv(coefficient_zone_dir/"comparison.csv",index=False)
print("Final 60-date comparison; positive improvement favors per-zone models:")
display(coefficient_zone_comparison.round(6))
coefficient_zone_by_zone = coefficient_zone_shared["scores"]["by_zone"].rename(columns={"mae_eur_mwh":"shared_mae_eur_mwh"}).merge(
    coefficient_zone_result["scores"]["by_zone"].rename(columns={"mae_eur_mwh":"per_zone_mae_eur_mwh"}),on="market_id",validate="one_to_one")
coefficient_zone_by_zone["improvement_vs_shared"] = coefficient_zone_by_zone.shared_mae_eur_mwh-coefficient_zone_by_zone.per_zone_mae_eur_mwh
coefficient_zone_by_zone.to_csv(coefficient_zone_dir/"mae_by_zone.csv",index=False)
display(coefficient_zone_by_zone.round(6))
coefficient_zone_best = min((coefficient_zone_shared,coefficient_zone_result),key=lambda r:r["summary"]["mae_eur_mwh"])
coefficient_zone_worst = coefficient_zone_best["scores"]["by_zone"].sort_values("mae_eur_mwh",ascending=False).iloc[0].market_id
print("Better final-period model:",coefficient_zone_best["summary"]["experiment"],"| worst zone:",coefficient_zone_worst)
coefficient_zone_hourly = zone_hourly_mae(coefficient_zone_shared["predictions"],validation_truth,coefficient_zone_worst).rename(
    columns={"mae_eur_mwh":"shared_mae_eur_mwh"}).merge(
    zone_hourly_mae(coefficient_zone_result["predictions"],validation_truth,coefficient_zone_worst).drop(columns="validation_days").rename(
    columns={"mae_eur_mwh":"per_zone_mae_eur_mwh"}),on="hour_index",validate="one_to_one")
coefficient_zone_hourly["improvement_vs_shared"] = coefficient_zone_hourly.shared_mae_eur_mwh-coefficient_zone_hourly.per_zone_mae_eur_mwh
coefficient_zone_hourly.to_csv(coefficient_zone_dir/"worst_zone_mae_by_hour.csv",index=False)
display(coefficient_zone_hourly.round(6))
feature_manifest["coefficient_zones"] = dict(columns=coefficient_zone_columns,zones=list(EXPECTED_ZONES),basis=COEFFICIENT_BASIS.tolist(),
    correction_weight=.8,parameters=COEFFICIENT_PARAMETERS,regressors_per_zone=3)
(coefficient_zone_dir/"settings.json").write_text(json.dumps(feature_manifest["coefficient_zones"],indent=2)+"\n")
(output_dir/"feature_sets.json").write_text(json.dumps(feature_manifest,indent=2)+"\n")


fold_1_shared: MAE = 3.876819 EUR/MWh | constraints: PASS | max hourly error = 2.51e-11
fold_1_per_zone: MAE = 3.940433 EUR/MWh | constraints: PASS | max hourly error = 2.51e-11


fold_2_shared: MAE = 3.657886 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11
fold_2_per_zone: MAE = 3.671816 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11
Earlier chronological blocks, using official reconstructed-price MAE:


,model,cv_mae_eur_mwh
0,per_zone,3.806124
1,shared,3.767353


lightgbm_pchip_coefficients_per_zone: MAE = 4.852717 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11
Final 60-date comparison; positive improvement favors per-zone models:


,model,cv_mae_eur_mwh,mae_eur_mwh,constraints_passed,improvement_vs_shared
0,shared,3.767353,4.797342,True,0.000000
1,per_zone,3.806124,4.852717,True,-0.055375


,market_id,shared_mae_eur_mwh,per_zone_mae_eur_mwh,improvement_vs_shared
0,ZONE_0,3.478044,3.647769,-0.169725
1,ZONE_1,6.032080,6.011386,0.020694
2,ZONE_2,5.341404,5.382967,-0.041563
3,ZONE_3,4.337842,4.368746,-0.030904


Better final-period model: lightgbm_pchip_coefficients_weight_0.8 | worst zone: ZONE_1


,hour_index,shared_mae_eur_mwh,validation_days,per_zone_mae_eur_mwh,improvement_vs_shared
0,1,4.838810,60,4.769020,0.069790
1,2,3.513619,60,3.487173,0.026446
2,3,2.224424,60,2.412774,-0.188350
3,4,2.031781,60,2.246193,-0.214412
4,5,2.571669,60,2.749470,-0.177801
5,6,3.599051,60,3.549614,0.049438
6,7,4.545374,60,4.793334,-0.247961
7,8,6.402519,60,6.060993,0.341526
8,9,8.513131,60,7.946176,0.566954
9,10,8.424378,60,7.959737,0.464641


116401

### 4.13 September-only validation: shared coefficient model

Use only the shared **PCHIP + polynomial basis + three LightGBM MAE regressors × 0.8**, with the original 50 predictors and fixed settings. Train on the **333 available dates through August 31, 2026** and validate on **September 1–30, 2026**. No extra feature sets, zone-specific models or parameter/weight tuning enter this experiment.

Weights stay fixed for all 30 validation dates. Historical features can use actual prices from strictly earlier dates, including earlier September dates, matching the jury's daily history input. September labels are used for scoring and subsequently available history, never for fitting the model. Hourly inputs for each target day remain known as required by the project.

This changes both the training cutoff and evaluation period relative to the original 60-date comparison, so the scores are reported separately. The original experiment results remain available. Save the inference bundle and official-schema prediction export under `outputs/experiments/september/`; reload `shared_pchip_polynomial_weight_0.8.joblib` with `load_coefficient_search`, then use `predict_coefficient_search(bundle, test_hourly, history_15min)`.


In [8]:
september_start = pd.Timestamp("2026-09-01")
september_end = pd.Timestamp("2026-10-01")
september_dir = output_dir / "september"
september_dir.mkdir(parents=True,exist_ok=True)
september_training_prices = prices[prices.delivery_date.lt(september_start)]
september_training_hourly = train_hourly[train_hourly.delivery_date.lt(september_start)]
september_hourly = train_hourly[train_hourly.delivery_date.ge(september_start)&train_hourly.delivery_date.lt(september_end)]
september_truth = prices[prices.delivery_date.ge(september_start)&prices.delivery_date.lt(september_end)]
september_training,_ = build_coefficient_features(september_training_hourly,prices,"pchip")
september_training = september_training.merge(residual_coefficients(september_training_prices,"pchip"),on=HOUR_KEYS,validate="one_to_one")
september_features,september_reference = build_coefficient_features(september_hourly,prices,"pchip")
september_columns = json.loads((output_dir/"lightgbm_coefficients_pchip.json").read_text())["columns"]
september_config = dict(id="pchip_polynomial_lgbm_mae",method="pchip",basis="polynomial",learner="lgbm_mae",family="lightgbm")
assert september_training.delivery_date.max()<september_start
assert september_training.delivery_date.nunique()==333 and september_features.delivery_date.nunique()==30
assert len(september_columns)==50
september_bundle = fit_coefficient_trial(september_config,september_training,september_training[COEFFICIENT_TARGETS].to_numpy(),september_columns)
september_bundle.update(weights=np.repeat(.8,3),parameters=COEFFICIENT_PARAMETERS.copy(),
    validation_start=str(september_start.date()),validation_end_exclusive=str(september_end.date()))
save_coefficient_search(september_bundle,september_dir/"shared_pchip_polynomial_weight_0.8.joblib")
september_design = coefficient_component_design(predict_trial_coefficients(september_bundle,september_features),september_bundle["basis"])
september_result = evaluate_experiment("shared_pchip_polynomial_weight_0.8",september_reference,september_hourly,september_truth,
    predicted_deviations=september_design@september_bundle["weights"],reference_type="pchip",reference_column=COEFFICIENT_REFERENCE,output_dir=september_dir)
september_result["summary"].update(model="lightgbm_shared",feature_set="original_coefficients",feature_count=len(september_columns),
    training_dates=september_training.delivery_date.nunique(),training_end=september_bundle["training_last_date"],
    validation_start=str(september_start.date()),validation_end="2026-09-30",correction_weight=.8)
september_results = pd.DataFrame([september_result["summary"]])
september_results.to_csv(september_dir/"results.csv",index=False)
print("Training: 333 dates through August 31; validation: September 1–30 only. Original 50 predictors; shared weights; correction 0.8.")
display(september_results[["training_dates","prediction_days","feature_count","mae_eur_mwh","constraints_passed"]].round(6))
september_result["scores"]["by_zone"].to_csv(september_dir/"mae_by_zone.csv",index=False)
display(september_result["scores"]["by_zone"].round(6))
september_worst_zone = september_result["scores"]["by_zone"].sort_values("mae_eur_mwh",ascending=False).iloc[0].market_id
print("MAE by hour for worst September zone:",september_worst_zone)
september_hourly_mae = zone_hourly_mae(september_result["predictions"],september_truth,september_worst_zone)
september_hourly_mae.to_csv(september_dir/"worst_zone_mae_by_hour.csv",index=False)
display(september_hourly_mae.round(6))
feature_manifest["coefficient_september"] = dict(columns=september_columns,parameters=COEFFICIENT_PARAMETERS,
    method="pchip",basis=COEFFICIENT_BASIS.tolist(),correction_weight=.8,training_dates=333,
    training_end="2026-08-31",validation_start="2026-09-01",validation_end="2026-09-30")
(september_dir/"settings.json").write_text(json.dumps(feature_manifest["coefficient_september"],indent=2)+"\n")
(output_dir/"feature_sets.json").write_text(json.dumps(feature_manifest,indent=2)+"\n")


shared_pchip_polynomial_weight_0.8: MAE = 5.149084 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11
Training: 333 dates through August 31; validation: September 1–30 only. Original 50 predictors; shared weights; correction 0.8.


,training_dates,prediction_days,feature_count,mae_eur_mwh,constraints_passed
0,333,30,50,5.149084,True


,market_id,mae_eur_mwh
0,ZONE_0,3.874773
1,ZONE_1,6.861952
2,ZONE_2,5.500973
3,ZONE_3,4.358636


MAE by hour for worst September zone: ZONE_1


,hour_index,mae_eur_mwh,validation_days
0,1,5.888132,30
1,2,4.336334,30
2,3,2.617917,30
3,4,2.798974,30
4,5,3.183839,30
5,6,4.302613,30
6,7,5.904913,30
7,8,7.310573,30
8,9,9.367424,30
9,10,10.150721,30


119463

### 4.14 Summary

Validation MAE is in EUR/MWh; lower is better. The main tables and audit compare the 69 experiments evaluated on August 2–September 30, with training through August 1. The September-only shared-model result is shown separately at the end, with its own training cutoff (August 31), validation dates and constraint audit; it is not ranked against the 60-date scores.


In [9]:
results = pd.DataFrame([result["summary"] for result in experiments.values()])
results.to_csv(output_dir / "results.csv", index=False)
baselines = results[results.model.eq("baseline")].pivot(
    index="reference_type", columns="feature_set", values="mae_eur_mwh",
)
comparison = results[~results.model.eq("baseline")].pivot(
    index="feature_set", columns=["model", "reference_type"], values="mae_eur_mwh",
).reindex(list(EXPERIMENTS["smooth"]) + ["history_only_tuned", "history_per_zone", "history_regime_per_zone", "history_calendar", "history_calendar_hourly", "history_calendar_temporal", "cnn_catboost_history", "cnn_catboost_tabm", "history_14d", "history_28d", "coefficients_weight_1.0", "coefficients_weight_0.8", "selected_coefficients", "curve", "recent_history", "weekday_profile", "neighbor_history", "crosszone_history", "all_histories", "coefficients_per_zone"])
display(baselines.round(6))
display(comparison.round(6))

checks = []
for result in results.itertuples(index=False):
    prediction = pd.read_csv(result.prediction_file, parse_dates=["delivery_date"])
    report = validate_prediction_output(prediction, validation_hourly)
    mae = challenge_mae(prediction, validation_truth, validation_hourly)["mae_eur_mwh"]
    np.testing.assert_allclose(mae, result.mae_eur_mwh, atol=1e-10, rtol=0)
    checks.append({"experiment": result.experiment, **report, "mae_eur_mwh": mae})
audit = pd.DataFrame(checks)
audit.to_csv(output_dir / "requirements_audit.csv", index=False)
print(f"All {len(audit)} exported experiments pass. Maximum hourly error: {audit.max_hourly_error_eur_mwh.max():.3g} EUR/MWh.")
best = results.loc[results.mae_eur_mwh.idxmin()]
print(f"Best validation MAE: {best.mae_eur_mwh:.6f} — {best.experiment}")

print("September-only result (333 training dates; different evaluation period from the tables above):")
september_checks = validate_prediction_output(september_result["predictions"],september_hourly)
september_mae = challenge_mae(september_result["predictions"],september_truth,september_hourly)["mae_eur_mwh"]
np.testing.assert_allclose(september_mae,september_result["summary"]["mae_eur_mwh"],atol=1e-10,rtol=0)
pd.DataFrame([dict(experiment=september_result["summary"]["experiment"],**september_checks,mae_eur_mwh=september_mae)]).to_csv(september_dir/"requirements_audit.csv",index=False)
display(september_results[["training_end","validation_start","validation_end","feature_count","mae_eur_mwh","constraints_passed"]].round(6))


feature_set,historical_profile,reference
reference_type,,
hourly,6.828230,9.753915
pchip,NaN,6.655777
smooth,5.319968,6.495620


model                              ridge            catboost            \
reference_type                    hourly    smooth    hourly    smooth   
feature_set                                                              
calendar                        6.777083  5.307619  6.705284  5.207788   
calendar_hourly                 5.388596  5.342185  5.418144  5.201000   
calendar_hourly_history         5.360577  5.171910  6.734017  5.331624   
calendar_hourly_both_histories  5.197943  5.150703  6.668349  5.537589   
all                             5.206980  5.167820  6.032024  5.083516   
history_only                    6.216592  5.146096  6.131472  5.068866   
history_zone                         NaN  5.155984       NaN  5.071473   
history_regime                       NaN  5.320902       NaN  5.496317   
history_zone_regime                  NaN  5.345835       NaN  5.491556   
history_only_tuned                   NaN       NaN       NaN  5.068817   
history_per_zone                     NaN       NaN       NaN  5.086959   
history_regime_per_zone              NaN       NaN       NaN  5.379953   
history_calendar                     NaN       NaN       NaN       NaN   
history_calendar_hourly              NaN       NaN       NaN       NaN   
history_calendar_temporal            NaN       NaN       NaN       NaN   
cnn_catboost_history                 NaN       NaN       NaN       NaN   
cnn_catboost_tabm                    NaN       NaN       NaN       NaN   
history_14d                          NaN       NaN       NaN       NaN   
history_28d                          NaN       NaN       NaN       NaN   
coefficients_weight_1.0              NaN       NaN       NaN       NaN   
coefficients_weight_0.8              NaN       NaN       NaN       NaN   
selected_coefficients                NaN       NaN       NaN       NaN   
curve                                NaN       NaN       NaN       NaN   
recent_history                       NaN       NaN       NaN       NaN   
weekday_profile                      NaN       NaN       NaN       NaN   
neighbor_history                     NaN       NaN       NaN       NaN   
crosszone_history                    NaN       NaN       NaN       NaN   
all_histories                        NaN       NaN       NaN       NaN   
coefficients_per_zone                NaN       NaN       NaN       NaN   

model                                mlp      cnn      tabm     blend  \
reference_type                    smooth   smooth    smooth    smooth   
feature_set                                                             
calendar                             NaN      NaN       NaN       NaN   
calendar_hourly                      NaN      NaN       NaN       NaN   
calendar_hourly_history              NaN      NaN       NaN       NaN   
calendar_hourly_both_histories       NaN      NaN       NaN       NaN   
all                                  NaN      NaN       NaN       NaN   
history_only                    5.063933      NaN       NaN       NaN   
history_zone                         NaN      NaN       NaN       NaN   
history_regime                       NaN      NaN       NaN       NaN   
history_zone_regime                  NaN      NaN       NaN       NaN   
history_only_tuned                   NaN      NaN       NaN       NaN   
history_per_zone                     NaN      NaN       NaN       NaN   
history_regime_per_zone              NaN      NaN       NaN       NaN   
history_calendar                5.036322      NaN  5.038548       NaN   
history_calendar_hourly         4.986964  4.95124  4.878572       NaN   
history_calendar_temporal       5.067417  5.00366       NaN       NaN   
cnn_catboost_history                 NaN      NaN       NaN  4.823639   
cnn_catboost_tabm                    NaN      NaN       NaN  4.784831   
history_14d                          NaN      NaN       NaN       NaN   
history_28d                          NaN      NaN       NaN       NaN   
coefficients_weight_1.0  

All 69 exported experiments pass. Maximum hourly error: 5e-11 EUR/MWh.
Best validation MAE: 4.784831 — blend_smooth_cnn_catboost_tabm
September-only result (333 training dates; different evaluation period from the tables above):


,training_end,validation_start,validation_end,feature_count,mae_eur_mwh,constraints_passed
0,2026-08-31,2026-09-01,2026-09-30,50,5.149084,True


### 4.15 Reconciliation with AET-HACKATHON

The AET-HACKATHON final model (PCHIP + 0.8 × three LightGBM coefficient regressors) reports **5.004** on its locked September block, while Section 4.13 gives **5.149**. That comparison is not like-for-like:

| | AET-HACKATHON locked block | Section 4.13 |
| --- | --- | --- |
| Training | through 31 Aug | through 31 Aug |
| Scored dates | **19–30 Sep (12 days)** | 1–30 Sep (30 days) |
| Data | docs say "Energy-Charts replica", but its stored artifacts contain this official file; flat (10.899) and PCHIP (6.927) reproduce exactly here, with zone labels permuted | official file |

Re-running AET's own code on this file gives 5.033 on 19–30 Sep. The 0.03 gap to 5.004 comes only from the zone relabelling, so treat differences below about 0.03 as noise. On both of our windows AET is still about 1 % better, so we compare the methods, not the reported numbers. This section uses one set of folds for every candidate:

* **8 monthly folds** (AET development protocol): train through the end of month *m*, score days 19–end of month *m+1*, January–August 2026. Main evidence, pooled with the official metric.
* **Sep 1–30** (train through 31 Aug) and **Aug 2–Sep 30** (train through 1 Aug), the windows used in Sections 4.9–4.13. These overlap and were inspected many times, so they are diagnostics only.

The implementations differ in three ways, ablated one at a time:

1. **Reference.** AET interpolates the *cumulative* hourly curve `C(h)` with PCHIP at integer-hour knots and differences it on the quarter grid. Hourly means are exact by telescoping, so no re-centring is needed. Section 4.9 interpolates prices at hour *centres*, uses constant extensions at the edges, and then shifts every hour to its mean. That shift creates artificial jumps at hour boundaries, and this reference is 2 % worse on its own (6.656 vs 6.516 on Aug 2–Sep 30).
2. **LightGBM settings.** Section 4.9: 800 trees, 31 leaves, learning rate 0.03, 60 rows per leaf, L2 = 5. AET: 500 trees, 15 leaves, learning rate 0.05, 100 rows per leaf.
3. **Features.** Section 4.9 fills out-of-day neighbours with the current price, so edge slopes become 0 ("flat") rather than missing. It also has four extra encodings of the season (`calendar_month`, `calendar_day_of_year`, month sin/cos). With one year of data, September's day-of-year (244–273) never occurs in training through August. AET instead uses missing edge slopes, slopes at ±2 hours, level relative to the day mean, and every zone's *next-hour* slope.

Requires `lightgbm` (`pip install -r requirements.txt`). Run Sections 1–3 and the 4.9 definitions cell first. AET-HACKATHON is imported read-only with bytecode writing disabled, so the folder is never modified. Runtime is about 6–8 minutes.

In [ ]:
# 4.15a Cumulative PCHIP reference (same construction as AET-HACKATHON aet/baselines.py::cumulative).
from scipy.interpolate import PchipInterpolator
from lightgbm import LGBMRegressor

def cumulative_pchip_quarters(curve):
    """[n, 24] hourly prices -> [n, 24, 4]. PCHIP through C(0)=0, C(h)=sum of the first h hours,
    differenced on the quarter grid; the four quarters average to the hourly price by telescoping."""
    curve = np.asarray(curve, dtype=float)
    cumulative = np.column_stack([np.zeros(len(curve)), np.cumsum(curve, axis=1)])
    interpolant = PchipInterpolator(np.arange(25), cumulative, axis=1)
    return (np.diff(interpolant(np.arange(97) / 4), axis=1) * 4).reshape(-1, 24, 4)

# Add a "cumpchip" method; "smooth" and "pchip" keep their Section 4.9 behaviour.
_coefficient_reference_original = globals().get("_coefficient_reference_original", coefficient_reference)

def coefficient_reference(hourly_inputs, method):
    if method != "cumpchip":
        return _coefficient_reference_original(hourly_inputs, method)
    reference = _coefficient_reference_original(hourly_inputs, "smooth")
    hourly = hourly_inputs.copy()
    hourly["delivery_date"] = pd.to_datetime(hourly.delivery_date)
    curve = hourly.pivot(index=DAY_KEYS, columns="hour_index", values="hourly_price_eur_mwh")
    reference[COEFFICIENT_REFERENCE] = cumulative_pchip_quarters(curve.reindex(columns=range(1, 25)).sort_index().to_numpy()).reshape(-1)
    return reference

check_curve = train_hourly.pivot(index=DAY_KEYS, columns="hour_index", values="hourly_price_eur_mwh").to_numpy()
np.testing.assert_allclose(cumulative_pchip_quarters(check_curve).mean(axis=2), check_curve, atol=1e-9, rtol=0)

In [ ]:
# 4.15b AET-style curve features added to the Section 4.9 hourly table (same-day hourly inputs only).
RECONCILE_EDGE_FALLBACKS = [
    "local_prev_hour_price_eur_mwh", "local_next_hour_price_eur_mwh",
    "temporal_hourly_change_from_previous_eur_mwh", "temporal_hourly_change_to_next_eur_mwh",
    "temporal_hourly_curvature_eur_mwh", *[f"crosszone_{zone}_change_from_previous_eur_mwh" for zone in EXPECTED_ZONES],
]
RECONCILE_SEASON_DUPLICATES = ["calendar_month", "calendar_day_of_year", "calendar_month_sin", "calendar_month_cos"]


def add_reconcile_features(features):
    """Edge-aware slopes: neighbours outside the delivery day stay NaN instead of copying the current price."""
    features = features.sort_values(HOUR_KEYS).reset_index(drop=True)
    price = features.local_hourly_price_eur_mwh
    by_day = features.groupby(DAY_KEYS).local_hourly_price_eur_mwh
    hm2, hm1, hp1, hp2 = (by_day.shift(lag) for lag in (2, 1, -1, -2))
    features["temporal_slope_minus_2_eur_mwh"] = hm1 - hm2
    features["temporal_slope_minus_1_eur_mwh"] = price - hm1
    features["temporal_slope_plus_1_eur_mwh"] = hp1 - price
    features["temporal_slope_plus_2_eur_mwh"] = hp2 - hp1
    features["temporal_curvature_nan_edges_eur_mwh"] = (hp1 - price) - (price - hm1)
    features["daily_relative_level_eur_mwh"] = price - features.daily_mean_price_eur_mwh
    for zone in EXPECTED_ZONES:
        column = f"crosszone_{zone}_hourly_price_eur_mwh"
        by_zone_day = features.groupby(DAY_KEYS)[column]
        features[f"crosszone_{zone}_slope_minus_1_eur_mwh"] = features[column] - by_zone_day.shift(1)
        features[f"crosszone_{zone}_slope_plus_1_eur_mwh"] = by_zone_day.shift(-1) - features[column]
    return features


RECONCILE_GROUPS = ("position", "calendar", "local", "daily", "temporal", "crosszone", "regime", "reference", "history_coefficient")
reconcile_tables = {}
for method in ("pchip", "cumpchip"):
    table, _ = build_coefficient_features(train_hourly, prices, method)
    table = table.merge(residual_coefficients(prices, method), on=HOUR_KEYS, validate="one_to_one")
    reconcile_tables[method] = add_reconcile_features(table)

new_columns = [c for c in reconcile_tables["cumpchip"] if c.startswith(("temporal_slope_", "temporal_curvature_nan", "daily_relative"))
               or (c.startswith("crosszone_") and "_slope_" in c)]
original_columns = [c for group in RECONCILE_GROUPS for c in reconcile_tables["pchip"]
                    if c.startswith(group + "_") and c not in new_columns]
assert len(original_columns) == 50
revised_columns = [c for c in original_columns if c not in RECONCILE_EDGE_FALLBACKS + RECONCILE_SEASON_DUPLICATES] + new_columns
print(f"Original predictors: {len(original_columns)}; revised predictors: {len(revised_columns)}")

In [ ]:
# 4.15c Shared folds, Section 4.9 runner and AET-HACKATHON runner (imported read-only).
import sys

def month_end(timestamp):
    return timestamp + pd.offsets.MonthEnd(0)

RECONCILE_FOLDS = []
for month in pd.date_range("2026-01-01", "2026-08-01", freq="MS"):
    RECONCILE_FOLDS.append((f"m{month:%Y_%m}", month - pd.Timedelta(days=1), month + pd.Timedelta(days=18), month_end(month)))
RECONCILE_FOLDS += [("sep_full", pd.Timestamp("2026-08-31"), pd.Timestamp("2026-09-01"), pd.Timestamp("2026-09-30")),
                    ("aug2_sep30", pd.Timestamp("2026-08-01"), pd.Timestamp("2026-08-02"), pd.Timestamp("2026-09-30"))]
DEVELOPMENT_FOLDS = [name for name, *_ in RECONCILE_FOLDS if name.startswith("m")]
AET_LGBM = dict(learning_rate=0.05, num_leaves=15, min_child_samples=100, reg_lambda=0.0, n_estimators=500)
NOTEBOOK_LGBM = {key: COEFFICIENT_PARAMETERS[key] for key in ("learning_rate", "num_leaves", "min_child_samples", "reg_lambda", "n_estimators")}


def run_notebook_coefficients(method, columns, lgbm, weight=0.8):
    """Fit three coefficient regressors per fold; score day/zone MAE of reconstructed prices."""
    table, records = reconcile_tables[method], []
    parameters = {**COEFFICIENT_PARAMETERS, **lgbm}
    for fold, train_end, start, end in RECONCILE_FOLDS:
        train = table[table.delivery_date.le(train_end)]
        query = table[table.delivery_date.between(start, end)]
        coefficients = np.column_stack([
            LGBMRegressor(**parameters).fit(train[columns], train[target], categorical_feature=["position_zone"]).predict(query[columns])
            for target in COEFFICIENT_TARGETS])
        residuals = query[COEFFICIENT_TARGETS].to_numpy() @ COEFFICIENT_BASIS
        errors = np.abs(residuals - weight * coefficients @ COEFFICIENT_BASIS).mean(axis=1)
        records.append(query[DAY_KEYS].assign(fold=fold, mae_eur_mwh=errors)
                       .groupby(["fold", *DAY_KEYS], as_index=False).mae_eur_mwh.mean())
    return pd.concat(records, ignore_index=True)


AET_ROOT = Path("AET-HACKATHON").resolve()
previous_bytecode_setting, sys.dont_write_bytecode = sys.dont_write_bytecode, True   # keep AET-HACKATHON untouched
if str(AET_ROOT) not in sys.path:
    sys.path.insert(0, str(AET_ROOT))
from aet.cube import QuarterCube
from aet.features import FeatureConfig
from aet.pipeline import PipelineConfig, ShapePipeline
sys.dont_write_bytecode = previous_bytecode_setting
aet_cube = QuarterCube.from_long(prices.assign(delivery_date=prices.delivery_date.dt.strftime("%Y-%m-%d")), EXPECTED_ZONES)


def run_aet_pipeline(lgbm, prev_day_edge=False, scale=0.8):
    """AET-HACKATHON ShapePipeline: cumulative PCHIP + its own features; history replay per scored date."""
    params = dict(learning_rate=lgbm["learning_rate"], num_leaves=lgbm["num_leaves"],
                  min_data_in_leaf=lgbm["min_child_samples"], lambda_l2=lgbm["reg_lambda"])
    config = PipelineConfig(name="aet", baseline="pchip", basis="poly", model="lgbm_coef", model_params=params,
                            num_boost_round=lgbm["n_estimators"], residual_scale=scale,
                            features=FeatureConfig(prev_day_edge=prev_day_edge))
    hourly, records = aet_cube.hourly(), []
    for fold, train_end, start, end in RECONCILE_FOLDS:
        pipe = ShapePipeline(config).fit(aet_cube.until(np.datetime64(train_end.date())))
        scored = np.flatnonzero((aet_cube.dates >= np.datetime64(start.date())) & (aet_cube.dates <= np.datetime64(end.date())))
        for i in scored:
            predicted, _ = pipe.predict_day(aet_cube.dates[i], hourly[i], aet_cube)   # history < date inside
            for z, zone in enumerate(aet_cube.zones):
                records.append(dict(fold=fold, delivery_date=pd.Timestamp(str(aet_cube.dates[i])), market_id=zone,
                                    mae_eur_mwh=np.abs(predicted[z] - aet_cube.values[i, z]).mean()))
    return pd.DataFrame(records)

In [ ]:
# 4.15d One change at a time, same folds and official metric (day -> zone -> mean of four zones).
reconcile_dir = output_dir / "aet_reconciliation"
reconcile_dir.mkdir(parents=True, exist_ok=True)
RECONCILE_CANDIDATES = {
    "4.9 control: centre PCHIP, 50 features, notebook LightGBM": lambda: run_notebook_coefficients("pchip", original_columns, NOTEBOOK_LGBM),
    "+ cumulative PCHIP": lambda: run_notebook_coefficients("cumpchip", original_columns, NOTEBOOK_LGBM),
    "+ cumulative PCHIP + revised features": lambda: run_notebook_coefficients("cumpchip", revised_columns, NOTEBOOK_LGBM),
    "AET-HACKATHON final (I2), its own code": lambda: run_aet_pipeline(AET_LGBM),
    "AET-HACKATHON features + notebook LightGBM": lambda: run_aet_pipeline(NOTEBOOK_LGBM),
}
reconcile_per_day = {}
for name, run in RECONCILE_CANDIDATES.items():
    reconcile_per_day[name] = run()
    print("Completed:", name, flush=True)


def official_score(per_day, folds):
    rows = per_day[per_day.fold.isin(folds)]
    return float(rows.groupby("market_id").mae_eur_mwh.mean().mean())


def fold_scores(per_day):
    return per_day.groupby(["fold", "market_id"]).mae_eur_mwh.mean().groupby("fold").mean()


control_name = next(iter(RECONCILE_CANDIDATES))
control_folds = fold_scores(reconcile_per_day[control_name])
reconcile_summary = pd.DataFrame([dict(
    candidate=name,
    development_8_folds=official_score(per_day, DEVELOPMENT_FOLDS),
    folds_better_than_control=f"{int((fold_scores(per_day)[DEVELOPMENT_FOLDS] < control_folds[DEVELOPMENT_FOLDS]).sum())}/8",
    sep_1_30=official_score(per_day, ["sep_full"]),
    aug2_sep30=official_score(per_day, ["aug2_sep30"]),
) for name, per_day in reconcile_per_day.items()])
reconcile_summary.to_csv(reconcile_dir / "summary.csv", index=False)
pd.concat({name: fold_scores(per_day) for name, per_day in reconcile_per_day.items()}, axis=1).to_csv(reconcile_dir / "fold_scores.csv")
display(reconcile_summary.round(4))
display(pd.concat({name: fold_scores(per_day) for name, per_day in reconcile_per_day.items()}, axis=1).round(3))

**Results.** Verified outside the notebook with the same code, LightGBM 4.7.0, 4 threads. MAE in EUR/MWh:

| Candidate | 8 monthly folds | folds better than control | Sep 1–30 | Aug 2–Sep 30 |
| --- | --- | --- | --- | --- |
| 4.9 control: centre PCHIP, 50 features, notebook LightGBM | 3.980 | — | 5.149 | 4.797 |
| + cumulative PCHIP | 3.941 | 6/8 | 5.157 | 4.855 |
| + cumulative PCHIP + revised features | **3.939** | 6/8 | 5.099 | 4.755 |
| AET-HACKATHON final (I2), its own code | 3.954 | 4/8 | 5.099 | **4.739** |
| AET-HACKATHON features + notebook LightGBM | **3.939** | 6/8 | **5.072** | 4.749 |

**Reading.**

* The 4.13 gap is about 1 %, not 3 %. Like-for-like, AET beats Section 4.13 by 0.05 on Sep 1–30 (better on 18 of 30 dates) and by 0.06 on Aug 2–Sep 30 (40 of 60 dates). The rest of the apparent gap comes from the 12-day locked block and the zone relabelling.
* The cumulative PCHIP reference is the better construction: exact by telescoping, with no re-centring jumps, and 2 % better on its own. As a reference under the learner, its 8-fold gain (−0.039, 6/8 folds) comes mostly from April (4.524 → 4.298); without April it is about −0.01. Adopt it for correctness and simplicity, not as a large accuracy gain.
* Changing only the reference and features of 4.9 (revised row) reaches 3.939 / 5.099 / 4.755, level with AET. No single feature change explains the Aug–Sep difference. Missing edge slopes, next-hour slopes for other zones, ±2-hour slopes and dropping the duplicate season encodings each help Aug–Sep a little and only work combined.
* The notebook's LightGBM settings are slightly better than AET's: −0.014 on 8 folds with AET features, and −0.027 on September. This is a small improvement for the AET side, below its own 0.5 % promotion gate.
* Noise level: relabelling zones moved AET's locked score by 0.03, reordering columns moved Sep 1–30 by 0.02, and on 30 days the paired daily differences have a standard deviation of about 0.19. Treat single-window differences below about 0.05 as ties; decide on the 8 folds.

### 4.16 Final combined model

This section combines the strongest parts of Section 4.9 and AET-HACKATHON (Section 4.15), using only this notebook's data, feature builders and official helpers:

| Component | Source | Evidence (Section 4.15) |
| --- | --- | --- |
| Cumulative PCHIP reference, exact hourly means by telescoping | AET-HACKATHON | 6/8 folds better than the centre-PCHIP reference |
| Edge-aware curve features: missing out-of-day neighbours, ±1/±2-hour slopes, relative level, every zone's ±1-hour slopes | AET-HACKATHON | level with AET on 8 folds; better on Aug–Sep |
| LightGBM settings: 800 trees, 31 leaves, learning rate 0.03, 60 rows per leaf, L2 = 5 | Section 4.9 | 6/8 folds better than AET's settings |
| Polynomial zero-sum basis, three L1 regressors, correction weight 0.8 | both | 0.8 chosen independently by both projects on earlier data |

Two feature selections are compared, both built from this notebook's hourly table:

* `aet_equivalent` (46): AET-HACKATHON's feature list mapped onto existing columns (zone, hour, weekday, weekend, year sin/cos, level, four slopes, curvature, day mean/range/std/negative hours, relative level, the other zones' spreads and ±1-hour slopes, reference coefficients, lag 1/7 and 7/28-day means with counts).
* `revised` (51): the original 50 predictors without edge fallbacks and duplicate season encodings, plus the AET-style curve features.

**Selection rule, fixed before scoring:** pooled official MAE over the **8 monthly folds** (train through the end of month *m*, score days 19–end of month *m+1*, January–August 2026). If the two are within 0.01 EUR/MWh, keep the one with fewer features. Sep 1–30 and Aug 2–Sep 30 are reported for comparison with Sections 4.9–4.13 and do not choose anything.

Every score passes through `evaluate_experiment`: reference + 0.8 × correction, hourly mean correction, `validate_prediction_output` (four finite quarters, |mean − hourly| ≤ 0.0001), CSV export at `%.10f`, read-back, re-validation and `challenge_mae` (day → zone → equal four-zone mean). Features for each date use only quarter-hours from strictly earlier dates; earlier dates in a scored window may feed later dates' history, as with the jury's daily `history_15min.csv`.

Requires `lightgbm` and cells 4.15a–4.15b (the reference and feature definitions); 4.15c–d are not needed. Runtime is about 5 minutes.

In [ ]:
# 4.16a Final model definition, training and jury-style inference.
import joblib

FINAL_METHOD = "cumpchip"
FINAL_WEIGHT = 0.8
FINAL_LGBM = COEFFICIENT_PARAMETERS.copy()
AET_EQUIVALENT_COLUMNS = [
    "position_zone", "position_hour", "calendar_day_of_week", "calendar_is_weekend", "calendar_year_sin", "calendar_year_cos",
    "local_hourly_price_eur_mwh",
    "temporal_slope_minus_2_eur_mwh", "temporal_slope_minus_1_eur_mwh", "temporal_slope_plus_1_eur_mwh",
    "temporal_slope_plus_2_eur_mwh", "temporal_curvature_nan_edges_eur_mwh",
    "daily_mean_price_eur_mwh", "daily_range_price_eur_mwh", "daily_std_price_eur_mwh", "regime_daily_negative_hours",
    "daily_relative_level_eur_mwh",
    *[f"crosszone_{zone}_{kind}" for zone in EXPECTED_ZONES
      for kind in ("minus_local_eur_mwh", "slope_minus_1_eur_mwh", "slope_plus_1_eur_mwh")],
    *[f"reference_{name}_coefficient_eur_mwh" for name in COEFFICIENT_NAMES],
    *[f"history_coefficient_{name}_{window}_eur_mwh" for window in ("lag_1d", "lag_7d", "mean_7d", "mean_28d")
      for name in COEFFICIENT_NAMES],
    "history_coefficient_count_7d", "history_coefficient_count_28d",
]
FINAL_FEATURE_SETS = {"aet_equivalent": AET_EQUIVALENT_COLUMNS, "revised": revised_columns}
final_table = reconcile_tables[FINAL_METHOD]
for columns in FINAL_FEATURE_SETS.values():
    missing = set(columns) - set(final_table.columns)
    if missing or len(columns) != len(set(columns)) or set(columns) & set(COEFFICIENT_TARGETS):
        raise ValueError(f"Invalid final feature selection: {sorted(missing)}")


def build_final_features(hourly_inputs, history_15min):
    """Hourly predictors and the cumulative PCHIP reference, both in date/zone/hour(/quarter) order."""
    features, reference = build_coefficient_features(hourly_inputs, history_15min, FINAL_METHOD)
    return add_reconcile_features(features), reference


def fit_final_model(training_rows, columns):
    regressors = [LGBMRegressor(**FINAL_LGBM).fit(training_rows[columns], training_rows[target], categorical_feature=["position_zone"])
                  for target in COEFFICIENT_TARGETS]
    return dict(method=FINAL_METHOD, columns=list(columns), basis=COEFFICIENT_BASIS.copy(),
                weights=np.repeat(FINAL_WEIGHT, 3), regressors=regressors, parameters=FINAL_LGBM,
                zone_categories=list(EXPECTED_ZONES), training_dates=int(training_rows.delivery_date.nunique()),
                training_last_date=str(training_rows.delivery_date.max().date()))


def final_deviations(bundle, features):
    coefficients = np.column_stack([model.predict(features[bundle["columns"]]) for model in bundle["regressors"]])
    return ((coefficients * bundle["weights"]) @ bundle["basis"]).reshape(-1)


def predict_final_model(bundle, test_hourly, history_15min):
    """Jury entry point: test_hourly.csv + history_15min.csv frames -> validated predictions in the official schema."""
    if bundle["zone_categories"] != list(EXPECTED_ZONES):
        raise ValueError("Checkpoint zone categories do not match this notebook.")
    hourly = test_hourly.copy()
    hourly["delivery_date"] = pd.to_datetime(hourly.delivery_date)
    features, reference = build_final_features(hourly, history_15min)
    return make_predictions(reference, hourly, final_deviations(bundle, features),
                            reference_type=FINAL_METHOD, reference_column=COEFFICIENT_REFERENCE)

In [ ]:
# 4.16b Select the feature set on the 8 monthly folds; every fold is scored through evaluate_experiment.
final_dir = output_dir / "final_combined"
final_dir.mkdir(parents=True, exist_ok=True)
FINAL_FOLDS = [(f"m{month:%Y_%m}", month - pd.Timedelta(days=1), month + pd.Timedelta(days=18), month + pd.offsets.MonthEnd(0))
               for month in pd.date_range("2026-01-01", "2026-08-01", freq="MS")]
FINAL_FOLDS += [("sep_full", pd.Timestamp("2026-08-31"), pd.Timestamp("2026-09-01"), pd.Timestamp("2026-09-30")),
                ("aug2_sep30", pd.Timestamp("2026-08-01"), pd.Timestamp("2026-08-02"), pd.Timestamp("2026-09-30"))]
FINAL_DEVELOPMENT = [name for name, *_ in FINAL_FOLDS if name.startswith("m")]

final_fold_inputs = {}
for fold, train_end, start, end in FINAL_FOLDS:
    hourly = train_hourly[train_hourly.delivery_date.between(start, end)]
    features, reference = build_final_features(hourly, prices)   # history strictly before each scored date
    final_fold_inputs[fold] = dict(train_end=train_end, hourly=hourly, features=features, reference=reference,
                                   truth=prices[prices.delivery_date.between(start, end)])

final_fold_results = {}
for feature_set, columns in FINAL_FEATURE_SETS.items():
    for fold, inputs in final_fold_inputs.items():
        bundle = fit_final_model(final_table[final_table.delivery_date.le(inputs["train_end"])], columns)
        final_fold_results[feature_set, fold] = evaluate_experiment(
            f"final_{feature_set}_{fold}", inputs["reference"], inputs["hourly"], inputs["truth"],
            predicted_deviations=final_deviations(bundle, inputs["features"]),
            reference_type=FINAL_METHOD, reference_column=COEFFICIENT_REFERENCE, output_dir=final_dir / "folds")


def pooled_official_mae(feature_set, folds):
    """Official metric over all scored day/zones of the given folds: day -> zone mean -> four-zone mean."""
    by_day_zone = pd.concat([final_fold_results[feature_set, fold]["scores"]["by_day_zone"] for fold in folds])
    return float(by_day_zone.groupby("market_id").mae_eur_mwh.mean().reindex(EXPECTED_ZONES).mean())


final_selection = pd.DataFrame([dict(
    feature_set=feature_set, feature_count=len(columns),
    development_8_folds=pooled_official_mae(feature_set, FINAL_DEVELOPMENT),
    sep_1_30=final_fold_results[feature_set, "sep_full"]["summary"]["mae_eur_mwh"],
    aug2_sep30=final_fold_results[feature_set, "aug2_sep30"]["summary"]["mae_eur_mwh"],
    max_hourly_error_eur_mwh=max(final_fold_results[feature_set, fold]["summary"]["max_hourly_error_eur_mwh"] for fold in final_fold_inputs),
) for feature_set, columns in FINAL_FEATURE_SETS.items()]).sort_values("development_8_folds").reset_index(drop=True)
best_score = final_selection.development_8_folds.min()
tied = final_selection[final_selection.development_8_folds.le(best_score + 0.01)]
FINAL_FEATURE_SET = tied.sort_values(["feature_count", "development_8_folds"]).iloc[0].feature_set
final_selection["selected"] = final_selection.feature_set.eq(FINAL_FEATURE_SET)
final_selection.to_csv(final_dir / "selection.csv", index=False)
final_by_fold = pd.DataFrame({feature_set: {fold: final_fold_results[feature_set, fold]["summary"]["mae_eur_mwh"]
                                            for fold in final_fold_inputs} for feature_set in FINAL_FEATURE_SETS})
final_by_fold.to_csv(final_dir / "mae_by_fold.csv")
print("Selected on the 8 monthly folds (ties within 0.01 -> fewer features):", FINAL_FEATURE_SET)
print("References: Section 4.13 control Sep 1-30 = 5.149; Section 4.9 control Aug 2-Sep 30 = 4.797; "
      "Section 4.9 control 8 folds = 3.980 (Section 4.15).")
display(final_selection.round({"development_8_folds": 4, "sep_1_30": 4, "aug2_sep30": 4}))
display(final_by_fold.round(4))
display(final_fold_results[FINAL_FEATURE_SET, "sep_full"]["scores"]["by_zone"].round(4))

In [ ]:
# 4.16c Refit the selected configuration on all 363 official dates, save it, and smoke-test jury-style inference.
final_bundle = fit_final_model(final_table, FINAL_FEATURE_SETS[FINAL_FEATURE_SET])
final_bundle["feature_set"] = FINAL_FEATURE_SET
final_model_path = final_dir / "final_combined_model.joblib"
joblib.dump(final_bundle, final_model_path)
(final_dir / "settings.json").write_text(json.dumps(dict(
    feature_set=FINAL_FEATURE_SET, columns=final_bundle["columns"], method=FINAL_METHOD, correction_weight=FINAL_WEIGHT,
    basis=COEFFICIENT_BASIS.tolist(), parameters=FINAL_LGBM, training_dates=final_bundle["training_dates"],
    training_last_date=final_bundle["training_last_date"]), indent=2) + "\n")

# Jury-style call for one date: that date's 24 hourly prices per zone and quarter-hours from strictly earlier dates.
# The model was trained on this date, so this checks the pipeline and constraints, not accuracy.
smoke_date = train_hourly.delivery_date.max()
smoke_hourly = train_hourly[train_hourly.delivery_date.eq(smoke_date)]
smoke_history = prices[prices.delivery_date.lt(smoke_date)]
loaded_bundle = joblib.load(final_model_path)
smoke_predictions = predict_final_model(loaded_bundle, smoke_hourly, smoke_history)
pd.testing.assert_frame_equal(smoke_predictions, predict_final_model(final_bundle, smoke_hourly, smoke_history))
smoke_path = final_dir / "smoke_predictions.csv"
smoke_predictions.to_csv(smoke_path, index=False, float_format="%.10f", date_format="%Y-%m-%d")
smoke_checks = validate_prediction_output(pd.read_csv(smoke_path, parse_dates=["delivery_date"]), smoke_hourly)
print(f"Final model: {FINAL_FEATURE_SET}, {len(final_bundle['columns'])} predictors, "
      f"{final_bundle['training_dates']} training dates through {final_bundle['training_last_date']}.")
print(f"Jury-style smoke test for {smoke_date.date()}: {smoke_checks}")
print(f"Inference: predict_final_model(joblib.load('{final_model_path}'), test_hourly, history_15min)")

**Results.** Verified outside the notebook with the same code, LightGBM 4.7.0. Official MAE in EUR/MWh; all fold exports pass the constraint checks (max hourly error 2.5e-11):

| Model | 8 monthly folds | Sep 1–30 | Aug 2–Sep 30 |
| --- | --- | --- | --- |
| Section 4.9 / 4.13 control (centre PCHIP, original 50) | 3.980 | 5.149 | 4.797 |
| AET-HACKATHON final, its own code (Section 4.15) | 3.954 | 5.099 | 4.739 |
| 4.16 `revised` (51) | 3.939 | 5.099 | 4.755 |
| **4.16 `aet_equivalent` (46), selected** | **3.941** | **5.085** | **4.742** |

The two 4.16 selections tie on the 8 folds (0.003 apart, 4/8 folds each), so the rule keeps the smaller `aet_equivalent` set. Against the Section 4.13 control it is 1.0 % lower on the 8 folds, 1.2 % on Sep 1–30 and 1.1 % on Aug 2–Sep 30. It is level with or slightly better than AET-HACKATHON's own pipeline. Differences below about 0.03 are within the noise measured in Section 4.15. Sep 1–30 and Aug 2–Sep 30 were inspected repeatedly in earlier sections, so they are comparisons, not an untouched test.

The submitted model is refitted on all 363 official dates with the fixed configuration; nothing is refitted or recalibrated at inference. For each evaluation date, call `predict_final_model(joblib.load("outputs/experiments/final_combined/final_combined_model.joblib"), test_hourly, history_15min)`. It returns the official columns, already checked by `validate_prediction_output`.